# Certificaciones de biomasa — por proveedor, quincenal y mensual

**Qué hace este cuaderno**
1. Lee el maestro **R-O-08-01** (hoja *Datos Entrada*) y arma la certificación de cada proveedor: 1ª quincena, 2ª quincena y mensual.
2. Controla los errores de tipeo: fechas fuera de secuencia, pesos imposibles, remitos repetidos, CUIT, precios y nombres.
3. Aplica los **descuentos de calidad** como la certificación final: **granulometría por camión** (finos < #04 hasta 10 % → sin descuento; de 10 a 15 % → lo que excede el 10 %; más de 15 % → todos los finos) y esos kilos se pagan como **ASERRÍN**; **humedad** solo a los proveedores con *DESCUENTA HUMEDAD = SI*.
4. Muestra en cada camión su **% de humedad y de granulometría**; nada queda bloqueado: si falta CUIT o precio, sale en blanco para completar.
5. Abre un **editor** donde revisás cada certificación y podés **corregir o agregar datos** antes de confirmarla.
6. Al **confirmar**, le asigna el número correlativo y genera el **PDF** y el **Excel** de esa certificación.
7. Al final descargás los **PDF** y **Excel** de cada certificación (archivos sueltos) y, si querés, **un único Excel por período** con todos los proveedores (una hoja cada uno), más el resumen y el registro de cambios.

**Cómo usarlo:** ejecutá las celdas en orden con ▶. Si el entorno se reinicia, volvé a ejecutar desde el principio y subí también el archivo `estado_AAAA-MM.json` para seguir donde quedaste.

## Paso 1 · Instalar

In [ ]:
!pip install -q pyxlsb openpyxl reportlab

## Paso 2 · Cargar el programa
(No hace falta tocar estas dos celdas: solo ejecutarlas.)

In [ ]:
%%writefile cert_biomasa.py
# =============================================================================
#  MOTOR — Certificaciones de entrega de biomasa (quincenal y mensual)
#  Lee el maestro "R-O-08-01 Control ingreso biomasa", hoja "Datos Entrada".
#
#  - Las correcciones manuales se guardan POR REMITO (no por certificación), así
#    1ª quincena, 2ª quincena y mensual siempre quedan iguales entre sí.
#  - Cada cambio queda en un registro: quién, cuándo, valor anterior, nuevo, motivo.
#  - Una certificación solo se confirma (y genera PDF) si no tiene errores
#    bloqueantes y las observaciones críticas fueron revisadas con comentario.
# =============================================================================
import io
import re
import json
import zipfile
import hashlib
import calendar
import difflib
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.page import PageMargins

MESES = ["", "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio",
         "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]

# Datos de la empresa y pie de la certificación: se cargan en la hoja "Parametros" del config
# (EMPRESA_LINEA_1..3 y PIE_1..3). Acá quedan solo valores de ejemplo.
EMPRESA_LINEAS = [
    "Nombre de la empresa / dirección",
    "Provincia, País",
    "C.U.I.T 00-00000000-0",
]
PIE = [
    "*Ante disconformidad tiene 3 días hábiles para realizar el reclamo pertinente. "
    "Pasado este período se tomará por aceptada.",
    "**Remitir la factura junto a la orden de compra a facturacion@ejemplo.com",
    "Importante: la factura debe indicar el número de orden de compra.",
]

TIPOLOGIAS = ["CHIP", "ASERRIN", "Fino/Raleo", "C LARGO", "C CORTO", "PATAS",
              "CENIZAS", "DESECHOS", "RAMAS"]
MOVIMIENTOS = {"ingreso", "mi", "me"}
PESO_MIN, PESO_MAX = 3000, 45000
DIA_CORTE = 12          # último día de la 1ª quincena (DIA_CORTE_QUINCENA en el config)
# Granulometría (como la planilla de certificación final): finos (< #04) por camión
#   hasta FINOS_TOLERANCIA %           → sin descuento
#   de FINOS_TOLERANCIA a FINOS_TOTALIDAD → se descuenta la diferencia
#   más de FINOS_TOTALIDAD %           → se descuenta la totalidad de los finos
# Lo descontado no se pierde: se paga como ASERRÍN.
FINOS_TOLERANCIA, FINOS_TOTALIDAD = 10.0, 15.0
NOMBRE_PRODUCTO = {"CHIP": "CHIP", "ASERRIN": "ASERRIN", "C CORTO": "C. CORTO", "C LARGO": "C. LARGO",
                   "Fino/Raleo": "FINO/RALEO", "PATAS": "PATAS", "CENIZAS": "CENIZAS", "DESECHOS": "DESECHOS",
                   "RAMAS": "RAMAS"}
PRODUCTOS_CUADRO = ["CHIP", "ASERRIN", "C CORTO", "C LARGO", "Fino/Raleo"]          # rango habitual por camión (kg)
SIN_REMITO = ("", "S/R", "SR", "0", "-")

# Columnas del maestro
C_DIA, C_REM_PLANTA, C_PROV = "DIA", "Nº Remito PBSR", "PROVEEDOR NOMBRE"
C_REM_PROV, C_TIPO, C_BATEA = "PROVEEDOR N°REMITO", "TIPOLOGIA", "P BATEA"
C_NETO, C_MOV, C_QUINC = "PESO NETO", "TIPO DE MOVIMINETO", "Quincena"
C_LLENO, C_VACIO = "PESO LLENO", "PESO VACIO"
C_PATENTE = "PATENTE TRANSPORTE"

# Campos editables de un remito → columna del maestro
CAMPOS = {"Batea": C_BATEA, "Fecha": C_DIA, "Rto. Planta": C_REM_PLANTA,
          "Remito Proveedor": C_REM_PROV, "Producto": C_TIPO, "Kilos": C_NETO,
          "Proveedor": C_PROV}
FILA_COLS = ["Batea", "Fecha", "Rto. Planta", "Remito Proveedor", "Producto", "Kilos"]

CRIT, ADV = "CRÍTICO", "ADVERTENCIA"


# -----------------------------------------------------------------------------
# 1. Lectura y preparación
# -----------------------------------------------------------------------------
def leer_maestro(ruta, hoja="Datos Entrada"):
    """Lee la hoja de remitos (.xlsb rápido, o .xlsx)."""
    ruta = str(ruta)
    if ruta.lower().endswith(".xlsb"):
        from pyxlsb import open_workbook
        filas, header, vacias = [], None, 0
        with open_workbook(ruta) as wb, wb.get_sheet(hoja) as sh:
            for r in sh.rows():
                v = [c.v for c in r]
                if header is None:
                    if C_DIA in v and C_NETO in v:
                        header = [h if h not in (None, "") else f"_col{j}" for j, h in enumerate(v)]
                    continue
                if v[0] in (None, "") and all(x in (None, "", 0, 0.0) for x in v[3:6]):
                    vacias += 1
                    if vacias > 3000:
                        break
                    continue
                vacias = 0
                filas.append((v + [None] * len(header))[:len(header)])
        df = pd.DataFrame(filas, columns=header)
        df[C_DIA] = pd.to_datetime(pd.to_numeric(df[C_DIA], errors="coerce"),
                                   unit="D", origin="1899-12-30")
    else:
        raw = pd.read_excel(ruta, sheet_name=hoja, header=None, nrows=10)
        fila_h = next(i for i, r in raw.iterrows() if C_DIA in r.values and C_NETO in r.values)
        df = pd.read_excel(ruta, sheet_name=hoja, header=fila_h)
        df[C_DIA] = pd.to_datetime(df[C_DIA], errors="coerce")
    df[C_DIA] = df[C_DIA].dt.normalize()
    return df


def _txt(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    return str(x).strip()


def _limpio(serie):
    return serie.apply(_txt).str.replace(r"\s+", " ", regex=True).str.strip()


def _canon_tipo(t):
    canon = {x.casefold(): x for x in TIPOLOGIAS}
    t = _txt(t)
    return canon.get(t.casefold(), t.upper() or "SIN TIPO")


def preparar(df, conf=None):
    """Ingresos certificables, normalizados. Cada fila recibe un _id estable
    (N° remito de planta + n° de aparición) para registrar correcciones."""
    d = df[_limpio(df[C_MOV]).str.lower() == "ingreso"].copy()
    d = d.dropna(subset=[C_DIA])
    d[C_PROV] = _limpio(d[C_PROV])
    for c in (C_NETO, C_LLENO, C_VACIO):
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d[C_NETO] = d[C_NETO].fillna(0)
    for c in (C_REM_PROV, C_REM_PLANTA, C_BATEA):
        d[c] = _limpio(d[c])
    tipo_raw = _limpio(d[C_TIPO])
    d["_tipo_valido"] = tipo_raw.str.casefold().isin({t.casefold() for t in TIPOLOGIAS})
    d[C_TIPO] = tipo_raw.map(_canon_tipo)

    clave = d[C_PROV].str.casefold()
    preferido = d.groupby(clave)[C_PROV].agg(lambda s: s.value_counts().index[0])
    unificados = {}
    for k, nombre in preferido.items():
        variantes = sorted(set(d.loc[clave == k, C_PROV]) - {nombre})
        if variantes:
            unificados[nombre] = variantes
    d[C_PROV] = clave.map(preferido).fillna("").astype(str)

    if conf is not None and "UNIFICAR CON" in conf.columns:
        for k, destino in conf["UNIFICAR CON"].items():
            if str(destino).strip():
                m = d[C_PROV].str.casefold() == k.casefold()
                if m.any():
                    d.loc[m, C_PROV] = destino
                    unificados.setdefault(destino, []).append(k)
    d = d[~d[C_PROV].str.casefold().isin(["", "none", "nan"])]
    # nombre con el que el laboratorio y el maestro registraron el camión (para asignar las muestras)
    d["_prov_lab"] = d[C_PROV]
    d = separar_por_patente(d, conf)
    base = d[C_REM_PLANTA].where(d[C_REM_PLANTA] != "", "SINREM")
    d["_id"] = base + "#" + (d.groupby(base).cumcount() + 1).astype(str)
    d["_origen"] = "maestro"
    return d, unificados


def _patente(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())


def separar_por_patente(d, conf):
    """Regla del config: si un proveedor tiene SOLO PATENTES (ej. PROVEEDOR A = AA123BB), sus camiones con
    otras patentes pasan al proveedor de OTRAS PATENTES A (ej. PROVEEDOR B). Así se separan dos proveedores que el
    maestro registra con el mismo nombre."""
    if conf is None or "SOLO PATENTES" not in conf.columns or C_PATENTE not in d.columns:
        return d
    d = d.copy()
    for prov, fila in conf.iterrows():
        solo = {_patente(x) for x in str(fila.get("SOLO PATENTES", "")).replace(";", ",").split(",") if x.strip()}
        destino = str(fila.get("OTRAS PATENTES A", "")).strip()
        if not solo or not destino:
            continue
        m = (d[C_PROV].str.casefold() == str(prov).casefold()) & ~d[C_PATENTE].map(_patente).isin(solo)
        d.loc[m, C_PROV] = destino
    return d


# -----------------------------------------------------------------------------
# 2. Configuración (proveedores, precios, numeración)
# -----------------------------------------------------------------------------
def plantilla_config(d):
    provs = sorted(d[C_PROV].unique(), key=str.casefold)
    conf = pd.DataFrame({"PROVEEDOR (maestro)": provs, "RAZÓN SOCIAL": provs, "CUIT": "",
                         "GRUPO": provs, "CERTIFICAR": "SI", "UNIFICAR CON": "",
                         "NOMBRE HUMEDAD": "", "DESCUENTA HUMEDAD": "NO", "TOLERANCIA HUMEDAD %": "",
                         "LIMITE HUMEDAD %": "", "DESCUENTA GRANULOMETRIA": "SI", "TOLERANCIA FINOS %": "",
                         "CERTIFICAR CON": "", "FORMATO": "", "SOLO PATENTES": "", "OTRAS PATENTES A": ""})
    precios = pd.DataFrame(columns=["GRUPO", "TIPOLOGIA", "DESDE_TN", "HASTA_TN", "PRECIO"])
    return conf.set_index("PROVEEDOR (maestro)"), precios, {"NUMERO_INICIAL": 1, "PREFIJO": "0001", "METODO_HUMEDAD": "PUNTOS",
                                                          "DIA_CORTE_QUINCENA": 12, "FINOS_TOLERANCIA": 10.0,
                                                          "FINOS_TOTALIDAD": 15.0}


def leer_config(ruta):
    conf = pd.read_excel(ruta, sheet_name="Proveedores", dtype=str).fillna("")
    conf.columns = [c.strip() for c in conf.columns]
    for col in ("UNIFICAR CON", "NOMBRE HUMEDAD", "DESCUENTA HUMEDAD", "TOLERANCIA HUMEDAD %", "LIMITE HUMEDAD %",
                "DESCUENTA GRANULOMETRIA", "TOLERANCIA FINOS %", "CERTIFICAR CON", "FORMATO", "SOLO PATENTES",
                "OTRAS PATENTES A"):
        if col not in conf.columns:
            conf[col] = ""
    conf["PROVEEDOR (maestro)"] = conf["PROVEEDOR (maestro)"].str.strip().str.replace(r"\s+", " ", regex=True)
    conf = conf.drop_duplicates("PROVEEDOR (maestro)").set_index("PROVEEDOR (maestro)")
    for c in conf.columns:
        conf[c] = conf[c].str.strip()
    precios = pd.read_excel(ruta, sheet_name="Precios")
    precios["GRUPO"] = precios["GRUPO"].astype(str).str.strip()
    precios["TIPOLOGIA"] = precios["TIPOLOGIA"].map(_canon_tipo)
    for c in ("DESDE_TN", "HASTA_TN", "PRECIO"):
        precios[c] = pd.to_numeric(precios[c], errors="coerce")
    precios = precios.dropna(subset=["GRUPO"]).reset_index(drop=True)
    try:
        par = pd.read_excel(ruta, sheet_name="Parametros").set_index("PARÁMETRO")["VALOR"].to_dict()
    except ValueError:
        par = {}
    par.setdefault("NUMERO_INICIAL", 1)
    par.setdefault("PREFIJO", "0001")
    par["METODO_HUMEDAD"] = str(par.get("METODO_HUMEDAD") or "PUNTOS").strip().upper()
    if par["METODO_HUMEDAD"] not in METODOS_HUMEDAD:
        par["METODO_HUMEDAD"] = "PUNTOS"
    try:
        par["DIA_CORTE_QUINCENA"] = int(float(par.get("DIA_CORTE_QUINCENA") or 12))
    except (TypeError, ValueError):
        par["DIA_CORTE_QUINCENA"] = 12
    if not 10 <= par["DIA_CORTE_QUINCENA"] <= 20:
        par["DIA_CORTE_QUINCENA"] = 12
    fijar_corte(par["DIA_CORTE_QUINCENA"])
    for k, defecto in (("FINOS_TOLERANCIA", 10.0), ("FINOS_TOTALIDAD", 15.0)):
        try:
            par[k] = float(str(par.get(k) if par.get(k) is not None else defecto).replace(",", ".").replace("%", ""))
        except ValueError:
            par[k] = defecto
    fijar_finos(par["FINOS_TOLERANCIA"], par["FINOS_TOTALIDAD"])
    fijar_textos(par)
    return conf, precios, par


def fijar_textos(par):
    """Datos de la empresa y pie de la certificación tomados de la hoja Parametros del config."""
    for prefijo, lista in (("EMPRESA_LINEA_", EMPRESA_LINEAS), ("PIE_", PIE)):
        for k in range(len(lista)):
            v = par.get(f"{prefijo}{k + 1}")
            if v is not None and str(v).strip() and str(v).strip().lower() != "nan":
                lista[k] = str(v).strip()
            par[f"{prefijo}{k + 1}"] = lista[k]


def fijar_finos(tolerancia, totalidad):
    global FINOS_TOLERANCIA, FINOS_TOTALIDAD
    FINOS_TOLERANCIA, FINOS_TOTALIDAD = float(tolerancia), float(totalidad)


def fijar_corte(dia):
    """Define el último día de la 1ª quincena (la 2ª empieza el día siguiente)."""
    global DIA_CORTE
    DIA_CORTE = int(dia)


def guardar_config(ruta, conf, precios, par):
    with pd.ExcelWriter(ruta, engine="openpyxl") as w:
        conf.reset_index().to_excel(w, sheet_name="Proveedores", index=False)
        precios.to_excel(w, sheet_name="Precios", index=False)
        textos = [f"EMPRESA_LINEA_{k + 1}" for k in range(len(EMPRESA_LINEAS))] + [f"PIE_{k + 1}" for k in range(len(PIE))]
        pd.DataFrame({"PARÁMETRO": ["NUMERO_INICIAL", "PREFIJO", "METODO_HUMEDAD", "DIA_CORTE_QUINCENA",
                                    "FINOS_TOLERANCIA", "FINOS_TOTALIDAD"] + textos,
                      "VALOR": [int(par["NUMERO_INICIAL"]), str(par["PREFIJO"]).zfill(4),
                                str(par.get("METODO_HUMEDAD", "PUNTOS")).upper(),
                                int(par.get("DIA_CORTE_QUINCENA", DIA_CORTE)),
                                float(par.get("FINOS_TOLERANCIA", FINOS_TOLERANCIA)),
                                float(par.get("FINOS_TOTALIDAD", FINOS_TOTALIDAD))]
                               + [par.get(t, (EMPRESA_LINEAS + PIE)[i]) for i, t in enumerate(textos)],
                      "AYUDA": ["Próximo número de certificación", "Punto de venta",
                                "PUNTOS = TN × puntos sobre el límite · BASE_SECA = TN × (1−H)/(1−L)",
                                "Último día de la 1ª quincena (ej. 12 = del 1 al 12 y del 13 a fin de mes)",
                                "Granulometría: % de finos sin descuento (general; se puede cambiar por proveedor)",
                                "Granulometría: por encima de este % se descuenta la totalidad de los finos"]
                               + ["Encabezado de la certificación (datos de la empresa)"] * len(EMPRESA_LINEAS)
                               + ["Pie de la certificación"] * len(PIE)}
                     ).to_excel(w, sheet_name="Parametros", index=False)
        pd.DataFrame({"Tipologías válidas": TIPOLOGIAS}).to_excel(w, sheet_name="Listas", index=False)
    wb = load_workbook(ruta)
    for ws in wb:
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = max(
                12, min(45, max(len(str(c.value or "")) for c in col) + 2))
        for c in ws[1]:
            c.font = Font(bold=True, color="FFFFFF")
            c.fill = PatternFill("solid", fgColor="3A7D44")
        ws.freeze_panes = "A2"
    ws = wb["Precios"]
    dv = DataValidation(type="list", formula1=f"=Listas!$A$2:$A${len(TIPOLOGIAS) + 1}",
                        showErrorMessage=True, errorTitle="Tipología inválida",
                        error="Elegí una tipología de la lista.")
    ws.add_data_validation(dv)
    dv.add("B2:B1000")
    dvp = DataValidation(type="decimal", operator="greaterThan", formula1="0", showErrorMessage=True,
                         errorTitle="Precio inválido", error="El precio debe ser un número mayor a 0.")
    ws.add_data_validation(dvp)
    dvp.add("E2:E1000")
    ws = wb["Proveedores"]
    dvs = DataValidation(type="list", formula1='"SI,NO"', showErrorMessage=True)
    ws.add_data_validation(dvs)
    cols = [c.value for c in ws[1]]
    for nombre in ("CERTIFICAR", "DESCUENTA HUMEDAD", "DESCUENTA GRANULOMETRIA"):
        if nombre in cols:
            letra = ws.cell(1, cols.index(nombre) + 1).column_letter
            dvs.add(f"{letra}2:{letra}{len(conf) + 300}")
    wb.save(ruta)


def cuit_valido(cuit):
    n = re.sub(r"\D", "", str(cuit))
    if len(n) != 11:
        return False
    s = sum(int(a) * b for a, b in zip(n[:10], [5, 4, 3, 2, 7, 6, 5, 4, 3, 2]))
    v = 11 - s % 11
    v = 0 if v == 11 else v
    return v != 10 and v == int(n[10])


def puntos(v):
    """Lee un porcentaje cargado en el config: '2', '2%', '2,5' → 0.02 / 0.025. Vacío → None."""
    t = str(v if v is not None else "").replace("%", "").replace(",", ".").strip()
    if t in ("", "nan", "None"):
        return None
    try:
        return float(t) / 100
    except ValueError:
        return "ERROR"


def unidad(conf, prov):
    """Nombre de la certificación a la que pertenece un punto de entrega (columna CERTIFICAR CON)."""
    if prov in conf.index and "CERTIFICAR CON" in conf.columns:
        v = str(conf.at[prov, "CERTIFICAR CON"]).strip()
        if v and v.lower() not in ("nan", "none"):
            return v
    return prov


def miembros(conf, nombre_unidad, presentes=None):
    """Puntos de entrega (nombres del maestro) que forman una certificación."""
    cand = presentes if presentes is not None else conf.index
    return sorted([p for p in cand if unidad(conf, p) == nombre_unidad], key=str.casefold) or [nombre_unidad]


def info_proveedor(conf, prov):
    if prov in conf.index:
        f = conf.loc[prov]
        return {"razon": f.get("RAZÓN SOCIAL") or prov, "cuit": f.get("CUIT", ""),
                "grupo": f.get("GRUPO") or prov,
                "certifica": str(f.get("CERTIFICAR", "SI")).upper() != "NO",
                "tolerancia": puntos(f.get("TOLERANCIA HUMEDAD %", "")),
                "limite": puntos(f.get("LIMITE HUMEDAD %", "")),
                "desc_hum": str(f.get("DESCUENTA HUMEDAD", "")).strip().upper() == "SI",
                "desc_gran": str(f.get("DESCUENTA GRANULOMETRIA", "")).strip().upper() != "NO",
                "tol_finos": puntos(f.get("TOLERANCIA FINOS %", "")),
                "formato": str(f.get("FORMATO", "") or "").strip().upper() or _formato_por_nombre(prov)}
    return {"razon": prov, "cuit": "", "grupo": prov, "certifica": True, "tolerancia": None, "limite": None,
            "desc_hum": False, "desc_gran": True, "tol_finos": None, "formato": _formato_por_nombre(prov)}


def _formato_por_nombre(prov):
    """Formato estándar salvo que el config indique otro (columna FORMATO, ej. GYG)."""
    return ""


def buscar_precio(precios, grupo, tipologia, tn_grupo_mes):
    p = precios[(precios["GRUPO"].str.casefold() == str(grupo).casefold()) &
                (precios["TIPOLOGIA"].astype(str).str.casefold() == str(tipologia).casefold())]
    for _, r in p.iterrows():
        desde = r["DESDE_TN"] if pd.notna(r["DESDE_TN"]) else 0
        hasta = r["HASTA_TN"] if pd.notna(r["HASTA_TN"]) else float("inf")
        if desde <= tn_grupo_mes < hasta and pd.notna(r["PRECIO"]) and r["PRECIO"] > 0:
            return float(r["PRECIO"])
    return None


def fijar_precio(precios, grupo, tipologia, tn_grupo_mes, valor):
    """Guarda un precio en la tabla: actualiza el tramo que corresponde o crea uno único."""
    m = ((precios["GRUPO"].str.casefold() == str(grupo).casefold()) &
         (precios["TIPOLOGIA"].astype(str).str.casefold() == str(tipologia).casefold()))
    for i in precios.index[m]:
        desde = precios.at[i, "DESDE_TN"] if pd.notna(precios.at[i, "DESDE_TN"]) else 0
        hasta = precios.at[i, "HASTA_TN"] if pd.notna(precios.at[i, "HASTA_TN"]) else float("inf")
        if desde <= tn_grupo_mes < hasta:
            precios.at[i, "PRECIO"] = float(valor)
            return precios
    nueva = pd.DataFrame([[grupo, tipologia, 0, None, float(valor)]], columns=precios.columns[:5])
    return pd.concat([precios, nueva], ignore_index=True)


def validar_config(conf, precios):
    obs = []
    grupos = {g.casefold() for g in conf["GRUPO"] if g} | {p.casefold() for p in conf.index}
    for i, r in precios.iterrows():
        fila = f"Precios fila {i + 2}"
        if r["GRUPO"].casefold() not in grupos:
            parecido = difflib.get_close_matches(r["GRUPO"], list(conf["GRUPO"].unique()), n=1, cutoff=0.6)
            obs.append((CRIT, r["GRUPO"], f"{fila}: el GRUPO '{r['GRUPO']}' no existe en Proveedores"
                        + (f" (¿quisiste decir '{parecido[0]}'?)" if parecido else "")))
        if r["TIPOLOGIA"] not in TIPOLOGIAS:
            obs.append((CRIT, r["GRUPO"], f"{fila}: tipología '{r['TIPOLOGIA']}' no válida"))
        if pd.isna(r["PRECIO"]) or r["PRECIO"] <= 0:
            obs.append((CRIT, r["GRUPO"], f"{fila}: precio vacío, cero o no numérico"))
        if pd.notna(r["HASTA_TN"]) and pd.notna(r["DESDE_TN"]) and r["HASTA_TN"] <= r["DESDE_TN"]:
            obs.append((CRIT, r["GRUPO"], f"{fila}: HASTA_TN menor o igual a DESDE_TN"))
    for (g, t), p in precios.dropna(subset=["PRECIO"]).groupby([precios["GRUPO"].str.casefold(), "TIPOLOGIA"]):
        p = p.sort_values("DESDE_TN")
        desde, hasta = p["DESDE_TN"].fillna(0).tolist(), p["HASTA_TN"].fillna(float("inf")).tolist()
        pr = p["PRECIO"].tolist()
        if desde[0] != 0:
            obs.append((CRIT, p["GRUPO"].iloc[0], f"Tramos de {t}: el primero empieza en {desde[0]} y no en 0"))
        for k in range(1, len(desde)):
            if desde[k] != hasta[k - 1]:
                obs.append((CRIT, p["GRUPO"].iloc[0], f"Tramos de {t}: hueco o superposición entre "
                            f"{hasta[k-1]:g} y {desde[k]:g} TN"))
            if pr[k] > pr[k - 1] * 1.5 or pr[k] < pr[k - 1] / 1.5:
                obs.append((CRIT, p["GRUPO"].iloc[0], f"Tramos de {t}: el precio salta de "
                            f"{pesos(pr[k-1])} a {pesos(pr[k])}. ¿Tipeo?"))
    for t, p in precios.dropna(subset=["PRECIO"]).groupby("TIPOLOGIA"):
        med = p["PRECIO"].median()
        if len(p) >= 3:
            for _, r in p.iterrows():
                if r["PRECIO"] > med * 3 or r["PRECIO"] < med / 3:
                    obs.append((CRIT, r["GRUPO"], f"Precio de {t} {pesos(r['PRECIO'])} muy distinto al "
                                f"habitual ({pesos(med)}). ¿Cero de más o de menos?"))
    return obs


# -----------------------------------------------------------------------------
# 3. Correcciones manuales (capa sobre el maestro)
# -----------------------------------------------------------------------------
def estado_nuevo(anio, mes, proximo_numero, prefijo="0001"):
    return {"anio": int(anio), "mes": int(mes), "proximo": int(proximo_numero), "prefijo": str(prefijo).zfill(4),
            "corr": {"editados": {}, "eliminados": {}, "agregados": {}, "finos": {}, "humedad_camion": {},
                     "desc_gran": {}, "pen_hum": {}},
            "meta": {}, "log": []}


def aplicar_correcciones(d, corr):
    d2 = d.copy()
    for _id, campos in corr["editados"].items():
        m = d2["_id"] == _id
        for campo, v in campos.items():
            col = CAMPOS[campo]
            if campo == "Fecha":
                v = pd.Timestamp(v)
            elif campo == "Kilos":
                v = float(v)
            elif campo == "Producto":
                d2.loc[m, "_tipo_valido"] = v in TIPOLOGIAS
            d2.loc[m, col] = v
        if "Kilos" in campos:          # el neto se fijó a mano: lleno/vacío ya no aplican
            d2.loc[m, [C_LLENO, C_VACIO]] = float("nan")
        if m.any():
            d2.loc[m, "_origen"] = "corregido"
    if corr["eliminados"]:
        d2 = d2[~d2["_id"].isin(list(corr["eliminados"]))]
    if corr["agregados"]:
        filas = []
        for _id, f in corr["agregados"].items():
            filas.append({"_id": _id, C_DIA: pd.Timestamp(f["Fecha"]), C_PROV: f["Proveedor"],
                          C_BATEA: f.get("Batea", ""), C_REM_PLANTA: f.get("Rto. Planta", ""),
                          C_REM_PROV: f.get("Remito Proveedor", ""), C_TIPO: f["Producto"],
                          C_NETO: float(f["Kilos"]), C_MOV: "Ingreso",
                          "_tipo_valido": f["Producto"] in TIPOLOGIAS, "_origen": "agregado"})
        d2 = pd.concat([d2, pd.DataFrame(filas)], ignore_index=True)
    d2[C_DIA] = pd.to_datetime(d2[C_DIA]).dt.normalize()
    return d2


# -----------------------------------------------------------------------------
# 4. Validación de remitos
# -----------------------------------------------------------------------------
def _num_remito(s):
    m = re.fullmatch(r"\d{6}-(\d+)", str(s))
    return int(m.group(1)) if m else None


def validar_maestro(df_raw, d, conf, anio, mes):
    """por_id: {_id: [(sev, msg, bloqueante)]}; globales: observaciones que no son de
    un remito que esté dentro de una certificación (ej. remito que quedó fuera del mes)."""
    ini = pd.Timestamp(anio, mes, 1)
    fin = ini + pd.offsets.MonthEnd(0)
    por_id, globales = {}, []

    def add(_id, sev, msg, bloq=False):
        por_id.setdefault(_id, []).append((sev, msg, bloq))

    mes_df = d[(d[C_DIA] >= ini) & (d[C_DIA] <= fin)]
    for _, f in mes_df.iterrows():
        i = f["_id"]
        if not f[C_REM_PLANTA]:
            add(i, CRIT, "Ingreso sin N° de remito de planta")
        if f[C_NETO] <= 0:
            add(i, CRIT, "Peso neto vacío o cero", True)
        elif f[C_NETO] > PESO_MAX * 1.2:
            add(i, CRIT, f"Peso neto {miles(f[C_NETO])} kg imposible para un camión. ¿Tipeo?", True)
        elif f[C_NETO] > PESO_MAX or f[C_NETO] < PESO_MIN:
            add(i, ADV, f"Peso neto fuera de rango habitual ({miles(f[C_NETO])} kg)")
        if pd.notna(f.get(C_LLENO)) and pd.notna(f.get(C_VACIO)) and abs(f[C_LLENO] - f[C_VACIO] - f[C_NETO]) > 1:
            add(i, CRIT, f"Lleno − vacío ({miles(f[C_LLENO] - f[C_VACIO])}) ≠ neto ({miles(f[C_NETO])})")
        if not f["_tipo_valido"]:
            add(i, CRIT, f"Tipología '{f[C_TIPO]}' no reconocida", True)
        if f[C_REM_PROV].upper() in SIN_REMITO:
            add(i, ADV, "Sin remito de proveedor")
        q = _txt(f.get(C_QUINC, ""))
        if q and f["_origen"] == "maestro" and q[0] != ("1" if f[C_DIA].day <= 15 else "2"):
            add(i, ADV, f"La columna Quincena dice '{q}' y la fecha es {f[C_DIA]:%d/%m}")
        if f["_origen"] in ("corregido", "agregado"):
            add(i, ADV, "Remito corregido o agregado a mano (ver registro de cambios)")

    med = mes_df.groupby(C_PROV)[C_NETO].transform("median")
    for _, f in mes_df[(mes_df[C_NETO] > med * 2.5) | (mes_df[C_NETO] < med * 0.35)].iterrows():
        add(f["_id"], ADV, f"Peso {miles(f[C_NETO])} kg muy distinto a lo habitual del proveedor")

    for rem, g in mes_df[mes_df[C_REM_PLANTA] != ""].groupby(C_REM_PLANTA):
        if len(g) > 1:
            for i in g["_id"]:
                add(i, CRIT, f"N° de remito de planta {rem} repetido {len(g)} veces")

    hist = d[(d[C_DIA] > ini - pd.DateOffset(months=6)) & (d[C_DIA] <= fin)]
    hist = hist[~hist[C_REM_PROV].str.upper().isin(SIN_REMITO)]
    # proveedores que habitualmente ponen el mismo remito a varios camiones : solo aviso, no crítico
    rep_ = hist.groupby([C_PROV, C_REM_PROV])[C_REM_PLANTA].transform("size") > 1
    habitual = (rep_.groupby(hist[C_PROV]).mean() >= 0.2).to_dict()
    for (prov, rp), g in hist.groupby([C_PROV, C_REM_PROV]):
        if len(g) > 1 and (g[C_DIA] >= ini).any():
            otros = ", ".join(f"{r} ({fe:%d/%m})" for r, fe in zip(g[C_REM_PLANTA], g[C_DIA]))
            for i in g.loc[g[C_DIA] >= ini, "_id"]:
                add(i, ADV if habitual.get(prov) else CRIT, f"Remito de proveedor {rp} usado más de una vez: {otros}"
                    + (" (habitual en este proveedor)" if habitual.get(prov) else ""))

    s = d.loc[d["_origen"] != "agregado", [C_DIA, C_REM_PLANTA, C_PROV, "_id"]].copy()
    s["n"] = s[C_REM_PLANTA].map(_num_remito)
    s = s.dropna(subset=["n"]).sort_values("n")
    s["ant"], s["sig"] = s[C_DIA].shift(1), s[C_DIA].shift(-1)
    vecinos_ok = (s["sig"] - s["ant"]).abs() <= pd.Timedelta(days=3)
    lejos = ((s[C_DIA] - s["ant"]).abs() > pd.Timedelta(days=1)) & ((s[C_DIA] - s["sig"]).abs() > pd.Timedelta(days=1))
    for _, f in s[vecinos_ok & lejos].iterrows():
        en_mes, vecino_en_mes = ini <= f[C_DIA] <= fin, ini <= f["ant"] <= fin
        if not (en_mes or vecino_en_mes):
            continue
        msg = (f"Fecha {f[C_DIA]:%d/%m/%Y} no coincide con los remitos vecinos "
               f"({f['ant']:%d/%m} y {f['sig']:%d/%m}). ¿Error de tipeo en la fecha?")
        if en_mes:
            add(f["_id"], CRIT, msg)
        else:
            globales.append({"sev": CRIT, "prov": f[C_PROV], "fecha": f[C_DIA], "rem": f[C_REM_PLANTA],
                             "msg": msg + f" Por esa fecha NO entra en {MESES[mes]}: si corresponde, "
                                          "traelo con 'Traer remito del maestro' y corregí la fecha."})

    m_raw = df_raw[(df_raw[C_DIA] >= ini) & (df_raw[C_DIA] <= fin)]
    tipo_mov = _limpio(m_raw[C_MOV]).str.lower()
    neto_raw = pd.to_numeric(m_raw[C_NETO], errors="coerce").fillna(0)
    ya = set(d[C_REM_PLANTA])
    for _, f in m_raw[~tipo_mov.isin(MOVIMIENTOS) & (neto_raw != 0)].iterrows():
        if _txt(f[C_REM_PLANTA]) in ya:
            continue
        globales.append({"sev": CRIT, "prov": _txt(f[C_PROV]), "fecha": f[C_DIA], "rem": _txt(f[C_REM_PLANTA]),
                         "msg": f"TIPO DE MOVIMIENTO '{_txt(f[C_MOV])}' no reconocido: no se certifica. "
                                "Si es un ingreso, traelo con 'Traer remito del maestro'."})

    if conf is not None:
        for prov in sorted(mes_df[C_PROV].unique()):
            if prov not in conf.index:
                parecido = difflib.get_close_matches(prov, list(conf.index), n=1, cutoff=0.8)
                globales.append({"sev": CRIT, "prov": prov, "fecha": None, "rem": "",
                                 "msg": "Proveedor nuevo, no está en el config" +
                                        (f". Muy parecido a '{parecido[0]}': ¿error de tipeo? Corregilo en "
                                         "UNIFICAR CON del config." if parecido else
                                         ". Cargá razón social, CUIT y precio en el editor.")})
    return por_id, globales



# -----------------------------------------------------------------------------
# 4b. Humedad (planilla de laboratorio "PROVEEDORES - HUMEDAD Y GRANULOMETRIA")
# -----------------------------------------------------------------------------
# Se usan los valores que calcula la propia planilla en la hoja GENERAL, por muestra:
#   Humedad = máx(horno, termobalanza) · Límite de referencia humedad (según el tipo de
#   proveedor: Normal = referencia chipera + 3 %, Especial 1 = 45 %, etc.)
# Descuento por período = promedio de humedad de las muestras del proveedor y producto
# en ese período contra el promedio de sus límites. Si no supera el límite, no hay descuento.
MATERIAL_HUM = {"CHIP": "CHIP", "ASERRIN": "ASERRIN", "RALEO": "Fino/Raleo", "FINO/RALEO": "Fino/Raleo"}
METODOS_HUMEDAD = {"PUNTOS": "TN × puntos de humedad sobre el límite",
                   "BASE_SECA": "TN equivalentes a la humedad límite: TN × (1 − H) / (1 − L)"}


def tipo_archivo(ruta):
    """Reconoce un archivo subido por su CONTENIDO (no por el nombre):
    'maestro', 'calidad', 'config', 'modelo', 'estado' o None."""
    r = str(ruta).lower()
    if r.endswith(".json"):
        return "estado"
    if r.endswith(".xlsb"):
        return "maestro"
    if not r.endswith((".xlsx", ".xlsm")):
        return None
    try:
        wb = load_workbook(ruta, read_only=True)
        hojas = [h.strip().upper() for h in wb.sheetnames]
        if "PROVEEDORES" in hojas and "PRECIOS" in hojas:
            return "config"
        if "DATOS ENTRADA" in hojas:
            return "maestro"
        if "GENERAL" in hojas:
            ws = wb.worksheets[hojas.index("GENERAL")]
            celdas = {str(c).strip() for fila in ws.iter_rows(max_row=15, values_only=True) for c in fila if c}
            if {"PROVEEDOR", "Fecha"} <= celdas and any("Humedad" in c for c in celdas):
                return "calidad"
        texto = " ".join(str(c) for ws in wb.worksheets[:2] for fila in ws.iter_rows(max_row=20, values_only=True)
                         for c in fila if c).upper()
        if "CERTIFICACIÓN ENTREGA BIOMASA" in texto or "CERTIFICACION ENTREGA BIOMASA" in texto:
            return "modelo"
    except Exception:
        return None
    return None


def leer_humedad(ruta, hoja="GENERAL"):
    raw = pd.read_excel(ruta, sheet_name=hoja, header=None, nrows=15)
    fila_h = next(i for i, r in raw.iterrows() if "PROVEEDOR" in r.values and "Fecha" in r.values)
    g = pd.read_excel(ruta, sheet_name=hoja, header=fila_h)
    g.columns = [str(c).strip() for c in g.columns]
    h = pd.DataFrame({
        "Fecha": pd.to_datetime(g["Fecha"], errors="coerce").dt.normalize(),
        "Proveedor": g["PROVEEDOR"].apply(_txt).str.replace(r"\s+", " ", regex=True),
        "Tipo": g.get("Tipo de proveedor", pd.Series("", index=g.index)).apply(_txt),
        "Material": g["Material"].apply(_txt).str.upper().map(lambda m: MATERIAL_HUM.get(m, m)),
        "Remito": g["REMITO"].apply(_txt),
        "Minuto": g["Hora"].apply(_minutos) if "Hora" in g.columns else float("nan"),
        "Humedad": pd.to_numeric(g["Humedad"], errors="coerce"),
        "Limite": pd.to_numeric(g["Limite de referencia humedad"], errors="coerce"),
        "Finos": pd.to_numeric(g["< #04 (%)"], errors="coerce") if "< #04 (%)" in g.columns else float("nan"),
        "FinosAlto": pd.to_numeric(g["Alto"], errors="coerce") if "Alto" in g.columns else float("nan"),
    })
    h.loc[~h["Humedad"].between(0, 1, inclusive="neither"), "Humedad"] = float("nan")
    h.loc[~h["Finos"].between(0, 1), "Finos"] = float("nan")
    h = h.dropna(subset=["Fecha"])
    h = h[(h["Proveedor"] != "") & (h["Humedad"].notna() | h["Finos"].notna())]
    return h.reset_index(drop=True)


def _minutos(v):
    """Hora de la muestra → minutos desde las 0 h (acepta time, datetime, fracción de día o texto hh:mm)."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return float("nan")
    if hasattr(v, "hour"):
        return v.hour * 60 + v.minute
    if isinstance(v, (int, float)):
        return float(v) * 1440 if v < 1 else float("nan")
    m = re.match(r"(\d{1,2})[:.](\d{2})", str(v))
    return int(m.group(1)) * 60 + int(m.group(2)) if m else float("nan")


def descuento_finos(finos_pct, tolerancia=None, totalidad=None):
    """% de finos medido (ej. 11.3) → % que pasa a aserrín, redondeado a 1 decimal como la planilla."""
    tol = FINOS_TOLERANCIA if tolerancia is None else tolerancia
    tot = FINOS_TOTALIDAD if totalidad is None else totalidad
    if finos_pct is not None and not pd.isna(finos_pct):
        finos_pct = round(float(finos_pct), 9)       # como Excel: 14,999999999 cuenta como 15
    if finos_pct is None or pd.isna(finos_pct) or finos_pct <= tol:
        return 0.0
    if finos_pct <= tot:
        return round(finos_pct - tol + 1e-9, 1)
    return round(finos_pct + 1e-9, 1)


def asignar_finos(h, prov, datos, manual=None, manual_h=None):
    """Asigna a cada camión del proveedor su muestra de granulometría.
    1) mismo N° de remito del proveedor; 2) el resto, por día: si hay tantas muestras como camiones,
    en orden (muestras por hora ↔ camiones por hora de entrada a planta); si no, por cercanía de horario
    (muestra ↔ entrada a planta, hasta 90 min). manual: {_id: fracción} cargado en el editor.
    Devuelve {_id: {"F": fracción, "hora": "hh:mm", "origen": texto}}."""
    out = {}
    if h is not None and len(datos):
        provs = list(prov) if isinstance(prov, (list, tuple, set)) else [prov]
        m = h[(h["ProvM"].isin(provs)) & (h["Finos"].notna() | h["Humedad"].notna())].copy()
        if len(m):
            camiones = datos.assign(_dia=datos[C_DIA].dt.normalize(),
                                    _min=pd.to_numeric(datos.get("ENTRADA"), errors="coerce") * 1440,
                                    _n=datos[C_REM_PLANTA].map(_num_remito))
            usadas = set()
            # 1) por remito del proveedor (solo si ese remito es de un único camión: hay proveedores que
            #    repiten el mismo remito en varios viajes)
            rp_cuenta = camiones[C_REM_PROV].astype(str).map(_rem_clave).value_counts()
            m_clave = m["Remito"].astype(str).map(_rem_clave)
            for i, f in camiones.iterrows():
                rp = _rem_clave(f[C_REM_PROV])
                if not rp or rp_cuenta.get(rp, 0) != 1:
                    continue
                cand = m[(m_clave == rp) & (m["Material"] == f[C_TIPO]) & ~m.index.isin(usadas)]
                if len(cand):
                    j = cand.index[0]
                    usadas.add(j)
                    out[f["_id"]] = dict(_muestra(m, j), origen="muestra por N° de remito")
            # 2) por día y orden / horario
            for (dia, prod), cs in camiones[~camiones["_id"].isin(out)].groupby(["_dia", C_TIPO]):
                ms = m[(m["Fecha"] == dia) & (m["Material"] == prod) & ~m.index.isin(usadas)].sort_values("Minuto")
                if not len(ms):
                    continue
                cs = _orden_entrada(cs)
                if len(ms) == len(cs):
                    pares = list(zip(cs.index, ms.index))
                    como = "muestra del día (en orden)"
                else:
                    pares, libres = [], list(ms.index)
                    for i, f in cs.iterrows():
                        if not libres or pd.isna(f["_min"]):
                            continue
                        j = min(libres, key=lambda k: abs((ms.at[k, "Minuto"] if pd.notna(ms.at[k, "Minuto"]) else 1e9) - f["_min"]))
                        if pd.notna(ms.at[j, "Minuto"]) and abs(ms.at[j, "Minuto"] - f["_min"]) <= 90:
                            pares.append((i, j))
                            libres.remove(j)
                    como = "muestra del día (por horario)"
                for i, j in pares:
                    usadas.add(j)
                    out[camiones.at[i, "_id"]] = dict(_muestra(ms, j), origen=como)
    ids = set(datos["_id"])
    for _id, v in (manual or {}).items():
        if _id in ids:
            previo = out.get(_id, {})
            out[_id] = dict(previo, F=None if v is None else float(v), origen="cargado a mano",
                            hora=previo.get("hora", ""))
    for _id, v in (manual_h or {}).items():
        if _id in ids:
            previo = out.get(_id, {"F": None, "hora": "", "L": None})
            out[_id] = dict(previo, H=None if v is None else float(v), origen_h="cargada a mano")
    return out


def _rem_clave(s):
    """Remito del proveedor comparable: '000-465', '0000-465' y '465' son el mismo; S/R → ''."""
    t = str(s).strip().upper()
    if t in SIN_REMITO or t in ("", "NAN", "NONE"):
        return ""
    nums = re.findall(r"\d+", t)
    return str(int(nums[-1])) if nums else t


def _orden_entrada(cs, margen=5):
    """Camiones en orden de entrada a planta; los que entraron con menos de `margen` minutos de diferencia
    se ordenan por N° de remito de planta (la balanza no distingue cuál se muestreó primero)."""
    cs = cs.sort_values(["_min", "_n"])
    grupo, ult, g = [], None, -1
    for v in cs["_min"]:
        if ult is None or pd.isna(v) or pd.isna(ult) or v - ult > margen:
            g += 1
        grupo.append(g)
        ult = v
    return cs.assign(_g=grupo).sort_values(["_g", "_n"])


def _muestra(m, j):
    val = lambda c: None if pd.isna(m.at[j, c]) else float(m.at[j, c])
    return {"F": val("Finos"), "H": val("Humedad"), "L": val("Limite"), "hora": _hhmm(m.at[j, "Minuto"])}


def _hhmm(minuto):
    return "" if minuto is None or pd.isna(minuto) else f"{int(minuto) // 60:02d}:{int(minuto) % 60:02d}"


def _clave(s):
    return re.sub(r"[^a-z0-9]", "", str(s).casefold())


def mapear_humedad(h, d, conf=None):
    """Vincula cada nombre de la planilla de humedad con un proveedor del maestro.
    Orden: columna 'NOMBRE HUMEDAD' del config → nombre igual → nombre contenido →
    remitos de proveedor en común → nombre parecido."""
    provs = sorted(d[C_PROV].unique())
    kp = {_clave(p): p for p in provs}
    manual = {}
    if conf is not None and "NOMBRE HUMEDAD" in conf.columns:
        for p, v in conf["NOMBRE HUMEDAD"].items():
            for n in str(v).split(","):
                if n.strip():
                    manual[_clave(n)] = p
    rp = d[~d[C_REM_PROV].str.upper().isin(SIN_REMITO)][[C_REM_PROV, C_PROV]]
    mapa, filas = {}, []
    for n in sorted(h["Proveedor"].unique()):
        k = _clave(n)
        dest, como = None, ""
        if k in manual:
            dest, como = manual[k], "config (NOMBRE HUMEDAD)"
        elif k in kp:
            dest, como = kp[k], "mismo nombre"
        else:
            cont = [p for kk, p in kp.items() if len(k) >= 5 and (k in kk or kk in k)]
            if len(cont) == 1:
                dest, como = cont[0], "nombre contenido"
            else:
                rems = set(h.loc[(h["Proveedor"] == n) & ~h["Remito"].str.upper().isin(SIN_REMITO), "Remito"])
                cnt = rp[rp[C_REM_PROV].isin(rems)][C_PROV].value_counts()
                if len(cnt) and cnt.iloc[0] >= 3 and cnt.iloc[0] >= 0.5 * len(rems):
                    dest, como = cnt.index[0], f"remitos en común ({cnt.iloc[0]})"
                else:
                    m = difflib.get_close_matches(k, list(kp), n=1, cutoff=0.82)
                    if m:
                        dest, como = kp[m[0]], "nombre parecido"
        mapa[n] = dest
        filas.append({"Nombre en planilla de humedad": n, "Proveedor en maestro": dest or "— sin vincular —",
                      "Cómo se vinculó": como or "no se encontró: cargalo en NOMBRE HUMEDAD del config"})
    h = h.assign(ProvM=h["Proveedor"].map(mapa))
    return h, pd.DataFrame(filas)


def calcular_humedad(h, prov, producto, desde, hasta, tn, metodo="PUNTOS", manual=None,
                     tolerancia=None, limite_fijo=None, aplica_hum=True, aplica_gran=False, tol_finos=None):
    """Descuentos de un producto en un período (promedio de las muestras del laboratorio).
    HUMEDAD (solo si el proveedor tiene DESCUENTA HUMEDAD = SI):
        límite = límite de la planilla (o LIMITE HUMEDAD % fijo) + TOLERANCIA HUMEDAD %
    GRANULOMETRÍA (si DESCUENTA GRANULOMETRIA = SI): % de finos (< #04) sobre TOLERANCIA FINOS %.
    Descuento total = descuento humedad + descuento granulometría (en puntos de % sobre las TN)."""
    manual = manual or {}
    r = {"muestras": 0, "H": None, "L": None, "L_lab": None, "tol": tolerancia or 0.0, "L_fijo": limite_fijo,
         "exceso": 0.0, "desc_hum": 0.0, "aplica_hum": bool(aplica_hum), "origen": "sin muestras",
         "g_muestras": 0, "F": None, "F_tol": tol_finos, "F_ref": None, "desc_gran": 0.0,
         "aplica_gran": bool(aplica_gran), "g_origen": "sin muestras"}
    if h is not None:
        provs = prov if isinstance(prov, (list, tuple, set)) else [prov]
        m = h[(h["ProvM"].isin(provs)) & (h["Material"] == producto) &
              (h["Fecha"].dt.date >= desde) & (h["Fecha"].dt.date <= hasta)]
        mh = m[m["Humedad"].notna()]
        if len(mh):
            r.update(muestras=len(mh), H=float(mh["Humedad"].mean()), L=float(mh["Limite"].mean()),
                     origen=f"promedio de {len(mh)} muestras del laboratorio")
        mg = m[m["Finos"].notna()]
        if len(mg):
            r.update(g_muestras=len(mg), F=float(mg["Finos"].mean()),
                     F_ref=float(mg["FinosAlto"].mean()) if mg["FinosAlto"].notna().any() else None,
                     g_origen=f"promedio de {len(mg)} muestras del laboratorio")
    if manual.get("H") is not None:
        r["H"], r["origen"] = float(manual["H"]), "humedad cargada a mano"
    if manual.get("L") is not None:
        r["L"] = float(manual["L"])
    if manual.get("F") is not None:
        r["F"], r["g_origen"] = float(manual["F"]), "finos cargados a mano"
    r["L_lab"] = r["L"]
    if limite_fijo is not None and manual.get("L") is None:
        r["L"] = limite_fijo
    if r["L"] is not None:
        r["L"] = r["L"] + r["tol"]
    if r["aplica_hum"] and r["H"] is not None and r["L"] is not None and r["H"] > r["L"]:
        r["exceso"] = r["H"] - r["L"]
        r["desc_hum"] = r["exceso"] if metodo == "PUNTOS" else 1 - (1 - r["H"]) / (1 - r["L"])
    if r["aplica_gran"] and r["F"] is not None and r["F_tol"] is not None and r["F"] > r["F_tol"]:
        r["desc_gran"] = r["F"] - r["F_tol"]
    r["desc_pct"] = r["desc_hum"] + r["desc_gran"]
    r["tn"] = round(tn, 2)
    r["tn_pagar"] = round(r["tn"] * (1 - r["desc_pct"]), 2)       # igual que la fórmula del Excel
    r["tn_desc"] = round(r["tn"] - r["tn_pagar"], 2)
    r["tn_desc_hum"] = round(r["tn"] * r["desc_hum"], 2)
    r["tn_desc_gran"] = round(r["tn_desc"] - r["tn_desc_hum"], 2)
    return r


# -----------------------------------------------------------------------------
# 5. Armado de certificaciones (en memoria)
# -----------------------------------------------------------------------------
def periodos(anio, mes, tipo="ambas"):
    """tipo: lista de períodos (ej. ["1Q", "Mensual"]) o un texto ("ambas", "1Q", "2Q+Mensual"...)."""
    ult = calendar.monthrange(anio, mes)[1]
    if isinstance(tipo, (list, tuple, set)):
        todos = {"1Q": (date(anio, mes, 1), date(anio, mes, DIA_CORTE)),
                 "2Q": (date(anio, mes, DIA_CORTE + 1), date(anio, mes, ult)),
                 "Mensual": (date(anio, mes, 1), date(anio, mes, ult))}
        return [(k, *todos[k]) for k in ("1Q", "2Q", "Mensual") if k in tipo]
    p = []
    if tipo in ("quincenal", "ambas", "1Q"):
        p.append(("1Q", date(anio, mes, 1), date(anio, mes, DIA_CORTE)))
    if tipo in ("quincenal", "ambas", "2Q+Mensual", "2Q"):
        p.append(("2Q", date(anio, mes, DIA_CORTE + 1), date(anio, mes, ult)))
    if tipo in ("mensual", "ambas", "2Q+Mensual"):
        p.append(("Mensual", date(anio, mes, 1), date(anio, mes, ult)))
    return p


# Formato GYG: humedad y granulometría camión por camión (planilla "Certificación GYG")
GYG_COLS = ["Hum", "HumRef", "Pen", "KgChip", "KgAse", "KgCorto", "KgLargo", "KgRaleo"]
GYG_OTROS = {"C CORTO": "KgCorto", "C LARGO": "KgLargo", "Fino/Raleo": "KgRaleo"}


def limites_por_dia(h):
    """Humedad de referencia del día (límite de la planilla del laboratorio, el más repetido de ese día)."""
    if h is None:
        return {}
    x = h.dropna(subset=["Limite"])
    return x.groupby(x["Fecha"].dt.date)["Limite"].agg(lambda s: float(s.round(4).mode().iloc[0])).to_dict()


def calcular_gyg(filas, muestras, alto, muy_alto, ref_dia, tol_hum=0.0, sin_gran=False, sin_hum=False,
                 man_desc=None, man_pen=None):
    """Como las fórmulas de la planilla de GYG, por camión:
       penalización = humedad medida − humedad de referencia (si la supera; solo CHIP y ASERRÍN)
       aserrín (CHIP): % < Alto → 0 · % < Muy Alto → (% − Alto) × kg · si no → % × kg ; × (1 − penalización)
       chip = (kg − aserrín) × (1 − penalización) · ASERRÍN = kg × (1 − penalización) · el resto, kg completos.
       sin_gran / sin_hum: descuento quitado a mano en la certificación; man_desc {_id: % aserrín} y
       man_pen {_id: fracción} reemplazan el valor calculado de ese camión."""
    man_desc, man_pen = man_desc or {}, man_pen or {}
    out = {k: [] for k in GYG_COLS + ["Finos", "FinosHora", "FinosOrigen", "DescPct", "DescKg", "KilosNeto",
                                      "DescManual", "PenManual"]}
    for _, f in filas.iterrows():
        mu = muestras.get(f["_id"], {})
        kg, prod = float(f["Kilos"]), f["Producto"]
        H, F = mu.get("H"), mu.get("F")
        ref = mu.get("L") if mu.get("L") is not None else ref_dia.get(f["Fecha"])
        ref = None if ref is None else ref + (tol_hum or 0.0)
        pen = H - ref if (prod in ("CHIP", "ASERRIN") and H is not None and ref is not None
                          and round(H, 12) > round(ref, 12)) else 0.0
        pen_man = prod in ("CHIP", "ASERRIN") and f["_id"] in man_pen
        if sin_hum:
            pen = 0.0
        elif pen_man:
            pen = float(man_pen[f["_id"]] or 0.0)
        desc_man = prod == "CHIP" and f["_id"] in man_desc
        chip = ase = 0.0
        otros = {v: 0.0 for v in GYG_OTROS.values()}
        exceso = 0.0
        if prod == "CHIP":
            Fv = round(F or 0.0, 12)          # como Excel (15 cifras): 0,14999999999999997 cuenta como 15 %
            exceso = 0.0 if Fv < alto else ((Fv - alto) if Fv < muy_alto else Fv)
            if sin_gran:
                exceso = 0.0
            elif desc_man:
                exceso = float(man_desc[f["_id"]] or 0.0) / 100
            ase = exceso * kg * (1 - pen)
            chip = (kg - ase) * (1 - pen)
        elif prod == "ASERRIN":
            ase = kg * (1 - pen)
        elif prod in GYG_OTROS:
            otros[GYG_OTROS[prod]] = kg
        for k, v in (("Hum", H), ("HumRef", ref), ("Pen", pen), ("KgChip", chip), ("KgAse", ase)):
            out[k].append(v)
        for k, v in otros.items():
            out[k].append(v)
        out["Finos"].append(F if prod == "CHIP" else None)
        out["FinosHora"].append(mu.get("hora", ""))
        out["FinosOrigen"].append(mu.get("origen", "") + (" · humedad cargada a mano" if mu.get("origen_h") else ""))
        out["DescPct"].append(exceso * 100)
        out["DescKg"].append(round(ase, 2) if prod == "CHIP" else 0.0)
        out["KilosNeto"].append(chip if prod == "CHIP" else (ase if prod == "ASERRIN" else kg))
        out["DescManual"].append(desc_man)
        out["PenManual"].append(pen_man)
    for k, v in out.items():
        filas[k] = v
    tn = {"CHIP": filas["KgChip"].sum() / 1000, "ASERRIN": filas["KgAse"].sum() / 1000}
    for prod, col in GYG_OTROS.items():
        tn[prod] = filas[col].sum() / 1000
    for prod in set(filas["Producto"]) - set(tn):
        tn[prod] = filas.loc[filas["Producto"] == prod, "Kilos"].sum() / 1000
    return {t: v for t, v in tn.items() if v > 0}


def calcular_estandar(filas, muestras, ip, tol_f, ref_dia, sin_gran=False, sin_hum=False, man_desc=None, man_pen=None):
    """Formato de la certificación final (todos menos el formato GYG), camión por camión:
       Desc. humedad % = humedad medida − referencia del día (solo si DESCUENTA HUMEDAD = SI; CHIP y ASERRÍN)
       Desc. granulometría % = finos por encima de la tolerancia (CHIP; hasta 15 % la diferencia, más, todo)
       Kilos a pagar = KG × (1 − hum.) × (1 − gran.) · Aserrín = KG × (1 − hum.) × gran.
       Ambos % se pueden poner a mano (por camión) o quitar en toda la certificación."""
    man_desc, man_pen = man_desc or {}, man_pen or {}
    tol_h = ip["tolerancia"] if isinstance(ip["tolerancia"], float) else 0.0
    lim_fijo = ip["limite"] if isinstance(ip["limite"], float) else None
    cols = {k: [] for k in ("Finos", "FinosHora", "FinosOrigen", "Hum", "HumRef", "Pen", "PenManual",
                            "DescPct", "DescManual", "DescKg", "KilosNeto")}
    for _, f in filas.iterrows():
        mu = muestras.get(f["_id"], {})
        prod, kg, i = f["Producto"], float(f["Kilos"]), f["_id"]
        F, H = mu.get("F"), mu.get("H")
        ref = lim_fijo if lim_fijo is not None else (mu.get("L") if mu.get("L") is not None else ref_dia.get(f["Fecha"]))
        ref = None if ref is None else ref + tol_h
        pen = 0.0
        if ip["desc_hum"] and prod in ("CHIP", "ASERRIN") and H is not None and ref is not None \
                and round(H, 12) > round(ref, 12):
            pen = H - ref
        pen_man = i in man_pen and prod in ("CHIP", "ASERRIN")
        if sin_hum:
            pen, pen_man = 0.0, False
        elif pen_man:
            pen = float(man_pen[i] or 0.0)
        auto = (descuento_finos(None if F is None or pd.isna(F) else F * 100, tol_f)
                if prod == "CHIP" and ip["desc_gran"] else 0.0)
        desc_man = prod == "CHIP" and i in man_desc and not sin_gran
        dp = 0.0 if sin_gran else (float(man_desc[i] or 0.0) if desc_man else auto)
        ase = round(kg * (1 - pen) * dp / 100, 2)
        for k, v in (("Finos", F), ("FinosHora", mu.get("hora", "")),
                     ("FinosOrigen", mu.get("origen", "") + (" · humedad cargada a mano" if mu.get("origen_h") else "")),
                     ("Hum", H), ("HumRef", ref), ("Pen", pen), ("PenManual", pen_man), ("DescPct", dp),
                     ("DescManual", desc_man), ("DescKg", ase), ("KilosNeto", kg * (1 - pen) - ase)):
            cols[k].append(v)
    for k, v in cols.items():
        filas[k] = v
    tn_prod = (filas.groupby("Producto")["KilosNeto"].sum() / 1000).to_dict()
    if filas["DescKg"].sum() > 0:
        tn_prod["ASERRIN"] = tn_prod.get("ASERRIN", 0) + filas["DescKg"].sum() / 1000
    return tn_prod


def construir(d, d_orig, conf, precios, estado, tipo="ambas", df_raw=None, proveedores=None,
              humedad=None, metodo_humedad="PUNTOS", solo=None, validacion=None):
    """solo: conjunto de claves 'período|proveedor' a armar (None = todas).
    validacion: (por_id, globales) ya calculados, para no repetir los controles del mes."""
    anio, mes = estado["anio"], estado["mes"]
    ini, fin = pd.Timestamp(anio, mes, 1), pd.Timestamp(anio, mes, 1) + pd.offsets.MonthEnd(0)
    mes_df = d[(d[C_DIA] >= ini) & (d[C_DIA] <= fin)].copy()
    if validacion is None:
        validacion = validar_maestro(df_raw if df_raw is not None else d, d, conf, anio, mes)
    por_id, globales = validacion
    obs_cfg = validar_config(conf, precios)
    mes_df["_grupo"] = mes_df[C_PROV].map(lambda p: info_proveedor(conf, p)["grupo"])
    mes_df["_cert"] = mes_df[C_PROV].map(lambda p: unidad(conf, p))
    if "_prov_lab" not in mes_df.columns:
        mes_df["_prov_lab"] = mes_df[C_PROV]
    mes_df["_prov_lab"] = mes_df["_prov_lab"].fillna(mes_df[C_PROV]).where(mes_df["_prov_lab"].astype(str) != "",
                                                                           mes_df[C_PROV])

    certs = {}
    for per, desde, hasta in periodos(anio, mes, tipo):
        sel = mes_df[(mes_df[C_DIA].dt.date >= desde) & (mes_df[C_DIA].dt.date <= hasta)]
        for prov, datos in sorted(sel.groupby("_cert"), key=lambda x: x[0].casefold()):
            puntos_ent = sorted(datos[C_PROV].unique(), key=str.casefold)
            if proveedores and not set(puntos_ent) & set(proveedores) and prov not in proveedores:
                continue
            key = f"{per}|{prov}"
            if solo is not None and key not in solo:
                continue
            con_cuit = [p for p in puntos_ent if info_proveedor(conf, p)["cuit"]]
            ip = info_proveedor(conf, (con_cuit or puntos_ent)[0])
            if not ip["certifica"]:
                continue
            meta = estado["meta"].get(key, {})
            sin_gran, sin_hum = bool(meta.get("sin_gran")), bool(meta.get("sin_hum"))
            man_desc = estado["corr"].get("desc_gran") or {}
            man_pen = estado["corr"].get("pen_hum") or {}
            datos = datos.sort_values([C_DIA, C_REM_PLANTA])
            filas = pd.DataFrame({k: datos[CAMPOS[k]].values for k in FILA_COLS})
            # punto de entrega tal como figura en el maestro (ej. un punto de entrega dentro de otra certificación)
            filas["Proveedor"] = datos["_prov_lab"].values
            filas["Fecha"] = pd.to_datetime(filas["Fecha"]).dt.date
            filas["_id"] = datos["_id"].values
            # Granulometría por camión: lo descontado pasa a ASERRÍN (como la planilla de certificación final)
            tol_f = ip["tol_finos"] * 100 if isinstance(ip["tol_finos"], float) else None
            es_gyg = ip["formato"] == "GYG"
            finos = {}
            if humedad is not None:            # muestras de cada camión: se muestran siempre (humedad y finos %)
                con_lab = set(humedad["ProvM"].dropna())
                for pe in sorted(set(datos["_prov_lab"])):
                    # todos los camiones del mes registrados con ese nombre (aunque se certifiquen en otra
                    # certificación, ej. dos proveedores separados por patente): el orden del día no cambia
                    ctx = mes_df[mes_df["_prov_lab"] == pe]
                    finos.update(asignar_finos(humedad, pe, ctx, estado["corr"].get("finos"),
                                               estado["corr"].get("humedad_camion")))
                # puntos de entrega sin nombre propio en la planilla del laboratorio (ej. un segundo punto de entrega):
                # usan las muestras de los otros puntos de la misma certificación
                sin_lab = [pe for pe in set(datos["_prov_lab"]) if pe not in con_lab]
                con_lab_u = [pe for pe in miembros(conf, prov) if pe in con_lab]      # todos los puntos de la unidad
                if sin_lab and con_lab_u:
                    finos.update(asignar_finos(humedad, con_lab_u,
                                               datos[datos["_prov_lab"].isin(sin_lab)], estado["corr"].get("finos"),
                                               estado["corr"].get("humedad_camion")))
            if es_gyg:
                alto = ip["tol_finos"] if isinstance(ip["tol_finos"], float) else None
                if alto is None and humedad is not None:
                    fa = humedad.loc[humedad["ProvM"].isin(puntos_ent), "FinosAlto"].dropna()
                    alto = float(fa.mode().iloc[0]) if len(fa) else None
                alto = FINOS_TOLERANCIA / 100 if alto is None else alto
                tol_f = alto * 100
                tn_prod = calcular_gyg(filas, finos, alto, FINOS_TOTALIDAD / 100, limites_por_dia(humedad),
                                       ip["tolerancia"] if isinstance(ip["tolerancia"], float) else 0.0,
                                       sin_gran, sin_hum, man_desc, man_pen)
            else:
                tn_prod = calcular_estandar(filas, finos, ip, tol_f, limites_por_dia(humedad),
                                            sin_gran, sin_hum, man_desc, man_pen)
            orden = {t: i for i, t in enumerate(TIPOLOGIAS)}
            tipos = sorted([t for t, v in tn_prod.items() if v > 0], key=lambda t: orden.get(t, 99))
            tn_auto = mes_df[(mes_df["_grupo"] == ip["grupo"]) & (mes_df[C_DIA].dt.date <= hasta)][C_NETO].sum() / 1000
            tn_grupo = float(meta.get("tn_grupo", tn_auto))
            precios_c, origen_p = {}, {}
            for t in tipos:
                if t in meta.get("precios", {}):
                    precios_c[t], origen_p[t] = float(meta["precios"][t]), "manual (solo esta certificación)"
                else:
                    precios_c[t], origen_p[t] = buscar_precio(precios, ip["grupo"], t, tn_grupo), "config"

            hum = {}        # la humedad ahora se descuenta camión por camión (columna Desc. humedad %)

            obs = []
            if humedad is not None:
                ult_m = humedad["Fecha"].max().date()
                if ult_m < hasta:
                    obs.append({"sev": ADV, "msg": f"La planilla de humedad tiene muestras hasta el {ult_m:%d/%m/%Y} "
                                f"y el período termina el {hasta:%d/%m/%Y}: subí la planilla actualizada",
                                "id": "", "rem": "", "bloq": False})
                for t, hh in hum.items():
                    if hh["aplica_hum"] and hh["muestras"] == 0 and hh["origen"] == "sin muestras" and t in ("CHIP", "ASERRIN"):
                        obs.append({"sev": ADV, "msg": f"Sin muestras de humedad de {t} en el período: no se descuenta",
                                    "id": "", "rem": "", "bloq": False})
                    if hh["origen"] == "humedad cargada a mano":
                        obs.append({"sev": ADV, "msg": f"Humedad de {t} cargada a mano ({decimal(hh['H']*100, 1)} %)",
                                    "id": "", "rem": "", "bloq": False})
            if sin_gran:
                obs.append({"sev": ADV, "msg": "Descuento de granulometría QUITADO a mano en esta certificación",
                            "id": "", "rem": "", "bloq": False})
            if sin_hum:
                obs.append({"sev": ADV, "msg": "Descuento de humedad QUITADO a mano en esta certificación",
                            "id": "", "rem": "", "bloq": False})
            for _, f in filas[filas["DescManual"]].iterrows() if not sin_gran else []:
                obs.append({"sev": ADV, "msg": f"Descuento de granulometría puesto a mano: {decimal(f['DescPct'], 2)} %",
                            "id": f["_id"], "rem": f["Rto. Planta"], "bloq": False})
            for _, f in filas[filas["PenManual"]].iterrows() if not sin_hum else []:
                obs.append({"sev": ADV, "msg": f"Descuento de humedad puesto a mano: {pct(f['Pen'], 2)}",
                            "id": f["_id"], "rem": f["Rto. Planta"], "bloq": False})
            for _, f in filas[filas["Pen"] > 0].iterrows() if not es_gyg else []:
                obs.append({"sev": ADV, "msg": f"Humedad {pct(f['Hum'], 1) if pd.notna(f['Hum']) else '—'} sobre la "
                            f"referencia → −{pct(f['Pen'], 2)} de los kilos", "id": f["_id"], "rem": f["Rto. Planta"],
                            "bloq": False})
            if es_gyg:
                for _, f in filas[filas["Producto"].isin(["CHIP", "ASERRIN"])].iterrows():
                    if f["Hum"] is None or pd.isna(f["Hum"]):
                        obs.append({"sev": ADV, "msg": "Camión sin muestra de humedad/granulometría asignada: "
                                    "no se le descuenta (se puede cargar a mano)", "id": f["_id"],
                                    "rem": f["Rto. Planta"], "bloq": False})
                for _, f in filas[filas["FinosOrigen"].str.contains("a mano")].iterrows():
                    obs.append({"sev": ADV, "msg": "Valores de laboratorio cargados a mano", "id": f["_id"],
                                "rem": f["Rto. Planta"], "bloq": False})
            elif ip["desc_gran"]:
                chip = filas[filas["Producto"] == "CHIP"]
                if humedad is not None and len(chip) and chip["Finos"].isna().all():
                    obs.append({"sev": ADV, "msg": "Ningún camión de CHIP con muestra de granulometría en el período",
                                "id": "", "rem": "", "bloq": False})
                for _, f in filas[filas["FinosOrigen"] == "cargado a mano"].iterrows():
                    obs.append({"sev": ADV, "msg": f"Finos cargados a mano: {pct(f['Finos'], 1)}", "id": f["_id"],
                                "rem": f["Rto. Planta"], "bloq": False})
                for _, f in filas[filas["DescPct"] > 0].iterrows():
                    obs.append({"sev": ADV, "msg": f"Granulometría: finos {decimal(f['Finos'] * 100, 1)} % → "
                                f"{decimal(f['DescPct'], 1)} % ({decimal(f['DescKg'], 1)} kg) pasa a aserrín",
                                "id": f["_id"], "rem": f["Rto. Planta"], "bloq": False})
            for campo, val in (("TOLERANCIA HUMEDAD %", ip["tolerancia"]), ("LIMITE HUMEDAD %", ip["limite"]),
                               ("TOLERANCIA FINOS %", ip["tol_finos"])):
                if val == "ERROR" or (isinstance(val, float) and not (0 <= val < (0.2 if "HUMEDAD" in campo and "TOL" in campo else 1))):
                    obs.append({"sev": CRIT, "msg": f"{campo} del proveedor mal cargado en el config", "id": "",
                                "rem": "", "bloq": True})
            if humedad is not None and isinstance(ip["tolerancia"], float) and ip["tolerancia"] > 0 and ip["desc_hum"]:
                obs.append({"sev": ADV, "msg": f"Tolerancia de humedad pactada: +{decimal(ip['tolerancia'] * 100, 1)} puntos "
                            "sobre el límite", "id": "", "rem": "", "bloq": False})
            if humedad is not None and isinstance(ip["limite"], float) and ip["desc_hum"]:
                obs.append({"sev": ADV, "msg": f"Límite de humedad fijo pactado: {decimal(ip['limite'] * 100, 1)} % "
                            "(reemplaza al de la planilla)", "id": "", "rem": "", "bloq": False})
            for i, rem in zip(filas["_id"], filas["Rto. Planta"]):
                for sev, msg, bloq in por_id.get(i, []):
                    obs.append({"sev": sev, "msg": msg, "id": i, "rem": rem, "bloq": bloq})
            if not ip["cuit"]:
                obs.append({"sev": ADV, "msg": "Falta CUIT del proveedor (sale en blanco: cargalo en el editor)",
                            "id": "", "rem": "", "bloq": False})
            elif not cuit_valido(ip["cuit"]):
                obs.append({"sev": CRIT, "msg": f"CUIT '{ip['cuit']}' inválido (dígito verificador)",
                            "id": "", "rem": "", "bloq": False})
            for t in tipos:
                if precios_c[t] is None:
                    obs.append({"sev": ADV, "msg": f"Falta precio de {t} (sale en blanco: cargalo en el editor o en el "
                                f"Excel de la certificación)", "id": "", "rem": "", "bloq": False})
            for sev, g, msg in obs_cfg:
                if str(g).casefold() == str(ip["grupo"]).casefold():
                    obs.append({"sev": sev, "msg": msg, "id": "", "rem": "", "bloq": False})
            for gl in globales:
                if gl["prov"] in puntos_ent or gl["prov"] == prov:
                    obs.append({"sev": gl["sev"], "msg": gl["msg"], "id": "", "rem": gl["rem"], "bloq": False})
            if abs(tn_grupo - tn_auto) > 0.005:
                obs.append({"sev": ADV, "msg": f"TN del grupo cargado a mano ({decimal(tn_grupo)}); "
                            f"según remitos es {decimal(tn_auto)}", "id": "", "rem": "", "bloq": False})
            for o in obs:                     # nada bloquea: los datos que faltan se completan después
                o["bloq"] = False

            c = {"key": key, "periodo": per, "desde": desde, "hasta": hasta, "proveedor": prov, "miembros": puntos_ent,
                 "razon": ip["razon"], "cuit": ip["cuit"], "grupo": ip["grupo"], "filas": filas,
                 "tn": round(filas["Kilos"].sum() / 1000, 2), "tn_grupo": tn_grupo, "tn_grupo_auto": tn_auto,
                 "precios": precios_c, "origen_precios": origen_p, "obs": obs, "humedad": hum, "tn_prod": tn_prod,
                 "tol_finos": tol_f if tol_f is not None else FINOS_TOLERANCIA,
                 "desc_gran": ip["desc_gran"] and not sin_gran, "sin_gran": sin_gran, "sin_hum": sin_hum,
                 "formato": "GYG" if es_gyg else "", "muy_alto": FINOS_TOTALIDAD,
                 "con_humedad": humedad is not None, "metodo_humedad": metodo_humedad,
                 "numero": meta.get("numero", ""),
                 "fecha": date.fromisoformat(meta["fecha"]) if meta.get("fecha") else hasta + timedelta(days=1)}
            c["tramos"] = armar_tramos(c, precios)
            if es_gyg:
                for ft in c["tramos"]:
                    ft["etiqueta"] = ft["etiqueta"].replace(nombre_producto(ft["producto"]),
                                                            etiqueta_gyg(ft["producto"]), 1)
            c["importe"] = importe(c)
            c["estado"] = estado_cert(c, meta)
            certs[key] = c
    globales_sueltos = [g for g in globales if not any(g["prov"] in c["miembros"] for c in certs.values())
                        and (not proveedores or g["prov"] in proveedores)]
    if validacion is not None and solo is None:
        pass
    construir.ultima_validacion = (por_id, globales)
    return certs, globales_sueltos, obs_cfg


def pct(x, n=2):
    return "—" if x is None else decimal(x * 100, n) + " %"


def _txt_limite(hh):
    if hh["L"] is None:
        return "—"
    base = hh["L"] - hh["tol"]
    txt = decimal(base * 100, 1) + " %"
    if hh["tol"]:
        txt += f" + {decimal(hh['tol'] * 100, 1)} = {decimal(hh['L'] * 100, 1)} %"
    return txt


def tn_pagar(c, t):
    """TN a pagar de un producto: después de la reclasificación por granulometría y del descuento de humedad."""
    if c.get("formato") == "GYG":
        return c.get("tn_prod", {}).get(t, 0.0)          # sin redondear, como la planilla de GYG
    h = c.get("humedad", {}).get(t)
    if h:
        return h["tn_pagar"]
    return round(c.get("tn_prod", {}).get(t, 0.0), 2)


def orden_productos(tipos):
    """Tipologías en el orden del cuadro de precios (CHIP, ASERRIN, C. CORTO, C. LARGO, FINO/RALEO, otras)."""
    orden = {t: i for i, t in enumerate(PRODUCTOS_CUADRO + [t for t in TIPOLOGIAS if t not in PRODUCTOS_CUADRO])}
    lista = [t for t, v in tipos.items() if v] if isinstance(tipos, dict) else list(tipos)
    return sorted(lista, key=lambda t: (orden.get(t, 99), str(t)))


def nombre_producto(t):
    return NOMBRE_PRODUCTO.get(t, t)


def _etiqueta_tramo(desde, hasta):
    desde = 0 if pd.isna(desde) else desde
    if desde == 0 and pd.notna(hasta):
        return f"-{hasta:g}TN"
    if desde > 0:
        return f"+{desde:g}TN"
    return ""


def armar_tramos(c, precios):
    """Filas del cuadro de precios como la planilla: CHIP, ASERRIN, C. CORTO, C. LARGO, FINO/RALEO (y lo que se
    haya entregado), con todos los tramos pactados del grupo; las TN van solo en el tramo que corresponde."""
    filas = []
    del_grupo = precios[precios["GRUPO"].str.casefold() == str(c["grupo"]).casefold()]
    orden = {t: i for i, t in enumerate(PRODUCTOS_CUADRO + [t for t in TIPOLOGIAS if t not in PRODUCTOS_CUADRO])}
    productos = sorted(set(PRODUCTOS_CUADRO) | set(c["precios"]), key=lambda t: orden.get(t, 99))
    for t in productos:
        entregado = t in c["precios"]
        tn_t = tn_pagar(c, t) if entregado else None
        p = del_grupo[del_grupo["TIPOLOGIA"].astype(str).str.casefold() == t.casefold()].sort_values("DESDE_TN")
        manual = entregado and c["origen_precios"].get(t, "").startswith("manual")
        if len(p) <= 1 or manual:
            precio = c["precios"][t] if entregado else (float(p["PRECIO"].iloc[0]) if len(p) else None)
            filas.append({"producto": t, "etiqueta": nombre_producto(t), "precio": precio, "tn": tn_t})
            continue
        for _, r in p.iterrows():
            desde = r["DESDE_TN"] if pd.notna(r["DESDE_TN"]) else 0
            hasta = r["HASTA_TN"] if pd.notna(r["HASTA_TN"]) else float("inf")
            aplica = entregado and desde <= c["tn_grupo"] < hasta
            filas.append({"producto": t, "etiqueta": f"{nombre_producto(t)} {_etiqueta_tramo(r['DESDE_TN'], r['HASTA_TN'])}",
                          "precio": float(r["PRECIO"]) if pd.notna(r["PRECIO"]) else None,
                          "tn": tn_t if aplica else None})
    return filas


def con_tramos(c):
    return any("TN" in f["etiqueta"].split()[-1] for f in c.get("tramos", []))


def mostrar_humedad(c):
    """El cuadro de humedad se muestra solo si al proveedor se le descuenta humedad y hay algo para informar."""
    return c.get("con_humedad") and any(hh["aplica_hum"] and (hh["muestras"] or hh["desc_hum"])
                                        for hh in c["humedad"].values())


def importe(c):
    """Importe a facturar; un producto sin precio cuenta 0 (se completa en el editor o en el Excel)."""
    tot = 0.0
    for t, p in c["precios"].items():
        tot += tn_pagar(c, t) * (p or 0.0)
    return round(tot, 2)


def huella(c):
    """Huella del contenido: si algo cambia después de confirmar, se detecta."""
    extra = ["DescKg"] + (["Hum", "Finos", "Pen"] if c.get("formato") == "GYG" else [])
    f = c["filas"][FILA_COLS + extra].astype(str).values.tolist()
    data = [c["razon"], c["cuit"], c["numero"], str(c["fecha"]), round(c["tn_grupo"], 2),
            sorted((k, v) for k, v in c["precios"].items()), f,
            sorted((k, round(v["desc_hum"], 6), round(v["desc_gran"], 6)) for k, v in c.get("humedad", {}).items())]
    return hashlib.md5(json.dumps(data, default=str).encode()).hexdigest()


def criticas_revisables(c):
    return sorted({o["msg"] for o in c["obs"] if o["sev"] == CRIT and not o["bloq"]})


def estado_cert(c, meta):
    if meta.get("huella"):
        return "CONFIRMADA" if meta["huella"] == huella(c) else "MODIFICADA"
    if any(o["bloq"] for o in c["obs"]):
        return "BLOQUEADA"
    pend = set(criticas_revisables(c)) - set(meta.get("ack", {}).get("msgs", []))
    return "REVISAR" if pend else "LISTA"


ICONOS = {"CONFIRMADA": "✅", "MODIFICADA": "🔁", "BLOQUEADA": "⛔", "REVISAR": "⚠️", "LISTA": "🟢"}


def cuadre(d_orig, d, certs, conf, anio, mes, proveedores=None):
    ini = pd.Timestamp(anio, mes, 1)
    fin = ini + pd.offsets.MonthEnd(0)
    filas = []
    for per in sorted({c["periodo"] for c in certs.values()}, key=["1Q", "2Q", "Mensual"].index):
        def base(x):
            x = x[(x[C_DIA] >= ini) & (x[C_DIA] <= fin)]
            if per == "1Q":
                x = x[x[C_DIA].dt.day <= DIA_CORTE]
            elif per == "2Q":
                x = x[x[C_DIA].dt.day > DIA_CORTE]
            return x[x[C_PROV].isin(proveedores)] if proveedores else x
        bo, bc = base(d_orig), base(d)
        excl = bc[~bc[C_PROV].map(lambda p: info_proveedor(conf, p)["certifica"])]
        tn_o, tn_c = bo[C_NETO].sum() / 1000, bc[C_NETO].sum() / 1000
        tn_cert = sum(c["filas"]["Kilos"].sum() for c in certs.values() if c["periodo"] == per) / 1000
        tn_e = excl[C_NETO].sum() / 1000
        dif = round(tn_c - tn_cert - tn_e, 2)
        filas.append({"Período": per, "TN maestro": round(tn_o, 2),
                      "Ajustes manuales": round(tn_c - tn_o, 2), "TN maestro corregido": round(tn_c, 2),
                      "TN certificadas": round(tn_cert, 2), "TN excluidas (CERTIFICAR=NO)": round(tn_e, 2),
                      "Diferencia": dif, "Resultado": "✔ CUADRA" if abs(dif) < 0.01 else "✖ NO CUADRA"})
    return pd.DataFrame(filas)


# -----------------------------------------------------------------------------
# 6. Formatos de número (estilo argentino)
# -----------------------------------------------------------------------------
def _ar(s):
    return s.replace(",", "X").replace(".", ",").replace("X", ".")


def miles(x):
    return _ar(f"{float(x):,.0f}")


def decimal(x, n=2):
    return _ar(f"{float(x):,.{n}f}")


def pesos(x):
    return "$ " + decimal(x) if x is not None else "—"


def texto_periodo(desde, hasta):
    if desde.day == 1 and hasta.day == calendar.monthrange(hasta.year, hasta.month)[1]:
        return f"Período: {MESES[desde.month]} {desde.year} (del 01 al {hasta.day:02d})"
    return f"Período del {desde.day} al {hasta.day} de {MESES[desde.month]} {desde.year}"


def etiqueta_periodo(per):
    return {"1Q": "Quincenal", "2Q": "Quincenal", "Mensual": "Mensual"}[per]


# -----------------------------------------------------------------------------
# 7. PDF de la certificación
# -----------------------------------------------------------------------------
def kilos_txt(x):
    """16.400 · 11.304,5 (como la planilla)."""
    x = float(x)
    return miles(x) if abs(x - round(x)) < 1e-6 else decimal(x, 1)


def _con_desc(f):
    """El camión tiene descuento (granulometría o, en GYG, humedad)."""
    pen = f.get("Pen", 0) if hasattr(f, "get") else 0
    return f["DescKg"] > 0 or (pen is not None and not pd.isna(pen) and pen > 0)


def texto_granulometria(f):
    partes = []
    if f["DescKg"] > 0:
        if f.get("DescManual"):
            partes.append(f"Desc. granulometría a mano {decimal(f['DescPct'], 1)}%")
        elif f["Finos"] is not None and not pd.isna(f["Finos"]):
            finos = f["Finos"] * 100
            partes.append(f"Según granulometría CHIP {decimal(100 - finos, 1)} // ASE {decimal(finos, 1)}% "
                          f"(Desc. {decimal(f['DescPct'], 1)}%)")
        else:
            partes.append(f"Desc. granulometría {decimal(f['DescPct'], 1)}%")
    pen = f.get("Pen", 0)
    if pen is not None and not pd.isna(pen) and pen > 0:
        if f.get("PenManual") or f.get("Hum") is None or pd.isna(f.get("Hum")):
            partes.append(f"Desc. humedad {decimal(pen * 100, 1)}%")
        else:
            partes.append(f"Humedad {decimal(f['Hum'] * 100, 1)}% (ref. {decimal(f['HumRef'] * 100, 1)}%) "
                          f"Desc. {decimal(pen * 100, 1)}%")
    return " · ".join(partes)


def nota_granulometria(c):
    t = (f"Granulometría (finos &lt; #04, por camión): hasta {decimal(c['tol_finos'], 0)} % sin descuento; "
         f"de {decimal(c['tol_finos'], 0)} % a {decimal(FINOS_TOTALIDAD, 0)} % se descuenta la diferencia; "
         f"desde {decimal(FINOS_TOTALIDAD, 0)} % la totalidad. Lo descontado se paga como aserrín.")
    if c.get("formato") == "GYG" and not c.get("sin_hum"):
        t += (" Humedad (por camión): si supera la referencia del día se descuenta el porcentaje con el que la "
              "supera, sobre los kilos de chip y aserrín.")
    return t


def etiqueta_producto(c):
    return etiqueta_gyg if c.get("formato") == "GYG" else nombre_producto


def tn_detalle(c):
    """TN de la columna Kilos del detalle (después del descuento de humedad por camión, si lo hay)."""
    fl = c["filas"]
    return round((fl["KilosNeto"].sum() + fl["DescKg"].sum()) / 1000, 2)


def pdf_certificacion(ruta, c, logo=None):
    if c.get("formato") == "GYG":          # GYG tiene su propio formato (planilla de GYG)
        return pdf_gyg(ruta, c, logo)
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.lib.utils import ImageReader
    from reportlab.platypus import Image, KeepTogether, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

    verde, amarillo, gris = colors.HexColor("#92D050"), colors.HexColor("#FFFF00"), colors.HexColor("#808080")
    amarillo_claro, azul, rojo = colors.HexColor("#FFF2A8"), colors.HexColor("#2E75B6"), colors.HexColor("#C00000")

    def P(t, size=9, bold=False, align=0, color=colors.black):
        return Paragraph(t, ParagraphStyle("p", fontName="Helvetica-Bold" if bold else "Helvetica",
                                           fontSize=size, leading=size * 1.25, alignment=align, textColor=color))

    ancho = A4[0] - 24 * mm
    el = []
    if logo:
        iw, ih = ImageReader(io.BytesIO(logo)).getSize()
        img = Image(io.BytesIO(logo), width=32 * mm, height=32 * mm * ih / iw)
        img.hAlign = "LEFT"
        el.append(img)
    el += [P(l, 9) for l in EMPRESA_LINEAS]
    num = Table([["", P(f"Nº {c['numero']}", 10, True, 1)]], colWidths=[ancho - 55 * mm, 55 * mm])
    num.setStyle(TableStyle([("BOX", (1, 0), (1, 0), 1.2, colors.black)]))
    el += [Spacer(1, 1 * mm), num, Spacer(1, 2 * mm), P("CERTIFICACIÓN ENTREGA BIOMASA", 12, True, 1, azul),
           Spacer(1, 3 * mm)]
    enc = Table([[P(texto_periodo(c["desde"], c["hasta"]), 10), P(f"Fecha: {c['fecha']:%d/%m/%Y}", 10, True, 1)],
                 [P(f"Proveedor: {c['razon']}", 10), ""],
                 [P(f"CUIT: {c['cuit']}", 10), ""]], colWidths=[ancho - 55 * mm, 55 * mm])
    enc.setStyle(TableStyle([("LINEBELOW", (0, 0), (0, 2), 1, colors.black),
                             ("BOX", (1, 0), (1, 0), 1.2, colors.black),
                             ("RIGHTPADDING", (0, 0), (0, -1), 30 * mm)]))
    el += [enc, Spacer(1, 4 * mm)]

    # Detalle como la planilla, con el % de humedad y de granulometría (finos) de cada camión:
    # Fecha | N° Rto Planta | Proveedor | Rto.Proveedor | Producto | Humedad % | Granulom. % | Kilos | KG TOTAL
    cols = ["Fecha", "N° Rto Planta", "Proveedor", "Rto. Prov.", "Producto", "Humedad %", "Granulom. %",
            "Kilos", "KG TOTAL"]
    data = [[P(x, 7.5, True, 1) for x in cols]]
    estilo = [("GRID", (0, 0), (-1, -2), 0.5, colors.black), ("BOX", (0, 0), (-1, 0), 1.2, colors.black),
              ("ALIGN", (0, 1), (-1, -1), "CENTER"), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
              ("FONTSIZE", (0, 1), (-1, -1), 8),
              ("TOPPADDING", (0, 1), (-1, -1), 1.2), ("BOTTOMPADDING", (0, 1), (-1, -1), 1.2)]
    etq = etiqueta_producto(c)
    pct_txt = lambda v: "" if v is None or pd.isna(v) else decimal(v * 100, 1) + " %"
    for _, f in c["filas"].iterrows():
        con_desc = _con_desc(f)
        data.append([f["Fecha"].strftime("%d/%m/%Y"), f["Rto. Planta"], P(f["Proveedor"], 7, align=1),
                     P(str(f["Remito Proveedor"]), 6.5 if len(str(f["Remito Proveedor"])) > 11 else 7.5, align=1), etq(f["Producto"]),
                     pct_txt(f.get("Hum")), pct_txt(f.get("Finos")),
                     kilos_txt(f["KilosNeto"]), kilos_txt(f["Kilos"]) if con_desc else ""])
        if con_desc:
            i = len(data) - 1
            data.append([P(texto_granulometria(f), 7.5), "", "", "", "ASERRÍN" if f["DescKg"] > 0 else "",
                         "", "", kilos_txt(f["DescKg"]) if f["DescKg"] > 0 else "", ""])
            estilo += [("SPAN", (0, i + 1), (3, i + 1)), ("BACKGROUND", (0, i), (-1, i + 1), amarillo_claro)]
    data.append(["", "", "", "", "", P("TOTAL TN", 9, True, 2), "", P(decimal(tn_detalle(c)), 9, True, 1), ""])
    estilo += [("BOX", (7, -1), (7, -1), 1.2, colors.black), ("SPAN", (5, -1), (6, -1))]
    w = [18 * mm, 22 * mm, 33 * mm, 22 * mm, 18 * mm, 18 * mm, 19 * mm, 19 * mm]
    w.append(ancho - sum(w))
    det = Table(data, colWidths=w, repeatRows=1)
    det.setStyle(TableStyle(estilo))
    el += [det]
    if c.get("desc_gran"):
        el += [Spacer(1, 1 * mm), P(nota_granulometria(c), 7)]
    el.append(Spacer(1, 4 * mm))

    bloque = []
    if con_tramos(c):
        tg = Table([[P("TOTAL DE TN ENTREGADAS EN EL MES DE TODOS LOS PUNTOS DE ENTREGA  "
                       f"{decimal(c['tn_grupo'])}", 10.5, True, 1)]], colWidths=[ancho])
        tg.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), verde)]))
        bloque += [tg, Spacer(1, 3 * mm)]
    if mostrar_humedad(c):
        hdata = [[P(x, 8, True, 1) for x in ["Producto", "Muestras", "Humedad prom.", "Límite", "Descuento",
                                              "TN", "TN a pagar"]]]
        hst = [("GRID", (0, 0), (-1, -1), 0.5, colors.black), ("ALIGN", (0, 0), (-1, -1), "CENTER"),
               ("FONTSIZE", (0, 1), (-1, -1), 8), ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E2EFDA"))]
        for t, hh in c["humedad"].items():
            if not hh["aplica_hum"]:
                continue
            hdata.append([nombre_producto(t), str(hh["muestras"] or "—"), pct(hh["H"], 1), _txt_limite(hh),
                          pct(hh["desc_hum"]), decimal(hh["tn"]), decimal(hh["tn_pagar"])])
            if hh["desc_hum"] > 0:
                hst.append(("BACKGROUND", (0, len(hdata) - 1), (-1, len(hdata) - 1), amarillo_claro))
        ht = Table(hdata, colWidths=[24 * mm, 18 * mm, 24 * mm, 36 * mm, 20 * mm, 20 * mm, 22 * mm])
        ht.setStyle(TableStyle(hst))
        bloque += [P("DESCUENTO POR HUMEDAD", 9.5, True), Spacer(1, 1 * mm), ht,
                   P("Humedad: promedio de las muestras del laboratorio en el período; límite según tipo de proveedor "
                     "más la tolerancia pactada.", 7), Spacer(1, 3 * mm)]
    pdata = [[P(x, 9, True, 1) for x in ["", "TN", "PRECIO", "TOTAL"]]]
    for f in c["tramos"]:
        pdata.append([f["etiqueta"], decimal(f["tn"]) if f["tn"] is not None else "", pesos(f["precio"]) if f["precio"] else "",
                      pesos(f["tn"] * f["precio"]) if (f["tn"] is not None and f["precio"] is not None) else ""])
    pdata.append(["", P(decimal(sum(tn_pagar(c, t) for t in c["precios"])), 9, True, 1), "",
                  P(pesos(c["importe"]), 11, True, 1)])
    pt = Table(pdata, colWidths=[40 * mm, 26 * mm, 34 * mm, 40 * mm])
    pt.setStyle(TableStyle([("GRID", (0, 0), (-1, -2), 0.5, colors.black),
                            ("BOX", (0, 0), (-1, -2), 1.2, colors.black),
                            ("ALIGN", (0, 0), (-1, -1), "CENTER"), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                            ("FONTSIZE", (0, 1), (-1, -1), 9),
                            ("BOX", (1, -1), (1, -1), 1.2, colors.black),
                            ("BACKGROUND", (3, -1), (3, -1), amarillo), ("BOX", (3, -1), (3, -1), 1.2, colors.black)]))
    fact_txt = Table([[P("FACTURAR", 10, True, 1)]], colWidths=[26 * mm])
    fact_txt.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, -1), amarillo)]))
    fact = Table([[pt, fact_txt]], colWidths=[146 * mm, ancho - 146 * mm])
    fact.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "BOTTOM"), ("LEFTPADDING", (0, 0), (0, 0), 12)]))
    firmas = Table([[[P("Aceptación Proveedor", 9, True)], "", [P("Firma Empresa", 9, True)]]],
                   colWidths=[70 * mm, ancho - 140 * mm, 70 * mm], rowHeights=[16 * mm])
    firmas.setStyle(TableStyle([("BOX", (0, 0), (0, 0), 1, colors.black), ("BOX", (2, 0), (2, 0), 1, colors.black),
                                ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    # Cuadro de precios, total a facturar y firmas siempre juntos (si no entran, pasan a la hoja siguiente)
    el.append(KeepTogether(bloque + [fact, Spacer(1, 6 * mm), firmas, Spacer(1, 3 * mm)]
                           + [P(t, 7, color=rojo) for t in PIE]))

    def pie(canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(gris)
        canvas.drawString(12 * mm, 8 * mm, f"Certificación Nº {c['numero']} · {c['razon']} · "
                                           f"{texto_periodo(c['desde'], c['hasta'])}")
        canvas.drawRightString(A4[0] - 12 * mm, 8 * mm, f"Página {doc.page}")
        canvas.restoreState()

    doc = SimpleDocTemplate(str(ruta), pagesize=A4, leftMargin=12 * mm, rightMargin=12 * mm,
                            topMargin=9 * mm, bottomMargin=14 * mm,
                            title=f"Certificación {c['numero']} - {c['razon']}", author=EMPRESA_LINEAS[0])
    doc.build(el, onFirstPage=pie, onLaterPages=pie)


# -----------------------------------------------------------------------------
# 8. Libro Excel único por período (una hoja por proveedor)
# -----------------------------------------------------------------------------
VERDE = PatternFill("solid", fgColor="92D050")
AMARILLO = PatternFill("solid", fgColor="FFFF00")
NARANJA = PatternFill("solid", fgColor="FCE4D6")
ROJO = PatternFill("solid", fgColor="C00000")
ROJO_CLARO = PatternFill("solid", fgColor="F8CBAD")
NARANJA_DESC = PatternFill("solid", fgColor="FFE0B2")     # filas con descuento aplicado
VERDE_CLARO = PatternFill("solid", fgColor="C6EFCE")
ENCAB = PatternFill("solid", fgColor="3A7D44")
FINO, MEDIO = Side(style="thin"), Side(style="medium")
B_FINO = Border(left=FINO, right=FINO, top=FINO, bottom=FINO)
B_MEDIO = Border(left=MEDIO, right=MEDIO, top=MEDIO, bottom=MEDIO)
MONEDA = '"$"\\ #,##0.00'
CENTRO = Alignment(horizontal="center", vertical="center")


def _nombre_hoja(prov, usados):
    base = re.sub(r"[\[\]\*\?/\\:]", "", prov)[:31].strip() or "Proveedor"
    n, k = base, 2
    while n.casefold() in usados:
        n = f"{base[:28]}~{k}"
        k += 1
    usados.add(n.casefold())
    return n


AMARILLO_CLARO = PatternFill("solid", fgColor="FFF2A8")    # camiones con descuento de granulometría


def _hoja_certificacion(ws, c, logo=None):
    """Formato de la certificación final (todos los proveedores menos GYG), editable a mano en el Excel:
    cada camión con su humedad % y granulometría % del laboratorio y dos celdas celestes editables
    (Desc. humedad % y Desc. granulometría %); los kilos, el aserrín, las TN y el importe se recalculan.
    Los precios también son celdas celestes editables."""
    from openpyxl.formatting.rule import FormulaRule
    from openpyxl.comments import Comment
    if c.get("formato") == "GYG":          # GYG tiene su propio formato (planilla de GYG)
        return _hoja_gyg(ws, c, logo)
    EDIT = PatternFill("solid", fgColor="DDEBF7")       # celdas para completar / corregir a mano
    for col, a in zip("ABCDEFGHIJKLM", [2, 11, 14, 22, 14, 16, 11, 10, 10, 11, 11, 12, 11]):
        ws.column_dimensions[col].width = a
    if logo:
        img = XLImage(io.BytesIO(logo))
        img.width, img.height = 170, 90
        ws.add_image(img, "B1")
    if c["estado"] != "CONFIRMADA":
        ws.merge_cells("G1:M3")
        txt = {"MODIFICADA": "MODIFICADA DESPUÉS DE CONFIRMAR — volver a confirmar",
               "BLOQUEADA": "BORRADOR — faltan datos",
               "REVISAR": "BORRADOR — observaciones sin revisar",
               "LISTA": "BORRADOR — lista para confirmar"}[c["estado"]]
        ws["G1"] = txt
        ws["G1"].font = Font(bold=True, color="FFFFFF", size=11)
        ws["G1"].fill = ROJO
        ws["G1"].alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for i, l in enumerate(EMPRESA_LINEAS, start=6):
        ws.cell(i, 2, l).font = Font(size=10)
    ws.merge_cells("K7:M7")
    ws["K7"] = f"Nº {c['numero'] or '(se asigna al confirmar)'}"
    ws["K7"].font, ws["K7"].alignment = Font(bold=True), CENTRO
    for col in "KLM":
        ws[f"{col}7"].border = B_MEDIO
    ws.merge_cells("B9:M9")
    ws["B9"] = "CERTIFICACIÓN ENTREGA BIOMASA"
    ws["B9"].font, ws["B9"].alignment = Font(bold=True, size=12, color="2E75B6"), CENTRO
    for r, t in [(11, texto_periodo(c["desde"], c["hasta"])), (12, f"Proveedor: {c['razon']}"),
                 (13, f"CUIT: {c['cuit'] or ''}")]:
        ws.merge_cells(start_row=r, start_column=2, end_row=r, end_column=7)
        ws.cell(r, 2, t).border = Border(bottom=MEDIO)
    if not c["cuit"]:
        ws["B13"].fill = EDIT
        ws["B13"].comment = Comment("Completá el CUIT del proveedor", "Certificaciones")
    ws.merge_cells("K11:M11")
    ws["K11"] = f"Fecha: {c['fecha']:%d/%m/%Y}"
    ws["K11"].font, ws["K11"].alignment = Font(bold=True), CENTRO
    for col in "KLM":
        ws[f"{col}11"].border = B_MEDIO
    cols = ["Fecha", "N° Rto Planta", "Proveedor", "Rto.Proveedor", "Producto", "KG TOTAL", "Humedad %",
            "Granulom. %", "Desc. humedad %", "Desc. granulom. %", "Kilos", "Aserrín (kg)"]
    for j, t in enumerate(cols, 2):
        x = ws.cell(15, j, t)
        x.font = Font(bold=True)
        x.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        x.border = B_MEDIO
        if j in (10, 11):
            x.fill = EDIT
    ws.row_dimensions[15].height = 44
    ws["J15"].comment = Comment("Editable: % de los kilos que se descuenta por humedad (0 = sin descuento)",
                                "Certificaciones")
    ws["K15"].comment = Comment("Editable: % de los kilos que pasa a aserrín por granulometría (0 = sin descuento)",
                                "Certificaciones")
    sev = {}
    for o in c["obs"]:
        if o["id"] and o["sev"] == CRIT:
            sev.setdefault(o["id"], set()).add(o["sev"])
    r = 16
    for _, f in c["filas"].iterrows():
        rp = f["Remito Proveedor"]
        vals = [f["Fecha"], f["Rto. Planta"], f["Proveedor"], int(rp) if re.fullmatch(r"\d{1,9}", rp or "") else rp,
                f["Producto"], float(f["Kilos"]),
                None if pd.isna(f.get("Hum")) else float(f["Hum"]),
                None if pd.isna(f.get("Finos")) else float(f["Finos"]),
                float(f.get("Pen") or 0.0), float(f.get("DescPct") or 0.0) / 100]
        for j, v in enumerate(vals, 2):
            ws.cell(r, j, v)
        # como la planilla: aserrín = KG TOTAL × (1 − desc. humedad) × desc. granulometría; kilos = el resto
        ws.cell(r, 13, f"=ROUND(G{r}*(1-J{r})*K{r},2)")
        ws.cell(r, 12, f"=G{r}*(1-J{r})-M{r}")
        ws.cell(r, 2).number_format = "dd/mm/yyyy"
        ws.cell(r, 7).number_format = "#,##0"
        for j in (8, 9, 10, 11):
            ws.cell(r, j).number_format = "0.0%"
        for j in (12, 13):
            ws.cell(r, j).number_format = "#,##0.0"
        for j in range(2, 14):
            x = ws.cell(r, j)
            x.border, x.alignment = B_FINO, CENTRO
            if CRIT in sev.get(f["_id"], set()) and j < 10:
                x.fill = ROJO_CLARO
        ws.cell(r, 10).fill = ws.cell(r, 11).fill = EDIT
        r += 1
    ult = r - 1
    if ult >= 16:
        # fila en amarillo si el camión tiene algún descuento (también si lo cargás a mano en el Excel)
        ws.conditional_formatting.add(f"B16:I{ult}", FormulaRule(formula=["OR($J16>0,$K16>0)"], fill=AMARILLO_CLARO))
        ws.conditional_formatting.add(f"L16:M{ult}", FormulaRule(formula=["OR($J16>0,$K16>0)"], fill=AMARILLO_CLARO))
    ws.cell(r, 6, "TOTAL").font = Font(bold=True)
    for j, f_ in ((7, f"=SUM(G16:G{ult})/1000"), (12, f"=SUM(L16:L{ult})/1000"), (13, f"=SUM(M16:M{ult})/1000")):
        x = ws.cell(r, j, f_)
        x.number_format, x.font, x.border, x.alignment = "0.00", Font(bold=True), B_MEDIO, CENTRO
    ws.cell(r, 8, "TN").font = Font(bold=True)
    r += 1
    ws.merge_cells(start_row=r, start_column=2, end_row=r, end_column=13)
    ws.row_dimensions[r].height = 36
    nota = ("Celdas celestes: se pueden corregir a mano (Desc. humedad %, Desc. granulom. % y precios) y todo se "
            "recalcula. Humedad % y Granulom. %: muestra del laboratorio de cada camión. ")
    if c.get("desc_gran"):
        nota += nota_granulometria(c).replace("&lt;", "<")
    ws.cell(r, 2, nota).font = Font(size=8, italic=True)
    ws.cell(r, 2).alignment = Alignment(wrap_text=True, vertical="top")
    r += 2
    if con_tramos(c):
        ws.merge_cells(start_row=r, start_column=2, end_row=r, end_column=13)
        x = ws.cell(r, 2, "TOTAL DE TN ENTREGADAS EN EL MES DE TODOS LOS PUNTOS DE ENTREGA  " + decimal(c["tn_grupo"]))
        x.font, x.fill, x.alignment = Font(bold=True, size=12), VERDE, CENTRO
        r += 2
    for j, t in zip(range(4, 7), ["TN", "PRECIO", "TOTAL"]):
        x = ws.cell(r, j, t)
        x.font, x.alignment, x.border = Font(bold=True), CENTRO, B_MEDIO
    ws.cell(r, 3).border = B_MEDIO
    ini = r + 1
    rng = lambda col: f"${col}$16:${col}${ult}"
    for f in c["tramos"]:
        r += 1
        ws.cell(r, 3, f["etiqueta"])
        t = f["producto"]
        tramo_sin_tn = f["tn"] is None and t in c["precios"]          # otro tramo del mismo producto
        if not tramo_sin_tn:
            extra = f'+SUM({rng("M")})' if t == "ASERRIN" else ""
            ws.cell(r, 4, f'=ROUND((SUMIF({rng("F")},"{t}",{rng("L")}){extra})/1000,2)')
        ws.cell(r, 5, f["precio"])
        ws.cell(r, 6, f'=IF(OR(D{r}="",E{r}=""),"",D{r}*E{r})')
        ws.cell(r, 4).number_format = "0.00"
        ws.cell(r, 5).number_format = ws.cell(r, 6).number_format = MONEDA
        for j in range(3, 7):
            ws.cell(r, j).border, ws.cell(r, j).alignment = B_FINO, CENTRO
        ws.cell(r, 5).fill = ROJO_CLARO if (t in c["precios"] and f["precio"] is None) else EDIT
    r += 1
    x = ws.cell(r, 4, f"=SUM(D{ini}:D{r-1})")
    x.number_format, x.font, x.border, x.alignment = "0.00", Font(bold=True), B_MEDIO, CENTRO
    r += 2
    ws.cell(r, 4, "FACTURAR").font = Font(bold=True)
    ws.cell(r, 4).fill = AMARILLO
    x = ws.cell(r, 6, f"=SUM(F{ini}:F{r-3})")
    x.number_format, x.font, x.fill, x.border = MONEDA, Font(bold=True, size=13), AMARILLO, B_MEDIO
    # Firmas en recuadros
    r += 3
    for col_ini, col_fin, texto in ((2, 5, "Aceptación Proveedor"), (9, 13, "Firma Empresa")):
        ws.merge_cells(start_row=r, start_column=col_ini, end_row=r + 3, end_column=col_fin)
        x = ws.cell(r, col_ini, texto)
        x.font, x.alignment = Font(bold=True), Alignment(vertical="top")
        for rr in range(r, r + 4):
            for cc in range(col_ini, col_fin + 1):
                ws.cell(rr, cc).border = Border(left=MEDIO if cc == col_ini else None,
                                                right=MEDIO if cc == col_fin else None,
                                                top=MEDIO if rr == r else None, bottom=MEDIO if rr == r + 3 else None)
    r += 6
    for t in PIE:
        ws.merge_cells(start_row=r, start_column=2, end_row=r, end_column=13)
        x = ws.cell(r, 2, t)
        x.font, x.alignment = Font(size=8, color="C00000"), Alignment(wrap_text=True, vertical="top")
        ws.row_dimensions[r].height = 20
        r += 1
    ws.freeze_panes = "B16"
    ws.print_title_rows = "15:15"
    ws.print_area = f"A1:M{r}"
    ws.page_setup.paperSize = ws.PAPERSIZE_A4
    ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 0
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_margins = PageMargins(left=0.3, right=0.3, top=0.4, bottom=0.4)
    ws.sheet_properties.tabColor = {"CONFIRMADA": "3A7D44", "LISTA": "92D050"}.get(c["estado"], "C00000")


# -----------------------------------------------------------------------------
# 7b. Formato GYG (humedad y granulometría por camión)
# -----------------------------------------------------------------------------
GYG_PRODUCTOS = ["CHIP", "ASERRIN", "C CORTO", "C LARGO", "Fino/Raleo"]
GYG_COL_KG = {"CHIP": "KgChip", "ASERRIN": "KgAse", "C CORTO": "KgCorto", "C LARGO": "KgLargo", "Fino/Raleo": "KgRaleo"}


def etiqueta_gyg(t):
    return "ROLLO" if t == "Fino/Raleo" else nombre_producto(t)


def criterio_gyg(c):
    alto, muy = decimal(c["tol_finos"], 0), decimal(c["muy_alto"], 0)
    return (f'Cuando el contenido de aserrín es superior al valor de "ALTO" ({alto} %) e inferior al valor de '
            f'"MUY ALTO" ({muy} %) se cuentan como aserrín los kilos que superen el valor de "ALTO". En caso de '
            f'igualar o superar el valor de "MUY ALTO" se cuenta como aserrín el total medido.',
            "Cuando la humedad excede el valor de referencia del día (chipeado propio de raleo fresco) se descuenta "
            "el porcentaje con el cual supera ese valor.")


def _pct_txt(v):
    return "" if v is None or pd.isna(v) else f"{v * 100:.2f}%".replace(".", ",")


def _kg2(v):
    return "" if v is None or pd.isna(v) else decimal(v, 2)


def pdf_gyg(ruta, c, logo=None):
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.lib.utils import ImageReader
    from reportlab.platypus import Image, KeepTogether, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

    verde, amarillo, naranja = colors.HexColor("#92D050"), colors.HexColor("#FFFF00"), colors.HexColor("#FFC000")
    azul, rojo, gris, celeste = colors.HexColor("#2E75B6"), colors.HexColor("#C00000"), colors.HexColor("#808080"), colors.HexColor("#DDEBF7")
    pag = landscape(A4)
    ancho = pag[0] - 20 * mm

    def P(t, size=8, bold=False, align=0, color=colors.black):
        return Paragraph(t, ParagraphStyle("p", fontName="Helvetica-Bold" if bold else "Helvetica",
                                           fontSize=size, leading=size * 1.2, alignment=align, textColor=color))

    el = []
    img = ""
    if logo:
        iw, ih = ImageReader(io.BytesIO(logo)).getSize()
        img = Image(io.BytesIO(logo), width=28 * mm, height=28 * mm * ih / iw)
    emp = [P(l, 8) for l in EMPRESA_LINEAS]
    caja = Table([["Fecha:", c["fecha"].strftime("%d/%m/%Y")], ["N°:", c["numero"] or "(al confirmar)"]],
                 colWidths=[16 * mm, 38 * mm])
    caja.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, colors.black), ("FONTSIZE", (0, 0), (-1, -1), 9),
                              ("FONTNAME", (1, 1), (1, 1), "Helvetica-Bold"), ("ALIGN", (1, 0), (1, -1), "CENTER")]))
    cab = Table([[img, emp, P("CERTIFICACIÓN ENTREGA BIOMASA", 13, True, 1, azul), caja]],
                colWidths=[32 * mm, 72 * mm, ancho - 32 * mm - 72 * mm - 58 * mm, 58 * mm])
    cab.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "MIDDLE")]))
    el += [cab, Spacer(1, 2 * mm)]

    g_txt, h_txt = criterio_gyg(c)
    datos = Table([["Período:", texto_periodo(c["desde"], c["hasta"]).replace("Período ", "")],
                   ["Proveedor:", c["razon"]], ["CUIT:", c["cuit"] or ""]], colWidths=[20 * mm, 72 * mm])
    datos.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, colors.black), ("FONTSIZE", (0, 0), (-1, -1), 8.5),
                               ("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold")]))
    crit = Table([[P("Criterio de evaluación", 8.5, True, 1), "", "", ""],
                  [P("Granulometría:", 7.5, True), P("Alto", 8, True, 1), P("Muy Alto", 8, True, 1), P(g_txt, 6.5)],
                  ["", P(f"{decimal(c['tol_finos'], 0)}%", 8, align=1), P(f"{decimal(c['muy_alto'], 0)}%", 8, align=1), ""],
                  [P("Humedad:", 8, True), P("Referencias extraídas del chipeado propio de raleo fresco", 7), "", P(h_txt, 6.5)]],
                 colWidths=[25 * mm, 22 * mm, 22 * mm, ancho - 92 * mm - 6 * mm - 69 * mm])
    crit.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, colors.black), ("SPAN", (0, 0), (-1, 0)),
                              ("SPAN", (0, 1), (0, 2)), ("SPAN", (3, 1), (3, 2)), ("SPAN", (1, 3), (2, 3)),
                              ("BACKGROUND", (1, 2), (2, 2), amarillo), ("VALIGN", (0, 0), (-1, -1), "MIDDLE")]))
    fila = Table([[datos, "", crit]], colWidths=[92 * mm, 6 * mm, ancho - 98 * mm])
    fila.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP"), ("LEFTPADDING", (0, 0), (-1, -1), 0)]))
    el += [fila, Spacer(1, 3 * mm)]

    cols = ["Fecha", "N° Rto Planta", "Rto.Proveedor", "Producto", "Kilos de carga", "Humedad medida",
            "Humedad de referencia", "Porcentaje de penalización", "Porcentaje medido de aserrín", "Kilos de chip",
            "Kilos de aserrín", "Kilos de C Corto", "Kilos C Largo", "Kilos Rollo"]
    data = [[P(x, 6.8, True, 1) for x in cols]]
    est = [("GRID", (0, 0), (-1, -1), 0.4, colors.black), ("ALIGN", (0, 1), (-1, -1), "CENTER"),
           ("VALIGN", (0, 0), (-1, -1), "MIDDLE"), ("FONTSIZE", (0, 1), (-1, -1), 7),
           ("TOPPADDING", (0, 1), (-1, -1), 0.6), ("BOTTOMPADDING", (0, 1), (-1, -1), 1.6), ("LEADING", (0, 1), (-1, -1), 8.4),
           ("BACKGROUND", (0, 0), (-1, 0), celeste)]
    alto, muy = c["tol_finos"] / 100, c["muy_alto"] / 100
    for _, f in c["filas"].iterrows():
        prod = f["Producto"]
        cs = prod in ("CHIP", "ASERRIN")
        i = len(data)
        data.append([f["Fecha"].strftime("%d-%m-%y"), f["Rto. Planta"], f["Remito Proveedor"], etiqueta_gyg(prod),
                     decimal(f["Kilos"], 2), _pct_txt(f["Hum"]), _pct_txt(f["HumRef"]), _pct_txt(f["Pen"]),
                     _pct_txt(f["Finos"] if prod == "CHIP" else 0.0),
                     _kg2(f["KgChip"]) if prod == "CHIP" else "", _kg2(f["KgAse"]) if cs else "",
                     _kg2(f["KgCorto"]) if f["KgCorto"] else "", _kg2(f["KgLargo"]) if f["KgLargo"] else "",
                     _kg2(f["KgRaleo"]) if f["KgRaleo"] else ""])
        if cs:
            est.append(("BACKGROUND", (7, i), (7, i), naranja if f["Pen"] > 0 else verde))
            fv = f["Finos"] if prod == "CHIP" and pd.notna(f["Finos"]) else 0.0
            est.append(("BACKGROUND", (8, i), (8, i), naranja if fv >= muy else (amarillo if fv >= alto else verde)))
    fl = c["filas"]
    data.append(["", "", "", P("TOTAL", 7, True, 1), P(decimal(fl["Kilos"].sum() / 1000, 2), 7, True, 1), "", "", "", "",
                 P(decimal(fl["KgChip"].sum(), 2), 7, True, 1), P(decimal(fl["KgAse"].sum(), 2), 7, True, 1),
                 P(decimal(fl["KgCorto"].sum(), 2), 7, True, 1), P(decimal(fl["KgLargo"].sum(), 2), 7, True, 1),
                 P(decimal(fl["KgRaleo"].sum(), 2), 7, True, 1)])
    w = [15, 21, 24, 17, 20, 18, 19, 20, 20, 21, 21, 18, 18]
    w = [x * mm for x in w]
    w.append(ancho - sum(w))
    det = Table(data, colWidths=w, repeatRows=1)
    det.setStyle(TableStyle(est))
    el += [det, Spacer(1, 4 * mm)]

    pdata = [["", P("Toneladas", 8.5, True, 1), P("Precio x TN", 8.5, True, 1), P("TOTAL", 8.5, True, 1)]]
    for t in GYG_PRODUCTOS + [t for t in c["precios"] if t not in GYG_PRODUCTOS]:
        tn = c["tn_prod"].get(t, 0.0)
        p = c["precios"].get(t)
        pdata.append([P(etiqueta_gyg(t), 8.5, True), decimal(tn, 2), pesos(p) if p else "",
                      pesos(tn * p) if p else pesos(0)])
    pdata.append([P("TN", 9, True), P(decimal(sum(c["tn_prod"].values()), 2), 9, True, 1), "",
                  P(pesos(c["importe"]), 9, True, 1)])
    pt = Table(pdata, colWidths=[24 * mm, 26 * mm, 30 * mm, 36 * mm])
    pt.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, colors.black), ("ALIGN", (1, 0), (-1, -1), "CENTER"),
                            ("FONTSIZE", (0, 1), (-1, -1), 8.5), ("VALIGN", (0, 0), (-1, -1), "MIDDLE")]))
    tot = Table([[P("Total a Facturar", 10, True, 1), P(pesos(c["importe"]), 12, True, 1)]],
                colWidths=[42 * mm, 52 * mm])
    tot.setStyle(TableStyle([("BOX", (0, 0), (-1, -1), 1.2, colors.black), ("BACKGROUND", (0, 0), (-1, -1), amarillo),
                             ("GRID", (0, 0), (-1, -1), 0.5, colors.black)]))
    firmas = Table([[[P("Aceptación Proveedor", 8.5, True)], "", [P("Firma Empresa", 8.5, True)]]],
                   colWidths=[62 * mm, 10 * mm, 62 * mm], rowHeights=[16 * mm])
    firmas.setStyle(TableStyle([("BOX", (0, 0), (0, 0), 1, colors.black), ("BOX", (2, 0), (2, 0), 1, colors.black),
                                ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    der = [tot, Spacer(1, 4 * mm), firmas, Spacer(1, 3 * mm)] + [P(t, 6.5, color=rojo) for t in PIE]
    abajo = Table([[pt, "", der]], colWidths=[118 * mm, 10 * mm, ancho - 128 * mm])
    abajo.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP"), ("LEFTPADDING", (0, 0), (-1, -1), 0)]))
    el.append(KeepTogether([abajo]))

    def pie(canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(gris)
        canvas.drawString(10 * mm, 7 * mm, f"Certificación Nº {c['numero']} · {c['razon']} · "
                                           f"{texto_periodo(c['desde'], c['hasta'])}")
        canvas.drawRightString(pag[0] - 10 * mm, 7 * mm, f"Hoja {doc.page}")
        canvas.restoreState()

    doc = SimpleDocTemplate(str(ruta), pagesize=pag, leftMargin=10 * mm, rightMargin=10 * mm,
                            topMargin=8 * mm, bottomMargin=12 * mm,
                            title=f"Certificación {c['numero']} - {c['razon']}", author=EMPRESA_LINEAS[0])
    doc.build(el, onFirstPage=pie, onLaterPages=pie)


def _hoja_gyg(ws, c, logo=None):
    """Formato de la planilla de GYG, con las mismas fórmulas: columnas A a N."""
    from openpyxl.formatting.rule import CellIsRule
    for col, a in zip("ABCDEFGHIJKLMN", [11, 15, 17, 17, 12, 11, 12, 13, 13, 13, 13, 12, 12, 12]):
        ws.column_dimensions[col].width = a
    if logo:
        img = XLImage(io.BytesIO(logo))
        img.width, img.height = 110, 58
        ws.add_image(img, "A1")
    for i, l in enumerate(EMPRESA_LINEAS, start=1):
        ws.cell(i, 3, l).font = Font(size=9)
    ws.merge_cells("E1:H3")
    ws["E1"] = "CERTIFICACIÓN ENTREGA BIOMASA"
    ws["E1"].font, ws["E1"].alignment = Font(bold=True, size=14, color="2E75B6"), CENTRO
    if c["estado"] != "CONFIRMADA":
        ws["E4"] = {"MODIFICADA": "MODIFICADA — volver a confirmar", "BLOQUEADA": "BORRADOR — tiene errores bloqueantes",
                    "REVISAR": "BORRADOR — observaciones sin revisar", "LISTA": "BORRADOR — lista para confirmar"}[c["estado"]]
        ws["E4"].font = Font(bold=True, color="C00000")
    for r, (k, v) in enumerate([("Fecha:", c["fecha"]), ("N° :", c["numero"] or "(al confirmar)")], start=1):
        ws.cell(r, 9, k).font = Font(bold=True)
        ws.merge_cells(start_row=r, start_column=10, end_row=r, end_column=11)
        x = ws.cell(r, 10, v)
        x.alignment, x.border = CENTRO, B_FINO
        if r == 1:
            x.number_format = "dd/mm/yyyy"
    ws["A5"], ws["B5"] = "Período:", texto_periodo(c["desde"], c["hasta"]).replace("Período ", "")
    ws["A6"], ws["B6"] = "Proveedor:", c["razon"]
    ws["A7"], ws["B7"] = "CUIT:", c["cuit"] or ""
    for r in (5, 6, 7):
        ws.cell(r, 1).font = Font(bold=True)
    g_txt, h_txt = criterio_gyg(c)
    ws.merge_cells("E5:N5")
    ws["E5"], ws["E5"].alignment = "Criterio de evaluación", CENTRO
    ws["E6"], ws["F6"], ws["G6"] = "Granulometría:", "Alto", "Muy Alto"
    ws["F7"], ws["G7"] = c["tol_finos"] / 100, c["muy_alto"] / 100
    for k in ("F7", "G7"):
        ws[k].number_format, ws[k].fill, ws[k].alignment = "0%", AMARILLO, CENTRO
    ws.merge_cells("H6:N7")
    ws["H6"], ws["H6"].alignment = g_txt, Alignment(wrap_text=True, vertical="top")
    ws["H6"].font = Font(size=8)
    ws["E8"], ws["F8"] = "Humedad:", "Referencias extraídas del chipeado propio de raleo fresco"
    ws.merge_cells("F8:G8")
    ws.merge_cells("H8:N8")
    ws["H8"], ws["H8"].alignment = h_txt, Alignment(wrap_text=True, vertical="top")
    ws["H8"].font = ws["F8"].font = Font(size=8)
    ws.row_dimensions[6].height = ws.row_dimensions[7].height = 22
    ws.row_dimensions[8].height = 24
    cols = ["Fecha", "N° Rto Planta ", "Rto.Proveedor", "Producto", "Kilos de carga", "Humedad medida",
            "Humedad de referencia", "Porcentaje de penalizacion", "Porcentaje medido de aserrin", "Kilos de chip",
            "kilos de aserrin", "Kilos de C Corto", "Kilos C Largo", "Kilos Rollo"]
    for j, t in enumerate(cols, 1):
        x = ws.cell(10, j, t)
        x.font, x.alignment, x.border = Font(bold=True), Alignment(horizontal="center", vertical="center", wrap_text=True), B_FINO
        x.fill = PatternFill("solid", fgColor="DDEBF7")
    ws.row_dimensions[10].height = 42
    n = len(c["filas"])
    ini, ult = 11, 10 + n
    tot = ult + 1
    pr = tot + 2                       # cuadro de precios
    fila_p = {t: pr + 1 + i for i, t in enumerate(GYG_PRODUCTOS)}
    lab = {t: f"$A${fila_p[t]}" for t in GYG_PRODUCTOS}
    for k, (_, f) in enumerate(c["filas"].iterrows()):
        r = ini + k
        rp = f["Remito Proveedor"]
        vals = [f["Fecha"], f["Rto. Planta"], rp, etiqueta_gyg(f["Producto"]), float(f["Kilos"]),
                None if pd.isna(f["Hum"]) else float(f["Hum"]), None if pd.isna(f["HumRef"]) else float(f["HumRef"])]
        for j, v in enumerate(vals, 1):
            ws.cell(r, j, v)
        if f.get("PenManual") or c.get("sin_hum"):
            ws.cell(r, 8, float(f["Pen"] or 0.0))           # descuento de humedad puesto / quitado a mano
        else:
            ws.cell(r, 8, f'=IF(F{r}="",0,IF(OR(D{r}={lab["CHIP"]},D{r}={lab["ASERRIN"]}),IF(F{r}>G{r},F{r}-G{r},0),0))')
        fin = f["Finos"] if f["Producto"] == "CHIP" and pd.notna(f["Finos"]) else 0
        ws.cell(r, 9, float(fin))
        ws.cell(r, 10, f'=IF(D{r}={lab["CHIP"]},(E{r}-K{r})*(1-H{r}),"")')
        if f["Producto"] == "CHIP" and (f.get("DescManual") or c.get("sin_gran")):
            # % de aserrín puesto / quitado a mano: reemplaza al criterio Alto / Muy Alto en este camión
            ws.cell(r, 11, f'={float(f["DescPct"] or 0) / 100:.15g}*E{r}*(1-H{r})')
        else:
            ws.cell(r, 11, f'=IF(D{r}={lab["CHIP"]},IF(I{r}<$F$7,0,IF(I{r}<$G$7,(I{r}-$F$7)*E{r},I{r}*E{r}))*(1-H{r}),'
                           f'IF(D{r}={lab["ASERRIN"]},E{r}*(1-H{r}),""))')
        for j, t in ((12, "C CORTO"), (13, "C LARGO"), (14, "Fino/Raleo")):
            ws.cell(r, j, f'=IF(D{r}={lab[t]},E{r},0)')
        ws.cell(r, 1).number_format = "dd-mm-yy"
        for j in (6, 7, 8, 9):
            ws.cell(r, j).number_format = "0.00%"
        for j in (5, 10, 11, 12, 13, 14):
            ws.cell(r, j).number_format = "#,##0.00"
        for j in range(1, 15):
            ws.cell(r, j).border, ws.cell(r, j).alignment = B_FINO, CENTRO
        if f["Producto"] in ("CHIP", "ASERRIN"):
            ws.cell(r, 8).fill = ws.cell(r, 9).fill = VERDE
        for j in (6, 9):                                   # humedad medida y % de aserrín: editables
            ws.cell(r, j).fill = PatternFill("solid", fgColor="DDEBF7")
    ws.conditional_formatting.add(f"H{ini}:H{ult}", CellIsRule(operator="greaterThan", formula=["0"],
                                                               fill=PatternFill("solid", bgColor="FFC000", fgColor="FFC000")))
    ws.conditional_formatting.add(f"I{ini}:I{ult}", CellIsRule(operator="greaterThanOrEqual", formula=["$G$7"],
                                                               fill=PatternFill("solid", bgColor="FFC000", fgColor="FFC000")))
    ws.conditional_formatting.add(f"I{ini}:I{ult}", CellIsRule(operator="between", formula=["$F$7", "$G$7"],
                                                               fill=PatternFill("solid", bgColor="FFFF00", fgColor="FFFF00")))
    ws.cell(tot, 4, "TOTAL").font = Font(bold=True)
    ws.cell(tot, 5, f"=SUM(E{ini}:E{ult})/1000")
    for j in range(10, 15):
        L = ws.cell(ini, j).column_letter
        ws.cell(tot, j, f"=SUM({L}{ini}:{L}{ult})")
    for j in (5, 10, 11, 12, 13, 14):
        x = ws.cell(tot, j)
        x.number_format, x.font, x.border, x.alignment = "#,##0.00", Font(bold=True), B_MEDIO, CENTRO
    for j, t in enumerate(["Toneladas", "Precio x Tn", "TOTAL"], 2):
        x = ws.cell(pr, j, t)
        x.font, x.alignment, x.border = Font(bold=True), CENTRO, B_FINO
    col_kg = {"CHIP": "J", "ASERRIN": "K", "C CORTO": "L", "C LARGO": "M", "Fino/Raleo": "N"}
    for t in GYG_PRODUCTOS:
        r = fila_p[t]
        ws.cell(r, 1, etiqueta_gyg(t)).font = Font(bold=True)
        ws.cell(r, 2, f"={col_kg[t]}{tot}/1000").number_format = "0.00"
        p = c["precios"].get(t)
        ws.cell(r, 3).fill = (ROJO_CLARO if (p is None and t in c["tn_prod"])
                              else PatternFill("solid", fgColor="DDEBF7"))     # precio editable
        ws.cell(r, 3, p)
        ws.cell(r, 4, f"=B{r}*C{r}")
        ws.cell(r, 3).number_format = ws.cell(r, 4).number_format = MONEDA
        for j in range(1, 5):
            ws.cell(r, j).border, ws.cell(r, j).alignment = B_FINO, CENTRO
    rt = pr + 1 + len(GYG_PRODUCTOS)
    ws.cell(rt, 1, "TN").font = Font(bold=True, size=12)
    ws.cell(rt, 2, f"=SUM(B{pr + 1}:B{rt - 1})").number_format = "0.00"
    ws.cell(rt, 4, f"=SUM(D{pr + 1}:D{rt - 1})").number_format = MONEDA
    for j in (1, 2, 4):
        ws.cell(rt, j).font, ws.cell(rt, j).border, ws.cell(rt, j).alignment = Font(bold=True), B_MEDIO, CENTRO
    ws.merge_cells(start_row=pr, start_column=7, end_row=pr, end_column=8)
    ws.cell(pr, 7, "Total a Facturar").font = Font(bold=True)
    ws.merge_cells(start_row=pr, start_column=9, end_row=pr, end_column=10)
    x = ws.cell(pr, 9, f"=D{rt}")
    x.number_format, x.font, x.fill, x.border, x.alignment = MONEDA, Font(bold=True, size=13), AMARILLO, B_MEDIO, CENTRO
    ws.cell(pr, 7).fill = AMARILLO
    r = pr + 2
    for col_ini, col_fin, texto in ((7, 9, "Aceptación Proveedor"), (11, 13, "Firma Empresa")):
        ws.merge_cells(start_row=r, start_column=col_ini, end_row=r + 3, end_column=col_fin)
        x = ws.cell(r, col_ini, texto)
        x.font, x.alignment = Font(bold=True), Alignment(vertical="top")
        for rr in range(r, r + 4):
            for cc in range(col_ini, col_fin + 1):
                ws.cell(rr, cc).border = Border(left=MEDIO if cc == col_ini else None, right=MEDIO if cc == col_fin else None,
                                                top=MEDIO if rr == r else None, bottom=MEDIO if rr == r + 3 else None)
    r += 5
    for t in PIE:
        ws.merge_cells(start_row=r, start_column=7, end_row=r, end_column=14)
        x = ws.cell(r, 7, t)
        x.font, x.alignment = Font(size=8, color="C00000"), Alignment(wrap_text=True, vertical="top")
        ws.row_dimensions[r].height = 22
        r += 1
    ws.freeze_panes = "A11"
    ws.print_title_rows = "10:10"
    ws.print_area = f"A1:N{max(r, rt)}"
    ws.page_setup.paperSize = ws.PAPERSIZE_A4
    ws.page_setup.orientation = "landscape"
    ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 0
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_margins = PageMargins(left=0.3, right=0.3, top=0.4, bottom=0.4)
    ws.sheet_properties.tabColor = {"CONFIRMADA": "3A7D44", "LISTA": "92D050"}.get(c["estado"], "C00000")


def excel_certificacion(ruta, c, logo=None):
    """Excel individual de una certificación, con el mismo formato que el PDF."""
    wb = Workbook()
    ws = wb.active
    ws.title = "Certificación"
    _hoja_certificacion(ws, c, logo)
    wb.properties.title = f"Certificación {c['numero']} - {c['proveedor']}"
    wb.save(ruta)


def escribir_libro(ruta, certs, logo=None):
    """Un solo Excel con todas las certificaciones del período: índice + una hoja por proveedor."""
    wb = Workbook()
    ix = wb.active
    ix.title = "Índice"
    enc = ["Estado", "Proveedor", "N° Certificación", "Remitos", "TN", "Importe", "Hoja"]
    ix.append(enc)
    usados = {"índice"}
    for c in certs:
        hoja = _nombre_hoja(c["proveedor"], usados)
        _hoja_certificacion(wb.create_sheet(hoja), c, logo)
        ix.append([f"{ICONOS[c['estado']]} {c['estado']}", c["proveedor"], c["numero"], len(c["filas"]),
                   c["tn"], c["importe"], hoja])
        x = ix.cell(ix.max_row, 7)
        x.hyperlink, x.style = f"#'{hoja}'!A1", "Hyperlink"
        ix.cell(ix.max_row, 6).number_format = MONEDA
        ix.cell(ix.max_row, 1).fill = VERDE_CLARO if c["estado"] == "CONFIRMADA" else ROJO_CLARO
    ix.append(["", "TOTAL", "", sum(len(c["filas"]) for c in certs), round(sum(c["tn"] for c in certs), 2),
               sum(c["importe"] or 0 for c in certs), ""])
    for x in ix[ix.max_row]:
        x.font = Font(bold=True)
    ix.cell(ix.max_row, 6).number_format = MONEDA
    for c in ix[1]:
        c.font, c.fill = Font(bold=True, color="FFFFFF"), ENCAB
    for col, a in zip("ABCDEFG", [16, 40, 18, 10, 12, 18, 34]):
        ix.column_dimensions[col].width = a
    ix.freeze_panes = "A2"
    ix.page_setup.orientation = "landscape"
    ix.page_setup.paperSize = ix.PAPERSIZE_A4
    ix.page_setup.fitToWidth, ix.page_setup.fitToHeight = 1, 0
    ix.sheet_properties.pageSetUpPr.fitToPage = True
    wb.save(ruta)


def escribir_resumen(ruta, certs, globales, obs_cfg, cuad, log, mapa_humedad=None):
    filas, ctrl, hum, gran, vistos = [], [], [], [], set()
    for c in certs.values():
        for t, h in c.get("humedad", {}).items():
            if not h["aplica_hum"]:
                continue
            hum.append({"Período": c["periodo"], "Certificación": c["proveedor"], "Producto": t,
                        "Muestras": h["muestras"], "Humedad prom.": h["H"], "Límite planilla": h["L_lab"],
                        "Límite fijo pactado": h["L_fijo"], "Tolerancia": h["tol"], "Límite aplicado": h["L"],
                        "Descuento %": h["desc_hum"], "TN": h["tn"], "TN descontadas": h["tn_desc"],
                        "TN a pagar": h["tn_pagar"], "Origen": h["origen"]})
    for c in sorted(certs.values(), key=lambda c: c["periodo"] != "Mensual"):
        for _, f in c["filas"].iterrows():
            if f["_id"] in vistos or f["Producto"] != "CHIP" or not c.get("desc_gran"):
                continue
            vistos.add(f["_id"])
            gran.append({"Certificación": c["proveedor"], "Proveedor (maestro)": f["Proveedor"], "Fecha": f["Fecha"],
                         "Rto. Planta": f["Rto. Planta"], "Rto. Proveedor": f["Remito Proveedor"],
                         "KG TOTAL": f["Kilos"], "Finos %": None if pd.isna(f["Finos"]) else round(f["Finos"] * 100, 2),
                         "Hora muestra": f["FinosHora"], "Cómo se asignó": f["FinosOrigen"] or "sin muestra",
                         "Desc. %": f["DescPct"], "Kg a aserrín": f["DescKg"], "Kg chip": f["KilosNeto"]})
    for c in certs.values():
        filas.append({"Estado": c["estado"], "Período": c["periodo"], "Proveedor": c["proveedor"],
                      "Grupo": c["grupo"], "N° Certificación": c["numero"], "Remitos": len(c["filas"]),
                      "TN entregadas": c["tn"],
                      "TN chip → aserrín (granulometría)": round(c["filas"]["DescKg"].sum() / 1000, 2),
                      "TN desc. humedad": round(sum(h["tn_desc_hum"] for h in c.get("humedad", {}).values()), 2),
                      "TN a pagar": round(sum(tn_pagar(c, t) for t in c["precios"]), 2), "Importe": c["importe"],
                      "Bloqueantes": sum(o["bloq"] for o in c["obs"]),
                      "Críticas": sum(o["sev"] == CRIT for o in c["obs"]),
                      "Advertencias": sum(o["sev"] == ADV for o in c["obs"])})
        for o in c["obs"]:
            ctrl.append({"Severidad": "BLOQUEANTE" if o["bloq"] else o["sev"], "Período": c["periodo"],
                         "Proveedor": c["proveedor"], "Rto. Planta": o["rem"], "Observación": o["msg"]})
    for g in globales:
        ctrl.append({"Severidad": g["sev"], "Período": "Mes", "Proveedor": g["prov"], "Rto. Planta": g["rem"],
                     "Observación": g["msg"]})
    for sev, g, msg in obs_cfg:
        ctrl.append({"Severidad": sev, "Período": "Config", "Proveedor": g, "Rto. Planta": "", "Observación": msg})
    ctrl = pd.DataFrame(ctrl, columns=["Severidad", "Período", "Proveedor", "Rto. Planta", "Observación"])
    orden = {"BLOQUEANTE": 0, CRIT: 1, ADV: 2}
    ctrl = (ctrl.drop_duplicates(["Severidad", "Proveedor", "Rto. Planta", "Observación"])
                .sort_values(["Severidad", "Proveedor"], key=lambda s: s.map(orden) if s.name == "Severidad" else s))
    with pd.ExcelWriter(ruta, engine="openpyxl") as w:
        cuad.to_excel(w, sheet_name="Cuadre", index=False)
        pd.DataFrame(filas).to_excel(w, sheet_name="Certificaciones", index=False)
        if gran:
            pd.DataFrame(gran).sort_values(["Certificación", "Fecha", "Rto. Planta"]).to_excel(
                w, sheet_name="Granulometría por camión", index=False)
        if hum:
            pd.DataFrame(hum).to_excel(w, sheet_name="Humedad", index=False)
        if mapa_humedad is not None:
            mapa_humedad.to_excel(w, sheet_name="Vínculo humedad", index=False)
        ctrl.to_excel(w, sheet_name="Control", index=False)
        pd.DataFrame(log, columns=["Fecha y hora", "Responsable", "Acción", "Certificación", "Rto. Planta",
                                   "Campo", "Antes", "Después", "Motivo"]
                     ).to_excel(w, sheet_name="Registro de cambios", index=False)
    wb = load_workbook(ruta)
    for ws in wb:
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = max(
                10, min(80, max(len(str(c.value or "")) for c in col) + 2))
        for c in ws[1]:
            c.font, c.fill = Font(bold=True, color="FFFFFF"), ENCAB
        ws.freeze_panes = "A2"
    if "Humedad" in wb.sheetnames:
        for row in wb["Humedad"].iter_rows(min_row=2):
            for i in (4, 5, 6, 7, 8, 9):
                row[i].number_format = "0.00%"
            if (row[9].value or 0) > 0:
                for x in row:
                    x.fill = AMARILLO_CLARO
    if "Granulometría por camión" in wb.sheetnames:
        for row in wb["Granulometría por camión"].iter_rows(min_row=2):
            row[2].number_format = "dd/mm/yyyy"
            if (row[10].value or 0) > 0:
                for x in row:
                    x.fill = AMARILLO_CLARO
    for row in wb["Certificaciones"].iter_rows(min_row=2):
        row[11].number_format = MONEDA
        row[0].fill = VERDE_CLARO if row[0].value == "CONFIRMADA" else ROJO_CLARO
    for row in wb["Cuadre"].iter_rows(min_row=2):
        row[-1].fill = VERDE_CLARO if "✔" in str(row[-1].value) else ROJO_CLARO
    wb.save(ruta)


def logo_por_defecto():
    """Sin logo incorporado: se toma de la certificación modelo que se sube en el Paso 3."""
    return None


def extraer_logo(ruta_modelo=None):
    """Logo de la certificación modelo subida; si no hay o no se puede leer, sin logo."""
    if ruta_modelo:
        try:
            with zipfile.ZipFile(ruta_modelo) as z:
                # la primera imagen dibujada en la hoja de certificación
                for dib in sorted(n for n in z.namelist() if re.fullmatch(r"xl/drawings/drawing\d+\.xml", n)):
                    rid = re.search(r'r:embed="([^"]+)"', z.read(dib).decode("utf-8", "ignore"))
                    rels = dib.replace("drawings/", "drawings/_rels/") + ".rels"
                    if rid and rels in z.namelist():
                        m = re.search(r'Id="%s"[^>]*Target="\.\./media/([^"]+)"|Target="\.\./media/([^"]+)"[^>]*Id="%s"'
                                      % (rid.group(1), rid.group(1)), z.read(rels).decode("utf-8", "ignore"))
                        if m:
                            return z.read("xl/media/" + (m.group(1) or m.group(2)))
                imgs = sorted(n for n in z.namelist() if n.startswith("xl/media/"))
                if imgs:
                    return z.read(imgs[0])
        except Exception:
            pass
    return logo_por_defecto()


In [ ]:
%%writefile editor_colab.py
# =============================================================================
#  EDITOR — revisar, corregir y confirmar certificaciones en Google Colab
# =============================================================================
import json
import shutil
from datetime import date, datetime
from pathlib import Path

import pandas as pd
import ipywidgets as W
from IPython.display import display

import cert_biomasa as cb

try:
    from google.colab import files as _colab_files
except ImportError:
    _colab_files = None

ANCHOS = {"del": "34px", "Batea": "90px", "Fecha": "150px", "Rto. Planta": "140px",
          "Remito Proveedor": "150px", "Producto": "120px", "Kilos": "105px", "Finos": "80px", "Hum": "80px", "DescMan": "80px", "PenMan": "80px",
          "Desc": "130px", "Punto": "150px", "obs": "40px"}


def _L(w, **kw):
    return W.Layout(width=w, **kw)


class Selector:
    """Paso de elección: año, mes, períodos (1Q / 2Q / Mensual por separado) y proveedor,
    con la lista de proveedores tomada del maestro para el mes elegido."""

    def __init__(self, maestro, conf=None, anio=None, mes=None):
        self.d, _ = cb.preparar(maestro, conf)
        hoy = date.today()
        ult = self.d[cb.C_DIA].max() - pd.Timedelta(days=10)   # último mes completo
        anio = anio or ult.year
        mes = mes or ult.month
        anios = sorted(int(a) for a in self.d[cb.C_DIA].dt.year.unique())
        st = {"description_width": "90px"}
        self.w_anio = W.Dropdown(options=anios, value=anio if anio in anios else anios[-1], description="Año:",
                                 style=st, layout=_L("200px"))
        self.w_mes = W.Dropdown(options=[(cb.MESES[m], m) for m in range(1, 13)], value=mes, description="Mes:",
                                style=st, layout=_L("240px"))
        self.w_1q = W.Checkbox(value=True, indent=False, description=f"1ª quincena (1 al {cb.DIA_CORTE})", layout=_L("220px"))
        self.w_2q = W.Checkbox(value=True, indent=False, description=f"2ª quincena ({cb.DIA_CORTE + 1} a fin de mes)", layout=_L("250px"))
        self.w_men = W.Checkbox(value=True, indent=False, description="Mensual (mes completo)", layout=_L("220px"))
        self.w_corte = W.BoundedIntText(value=cb.DIA_CORTE, min=10, max=20, description="Último día de la 1ª quincena:",
                                        style={"description_width": "200px"}, layout=_L("280px"))
        self.w_corte.observe(lambda ch: self._etiquetas(), "value")
        self.w_buscar = W.Text(placeholder="Buscar proveedor…", description="Proveedor:", style=st, layout=_L("330px"))
        self.w_prov = W.Select(description="", rows=12, layout=_L("620px"))
        self.w_info = W.HTML()
        for w in (self.w_anio, self.w_mes, self.w_buscar):
            w.observe(lambda ch: self._listar(), "value")
        for w in (self.w_prov, self.w_1q, self.w_2q, self.w_men):
            w.observe(lambda ch: self._info(), "value")
        self._listar()

    def _etiquetas(self):
        dc = self.w_corte.value
        self.w_1q.description = f"1ª quincena (1 al {dc})"
        self.w_2q.description = f"2ª quincena ({dc + 1} a fin de mes)"
        self._info()

    @property
    def corte(self):
        return int(self.w_corte.value)

    def _listar(self):
        a, m = self.w_anio.value, self.w_mes.value
        x = self.d[(self.d[cb.C_DIA].dt.year == a) & (self.d[cb.C_DIA].dt.month == m)]
        g = x.groupby(cb.C_PROV)[cb.C_NETO].agg(["size", "sum"]).sort_index(key=lambda s: s.str.casefold())
        q = self.w_buscar.value.strip().casefold()
        ops = [(f"— Todos los proveedores ({len(g)}) —", "__TODOS__")]
        tipos = (x.groupby([cb.C_PROV, cb.C_TIPO])[cb.C_NETO].sum() / 1000).sort_values(ascending=False)
        txt_t = lambda p: " / ".join(cb.nombre_producto(t) for t in tipos.loc[p].index) if p in tipos.index else ""
        ops += [(f"{p}   ·   {int(r['size'])} remitos · {cb.decimal(r['sum'] / 1000)} TN · {txt_t(p)}", p)
                for p, r in g.iterrows() if q in p.casefold()]
        actual = self.w_prov.value
        self.w_prov.options = ops
        if q and len(ops) > 1 and actual in (None, "__TODOS__"):
            self.w_prov.value = ops[1][1]
        else:
            self.w_prov.value = actual if actual in [v for _, v in ops] else "__TODOS__"
        self._info()

    @property
    def periodos(self):
        return [k for k, w in (("1Q", self.w_1q), ("2Q", self.w_2q), ("Mensual", self.w_men)) if w.value]

    @property
    def proveedores(self):
        return None if self.w_prov.value in (None, "__TODOS__") else [self.w_prov.value]

    def _info(self):
        per = self.periodos
        quien = "todos los proveedores" if self.proveedores is None else f"<b>{self.w_prov.value}</b>"
        if not per:
            self.w_info.value = "<b style=color:red>Marcá al menos un período.</b>"
        else:
            self.w_info.value = (f"Se va a certificar {quien} · <b>{', '.join(per)}</b> de "
                                 f"<b>{cb.MESES[self.w_mes.value]} {self.w_anio.value}</b> "
                                 f"(1ª quincena del 1 al {self.w_corte.value}). "
                                 "Si cambiás algo acá, volvé a ejecutar el Paso 5.")
            return
            self.w_info.value = (f"Se va a certificar {quien} · <b>{', '.join(per)}</b> de "
                                 f"<b>{cb.MESES[self.w_mes.value]} {self.w_anio.value}</b>. "
                                 "Si cambiás algo acá, volvé a ejecutar el Paso 5.")

    def mostrar(self):
        display(W.VBox([W.HBox([self.w_anio, self.w_mes]),
                        W.HTML("<b>Períodos a certificar</b>"), self.w_corte, W.HBox([self.w_1q, self.w_2q, self.w_men]),
                        W.HTML("<b>Proveedor</b> (la lista sale del maestro para el mes elegido)"),
                        self.w_buscar, self.w_prov, self.w_info]))


class App:
    def __init__(self, maestro, conf, precios, par, anio, mes, tipo="ambas", salida="certificaciones",
                 logo=None, proveedores=None, estado=None, origen="", descargar_pdf=True, ruta_config=None,
                 humedad=None, corte=None):
        anio, mes = int(anio), int(mes)          # nunca tipos de numpy: se guardan en el JSON del estado
        self.raw, self.conf, self.precios, self.par = maestro, conf, precios, par
        self.metodo_humedad = str(par.get("METODO_HUMEDAD", "PUNTOS")).upper()
        self.tipo, self.logo, self.proveedores, self.origen = tipo, logo or cb.logo_por_defecto(), proveedores, origen
        self.descargar_pdf = descargar_pdf
        self.salida = Path(salida)
        self.salida.mkdir(parents=True, exist_ok=True)
        # En la PC el config vive en la carpeta principal (ruta_config); en Colab, junto a la salida
        self.ruta_config = Path(ruta_config) if ruta_config else self.salida / "config_proveedores.xlsx"
        self._respaldo_hecho = False
        cb.fijar_finos(par.get("FINOS_TOLERANCIA", cb.FINOS_TOLERANCIA), par.get("FINOS_TOTALIDAD", cb.FINOS_TOTALIDAD))
        self.d0, self.unificados = cb.preparar(maestro, conf)
        # Si se eligió un punto de entrega que se certifica junto con otros (CERTIFICAR CON), se toman todos
        if proveedores:
            unidades = {cb.unidad(conf, p) for p in proveedores}
            self.proveedores = sorted(set(proveedores) | {p for p in self.d0[cb.C_PROV].unique()
                                                          if cb.unidad(conf, p) in unidades})
        self.humedad, self.mapa_humedad = (cb.mapear_humedad(humedad, self.d0, conf) if humedad is not None
                                           else (None, None))
        self.estado = estado or cb.estado_nuevo(anio, mes, int(par["NUMERO_INICIAL"]), par["PREFIJO"])
        self.estado["anio"], self.estado["mes"] = int(self.estado["anio"]), int(self.estado["mes"])
        self.estado["corr"].setdefault("finos", {})
        for k in ("humedad_camion", "desc_gran", "pen_hum"):
            self.estado["corr"].setdefault(k, {})
        # Día de corte de la quincena: si el mes ya tiene certificaciones confirmadas se mantiene el que se usó
        pedido = int(corte if corte is not None else par.get("DIA_CORTE_QUINCENA", cb.DIA_CORTE))
        confirmadas = any(m.get("huella") for m in self.estado["meta"].values())
        if confirmadas and self.estado.get("corte") and int(self.estado["corte"]) != pedido:
            print(f"⚠ {cb.MESES[mes]} ya tiene certificaciones confirmadas con la 1ª quincena del 1 al "
                  f"{self.estado['corte']}: se mantiene ese corte (pediste {pedido}).")
            pedido = int(self.estado["corte"])
        self.estado["corte"] = pedido
        cb.fijar_corte(pedido)
        # La numeración es una sola para todos los meses: nunca usar un número ya entregado en otro mes
        self.estado["proximo"] = max(int(self.estado["proximo"]), int(par["NUMERO_INICIAL"]))
        self.ruta_estado = self.salida / f"estado_{anio}-{mes:02d}.json"
        self.key = None
        self.reconstruir()

    # ------------------------------------------------------------------ datos
    def reconstruir(self):
        self.d = cb.aplicar_correcciones(self.d0, self.estado["corr"])
        self.certs, self.globales, self.obs_cfg = cb.construir(
            self.d, self.d0, self.conf, self.precios, self.estado, self.tipo, self.raw, self.proveedores,
            self.humedad, self.metodo_humedad)
        self._validacion = cb.construir.ultima_validacion
        self.cuad = cb.cuadre(self.d0, self.d, self.certs, self.conf, self.estado["anio"],
                              self.estado["mes"], self.proveedores)

    def recalcular(self, key):
        """Rearma solo una certificación (cuando cambian número, fecha o confirmación, no los datos)."""
        nuevos, _, _ = cb.construir(self.d, self.d0, self.conf, self.precios, self.estado, self.tipo, self.raw,
                                    self.proveedores, self.humedad, self.metodo_humedad, solo={key},
                                    validacion=self._validacion)
        if key in nuevos:
            self.certs[key] = nuevos[key]

    def actualizar_humedad(self, humedad):
        """Reemplaza la planilla de calidad (humedad y granulometría) sin perder correcciones ni confirmaciones.
        Las certificaciones ya confirmadas cuyo descuento cambie pasan a MODIFICADA."""
        antes = {k: c["estado"] for k, c in self.certs.items()}
        self.humedad, self.mapa_humedad = cb.mapear_humedad(humedad, self.d0, self.conf)
        self.reconstruir()
        cambiadas = [k for k, c in self.certs.items() if antes.get(k) == "CONFIRMADA" and c["estado"] == "MODIFICADA"]
        return cambiadas

    def guardar_estado(self):
        self.par["NUMERO_INICIAL"] = max(int(self.par["NUMERO_INICIAL"]), int(self.estado["proximo"]))
        self.ruta_estado.write_text(json.dumps(self.estado, ensure_ascii=False, indent=1, default=str),
                                    encoding="utf-8")
        self._respaldar_config()
        try:
            cb.guardar_config(self.ruta_config, self.conf, self.precios, self.par)
        except PermissionError:
            raise PermissionError(f"No pude guardar {self.ruta_config.name}: está abierto en Excel. "
                                  "Cerralo y volvé a tocar Guardar (no se perdió nada).") from None

    def _respaldar_config(self, conservar=30):
        """Antes del primer cambio de cada sesión copia el config a config_respaldos/ (guarda los últimos 30)."""
        if self._respaldo_hecho or not self.ruta_config.exists():
            return
        import shutil
        carpeta = self.ruta_config.parent / "config_respaldos"
        carpeta.mkdir(exist_ok=True)
        shutil.copy2(self.ruta_config, carpeta / f"config_proveedores_{datetime.now():%Y-%m-%d_%H%M%S}.xlsx")
        for viejo in sorted(carpeta.glob("config_proveedores_*.xlsx"))[:-conservar]:
            viejo.unlink()
        self._respaldo_hecho = True

    def _log(self, accion, cert="", rem="", campo="", antes="", despues="", motivo=""):
        self.estado["log"].append([datetime.now().strftime("%d/%m/%Y %H:%M:%S"), self.w_usuario.value.strip(),
                                   accion, cert, rem, campo, str(antes), str(despues), motivo])

    def resumen_estados(self):
        s = pd.Series([c["estado"] for c in self.certs.values()]).value_counts()
        return " · ".join(f"{cb.ICONOS[k]} {k}: {v}" for k, v in s.items())

    # ---------------------------------------------------------------- interfaz
    def mostrar(self):
        self.w_usuario = W.Text(description="Responsable:", placeholder="Tu nombre (obligatorio)",
                                style={"description_width": "initial"}, layout=_L("330px"))
        self.w_filtro = W.ToggleButtons(options=["Pendientes", "Todas", "Confirmadas"], value="Pendientes")
        self.w_sel = W.Dropdown(description="Certificación:", style={"description_width": "initial"},
                                layout=_L("640px"))
        self.w_cont = W.HTML()
        self.w_panel = W.VBox()
        self.w_out = W.Output(layout=W.Layout(border="1px solid #ddd", padding="4px"))
        self.w_filtro.observe(lambda ch: self._refrescar_lista(), "value")
        self.w_sel.observe(self._on_sel, "value")
        self._refrescar_lista()
        display(W.VBox([W.HBox([self.w_usuario, self.w_filtro]), W.HBox([self.w_sel]), self.w_cont,
                        self.w_panel, self.w_out]))

    def _etiqueta(self, c):
        return (f"{cb.ICONOS[c['estado']]} {c['periodo']:<8} {c['proveedor']}  ·  {len(c['filas'])} remitos  ·  "
                f"{cb.decimal(c['tn'])} TN" + (f"  ·  Nº {c['numero']}" if c["numero"] else ""))

    def _refrescar_lista(self, mantener=None):
        f = self.w_filtro.value
        ks = [k for k, c in self.certs.items()
              if f == "Todas" or (f == "Confirmadas") == (c["estado"] == "CONFIRMADA")]
        orden = {"1Q": 0, "2Q": 1, "Mensual": 2}
        ks.sort(key=lambda k: (orden[self.certs[k]["periodo"]], self.certs[k]["proveedor"].casefold()))
        opciones = [(self._etiqueta(self.certs[k]), k) for k in ks]
        destino = mantener if mantener in ks else (ks[0] if ks else None)
        self.w_sel.unobserve(self._on_sel, "value")
        self.w_sel.options = opciones
        self.w_sel.value = destino
        self.w_sel.observe(self._on_sel, "value")
        cuadra = all("✔" in r for r in self.cuad["Resultado"])
        self.w_cont.value = (f"<b>{len(self.certs)} certificaciones</b> — {self.resumen_estados()} &nbsp;|&nbsp; "
                             f"Cuadre con el maestro: {'<span style=color:green>✔ cuadra</span>' if cuadra else '<span style=color:red>✖ NO CUADRA</span>'}"
                             f" &nbsp;|&nbsp; Próximo Nº: {self.estado['prefijo']}-{self.estado['proximo']:08d}")
        self._render(destino)

    def _on_sel(self, ch):
        self._seguro(lambda: self._render(ch["new"]))

    # --------------------------------------------------------------- render
    def _render(self, key):
        self.key = key
        if key is None:
            self.w_panel.children = [W.HTML("<i>No hay certificaciones en esta vista.</i>")]
            return
        c = self.certs[key]
        meta = self.estado["meta"].get(key, {})
        est = c["estado"]
        color = {"CONFIRMADA": "#3A7D44", "LISTA": "#3A7D44", "MODIFICADA": "#B26B00",
                 "REVISAR": "#B26B00", "BLOQUEADA": "#C00000"}[est]
        self.multi = (len(c.get("miembros", [])) > 1 or c["filas"]["Proveedor"].nunique() > 1
                      or (c["filas"]["Proveedor"] != c["proveedor"]).any())
        self.gyg = c.get("formato") == "GYG"
        cab = W.HTML(f"<h3 style='margin:6px 0'>{c['proveedor']} — {cb.texto_periodo(c['desde'], c['hasta'])}</h3>"
                     + (f"<small>Incluye los puntos de entrega: <b>{' + '.join(c['miembros'])}</b></small><br>"
                        if self.multi else "")
                     + f"<span style='background:{color};color:white;padding:3px 8px;border-radius:4px'>"
                     f"{cb.ICONOS[est]} {est}</span> &nbsp; Grupo de precios: <b>{c['grupo']}</b><br>"
                     + "<small>Tipologías: " + " · ".join(
                         f"<b>{(cb.etiqueta_gyg if c.get('formato') == 'GYG' else cb.nombre_producto)(t)}</b> "
                         f"{cb.decimal(cb.tn_pagar(c, t))} TN" for t in cb.orden_productos(c["tn_prod"])) + "</small>")

        # Encabezado editable
        self.w_razon = W.Text(value=c["razon"], description="Razón social:", layout=_L("420px"),
                              style={"description_width": "110px"})
        self.w_cuit = W.Text(value=c["cuit"], description="CUIT:", placeholder="30-12345678-9", layout=_L("260px"),
                             style={"description_width": "50px"})
        self.w_num = W.Text(value=c["numero"], description="Nº:", placeholder="se asigna al confirmar",
                            layout=_L("260px"), style={"description_width": "30px"})
        self.w_fecha = W.DatePicker(value=c["fecha"], description="Fecha cert.:", layout=_L("250px"),
                                    style={"description_width": "80px"})
        self.w_tng = W.FloatText(value=round(c["tn_grupo"], 2), description="TN grupo mes:", layout=_L("230px"),
                                 style={"description_width": "100px"})
        encab = W.VBox([W.HBox([self.w_razon, self.w_cuit]), W.HBox([self.w_num, self.w_fecha, self.w_tng]),
                        W.HTML(f"<small>Según remitos el grupo suma {cb.decimal(c['tn_grupo_auto'])} TN en el mes "
                               "(define el tramo de precio).</small>")])

        # Tabla de remitos
        obs_id = {}
        for o in c["obs"]:
            if o["id"]:
                obs_id.setdefault(o["id"], []).append(o)
        tit = W.HBox([W.HTML(f"<b>{t}</b>", layout=_L(ANCHOS[k])) for k, t in
                      [("del", "🗑"), ("Batea", "Batea"), ("Fecha", "Fecha"), ("Rto. Planta", "Rto. Planta"),
                       ("Remito Proveedor", "Remito proveedor"), ("Producto", "Producto"), ("Kilos", "Kilos"),
                       ("Finos", "Aserrín lab %" if self.gyg else "Finos lab %"), ("Hum", "Humedad lab %"),
                       ("DescMan", "Desc. gran. %"), ("PenMan", "Desc. hum. %")]
                      + [("Desc", "resultado")] + ([("Punto", "Punto de entrega")] if self.multi else [])
                      + [("obs", "")]])
        self.filas_w = []
        self.w_tabla = W.VBox([tit])
        for _, f in c["filas"].iterrows():
            self._agregar_fila_w(dict(f), origen="cert", obs=obs_id.get(f["_id"], []))
        self.w_tn_pantalla = W.HTML()
        self._actualizar_tn()

        self.w_nuevo_rem = W.Text(placeholder="N° remito de planta, ej. 202511-12614", layout=_L("300px"))
        b_traer = W.Button(description="Traer remito del maestro", icon="search", layout=_L("230px"))
        b_vacia = W.Button(description="Agregar fila vacía", icon="plus", layout=_L("180px"))
        b_traer.on_click(lambda b: self._seguro(self._traer))
        b_vacia.on_click(lambda b: self._agregar_fila_w(
            {"Batea": "", "Fecha": c["hasta"], "Rto. Planta": "", "Remito Proveedor": "", "Producto": "CHIP",
             "Kilos": 0.0, "_id": None}, origen="manual"))
        agregar = W.HBox([b_vacia, self.w_nuevo_rem, b_traer])

        # Humedad y granulometría
        self.w_hum = {}
        self.w_cfgdesc = {}
        filas_h = []
        if True:   # opciones de descuento del proveedor: siempre visibles
            prov_cfg = self._cfg_unidad(c)
            g = lambda col: str(prov_cfg.get(col, "") if hasattr(prov_cfg, "get") else "").strip()
            ref = None
            self.w_cfgdesc = {
                "DESCUENTA HUMEDAD": W.Checkbox(value=g("DESCUENTA HUMEDAD").upper() == "SI", indent=False,
                                                description="Descuenta HUMEDAD", layout=_L("190px")),
                "TOLERANCIA HUMEDAD %": W.Text(value=g("TOLERANCIA HUMEDAD %"), placeholder="0",
                                               description="tolerancia (+ puntos %):", layout=_L("250px"),
                                               style={"description_width": "160px"}),
                "LIMITE HUMEDAD %": W.Text(value=g("LIMITE HUMEDAD %"), placeholder="el del laboratorio",
                                           description="límite fijo %:", layout=_L("250px"),
                                           style={"description_width": "100px"}),
                "DESCUENTA GRANULOMETRIA": W.Checkbox(value=g("DESCUENTA GRANULOMETRIA").upper() != "NO", indent=False,
                                                      description="Descuenta GRANULOMETRÍA", layout=_L("210px")),
                "TOLERANCIA FINOS %": W.Text(value=g("TOLERANCIA FINOS %"),
                                             placeholder=f"{cb.decimal(cb.FINOS_TOLERANCIA, 0)} (general)",
                                             description="tolerancia finos %:", layout=_L("250px"),
                                             style={"description_width": "130px"}),
            }
            self.w_sin_gran = W.Checkbox(value=bool(meta.get("sin_gran")), indent=False, layout=_L("420px"),
                                         description="Quitar el descuento de GRANULOMETRÍA en esta certificación")
            self.w_sin_hum = W.Checkbox(value=bool(meta.get("sin_hum")), indent=False, layout=_L("420px"),
                                        description="Quitar el descuento de HUMEDAD en esta certificación")
            filas_h.append(W.HTML("<b>Solo en esta certificación</b> <small>(pide motivo y queda registrado · para un "
                                  "camión puntual usá las columnas <i>Desc. gran. %</i> y <i>Desc. hum. %</i>"
                                  " de la tabla: escribí el % "
                                  "o 0 para quitarlo; borralo para volver al automático)</small>"))
            filas_h.append(W.HBox([self.w_sin_gran, self.w_sin_hum]))
            filas_h.append(W.HTML("<b>Qué se le descuenta a este proveedor</b> <small>(vale para todos sus períodos y se "
                                  "guarda en el config · cambiarlo pide motivo)</small>"))
            filas_h.append(W.HBox([self.w_cfgdesc["DESCUENTA HUMEDAD"], self.w_cfgdesc["TOLERANCIA HUMEDAD %"],
                                   self.w_cfgdesc["LIMITE HUMEDAD %"]]))
            filas_h.append(W.HBox([self.w_cfgdesc["DESCUENTA GRANULOMETRIA"], self.w_cfgdesc["TOLERANCIA FINOS %"],
                                   W.HTML(f"<small>&nbsp;por camión: finos hasta la tolerancia → 0 · hasta "
                                          f"{cb.decimal(cb.FINOS_TOTALIDAD, 0)} % → lo que excede · más de "
                                          f"{cb.decimal(cb.FINOS_TOTALIDAD, 0)} % → todos los finos · lo descontado se "
                                          "paga como ASERRÍN</small>")]))
            if self.gyg:
                filas_h.append(W.HTML(f"<small><b>Formato GYG</b>: humedad y granulometría <b>camión por camión</b>. "
                                      f"Aserrín: menos de {cb.decimal(c['tol_finos'], 0)} % (Alto) → 0 · hasta "
                                      f"{cb.decimal(c['muy_alto'], 0)} % (Muy Alto) → lo que excede · desde "
                                      f"{cb.decimal(c['muy_alto'], 0)} % → todo. Humedad: se descuenta lo que supera "
                                      "la referencia del día. Los valores de cada camión se corrigen en la tabla.</small>"))
            if c.get("desc_gran") and c.get("con_humedad"):
                fl = c["filas"]
                n_m, n_d = int(fl["Finos"].notna().sum()), int((fl["DescPct"] > 0).sum())
                filas_h.append(W.HTML(f"<small>Granulometría del período: {n_m} camiones con muestra · {n_d} con "
                                      f"descuento · <b>{cb.decimal(fl['DescKg'].sum() / 1000)} TN</b> de CHIP pasan a "
                                      "ASERRÍN. Los finos de cada camión se corrigen en la tabla de remitos.</small>"))
            if not c.get("con_humedad"):
                filas_h.append(W.HTML("<small style=color:#B26B00>⚠ No se cargó la planilla de calidad: los descuentos "
                                      "no se pueden calcular. Subila en el Paso 3 o en el Paso 5b.</small>"))
            elif any(hh["aplica_hum"] for hh in c["humedad"].values()):
                filas_h.append(W.HTML("<b>Humedad del período</b> <small>(promedio de las muestras del laboratorio · "
                                      "si la corregís a mano pide motivo)</small>"))
            for t, hh in ([(t, hh) for t, hh in c["humedad"].items() if hh["aplica_hum"]] if c.get("con_humedad") else []):
                base_l = None if hh["L"] is None else hh["L"] - hh["tol"]
                wh = W.FloatText(value=round(hh["H"] * 100, 2) if hh["H"] is not None else 0.0,
                                 description=f"{t} humedad %:", layout=_L("230px"), style={"description_width": "120px"})
                wl = W.FloatText(value=round(base_l * 100, 2) if base_l is not None else 0.0,
                                 description="límite %:", layout=_L("160px"), style={"description_width": "60px"})
                self.w_hum[t] = (wh, wl, dict(hh, L=base_l))
                partes = []
                if hh["aplica_hum"]:
                    partes.append(f"humedad {hh['muestras']} muestras, límite {cb._txt_limite(hh)} → "
                                  f"<b>−{cb.decimal(hh['desc_hum'] * 100)} %</b>")
                else:
                    partes.append("humedad: no se descuenta")
                partes.append(f"a pagar <b>{cb.decimal(hh['tn_pagar'])} TN</b> de {cb.decimal(hh['tn'])}")
                filas_h.append(W.HBox([wh, wl]))
                filas_h.append(W.HTML("<small>&nbsp;&nbsp;" + " · ".join(partes) + "</small>"))

        # Precios: todos los productos del cuadro, editables (cada proveedor tiene los suyos)
        self.w_precios, self.precios_base = {}, {}
        etiqueta = cb.etiqueta_gyg if self.gyg else cb.nombre_producto
        productos = list(dict.fromkeys(list(c["precios"]) + cb.PRODUCTOS_CUADRO))
        st = {"description_width": "110px"}
        filas_p = [W.HBox([W.HTML("<b>Producto</b>", layout=_L("110px")), W.HTML("<b>$ por TN</b>", layout=_L("150px")),
                           W.HTML("<b>TN a pagar</b>", layout=_L("110px")), W.HTML("<b>Subtotal</b>", layout=_L("150px")),
                           W.HTML("<b>De dónde sale</b>")])]
        for t in productos:
            entregado = t in c["precios"]
            p = c["precios"][t] if entregado else cb.buscar_precio(self.precios, c["grupo"], t, c["tn_grupo"])
            self.precios_base[t] = p
            w = W.FloatText(value=p if p is not None else 0.0, layout=_L("150px"))
            self.w_precios[t] = w
            tn = cb.tn_pagar(c, t) if entregado else 0.0
            origen = (c["origen_precios"][t] if entregado else ("config" if p is not None else "sin precio cargado"))
            falta = entregado and p is None
            filas_p.append(W.HBox([W.HTML(f"<b>{etiqueta(t)}</b>", layout=_L("110px")), w,
                                   W.HTML(cb.decimal(tn) if entregado else "—", layout=_L("110px")),
                                   W.HTML(cb.pesos(tn * p) if (entregado and p) else "", layout=_L("150px")),
                                   W.HTML(f"<small>{origen}</small>" + (" <b style=color:red>FALTA</b>" if falta else ""))]))
        self.w_guardar_cfg = W.RadioButtons(
            options=[("Guardar para este proveedor (se usan en las próximas certificaciones)", True),
                     ("Usar solo en esta certificación", False)], value=not meta.get("precios"), layout=_L("620px"))
        self.w_importe = W.HTML(f"<b style='font-size:15px;background:#FFFF00;padding:2px 8px'>Importe a facturar: "
                                f"{cb.pesos(c['importe'])}</b>")
        nota_g = (f"<small>Grupo de precios: <b>{c['grupo']}</b>"
                  + (f" · los comparten: {', '.join(self._grupo_comparten(c))}" if self._grupo_comparten(c) else "")
                  + " · si hay tramos por TN del mes, se cambia el del tramo actual "
                  f"({cb.decimal(c['tn_grupo'])} TN).</small>")
        descuentos_box = W.VBox(filas_h)
        precios_box = W.VBox(filas_p + [W.HTML(nota_g), self.w_guardar_cfg, self.w_importe])

        # Observaciones
        lineas = []
        for o in sorted(c["obs"], key=lambda o: (not o["bloq"], o["sev"] != cb.CRIT)):
            col = "#C00000" if o["bloq"] else ("#B26B00" if o["sev"] == cb.CRIT else "#666")
            tag = "BLOQUEANTE" if o["bloq"] else o["sev"]
            lineas.append(f"<li style='color:{col}'><b>{tag}</b> {('· ' + o['rem'] + ' · ') if o['rem'] else ''}{o['msg']}</li>")
        obs_html = W.HTML("<ul style='margin:2px'>" + "".join(dict.fromkeys(lineas)) + "</ul>" if lineas
                          else "<span style=color:green>✔ Sin observaciones</span>")
        revisables = cb.criticas_revisables(c)
        ack = meta.get("ack", {})
        self.w_ack = W.Checkbox(value=bool(revisables) and set(revisables) <= set(ack.get("msgs", [])), indent=False,
                                description="Revisé las observaciones críticas y la certificación es correcta")
        self.w_ack_com = W.Textarea(value=ack.get("comentario", ""), placeholder="Comentario obligatorio: qué revisaste "
                                    "(ej. 'el proveedor confirma que un remito cubre 7 viajes')", layout=_L("760px", height="54px"))
        ack_box = W.VBox([self.w_ack, self.w_ack_com]) if revisables else W.HTML("")

        self.w_motivo = W.Textarea(placeholder="Motivo de los cambios (obligatorio si cambiás remitos, descuentos o "
                                   "valores del laboratorio)", layout=_L("760px", height="54px"))
        b_guardar = W.Button(description="Guardar cambios", icon="save", button_style="info", layout=_L("190px"))
        b_conf = W.Button(description="Confirmar y generar PDF", icon="check", button_style="success", layout=_L("240px"))
        b_sig = W.Button(description="Siguiente pendiente", icon="arrow-right", layout=_L("190px"))
        b_desc = W.Button(description="Descartar cambios", icon="undo", layout=_L("170px"))
        b_guardar.on_click(lambda b: self._seguro(self._guardar))
        b_conf.on_click(lambda b: self._seguro(self._confirmar))
        b_sig.on_click(lambda b: self._seguro(self._siguiente))
        b_desc.on_click(lambda b: self._seguro(lambda: self._render(self.key)))

        sec = lambda t: W.HTML(f"<h4 style='margin:12px 0 4px;border-bottom:2px solid #3A7D44'>{t}</h4>")
        self.w_panel.children = [cab, sec("1 · Encabezado"), encab,
                                 sec(f"2 · Remitos ({len(c['filas'])})"), self.w_tabla, self.w_tn_pantalla, agregar,
                                 sec("3 · Descuentos de humedad y granulometría"), descuentos_box,
                                 sec("4 · Precios ($ por TN)"), precios_box,
                                 sec("5 · Observaciones"), obs_html, ack_box,
                                 sec("6 · Guardar y confirmar"), self.w_motivo,
                                 W.HBox([b_guardar, b_conf, b_sig, b_desc])]

    def _seguro(self, fn):
        """Ejecuta la acción de un botón y, si falla, muestra el error en pantalla (Colab lo oculta)."""
        import traceback
        try:
            fn()
        except Exception as e:
            with self.w_out:
                self.w_out.clear_output()
                display(W.HTML(f"<b style=color:red>Error: {type(e).__name__}: {e}</b><br><small>Copiá este detalle si "
                               "necesitás ayuda:</small><pre style='font-size:11px'>"
                               + traceback.format_exc()[-2500:].replace("<", "&lt;") + "</pre>"))

    def _cambios_descuentos(self, prov):
        """(columna, valor nuevo, valor anterior) de la configuración de descuentos que cambió en pantalla."""
        out = []
        if not self.w_cfgdesc:
            return out
        actual = self._cfg_unidad(self.certs[self.key])
        for col, w in self.w_cfgdesc.items():
            viejo = str(actual.get(col, "") if hasattr(actual, "get") else "").strip()
            if isinstance(w, W.Checkbox):
                nuevo = "SI" if w.value else "NO"
                defecto = "NO" if col == "DESCUENTA HUMEDAD" else "SI"
                if (viejo.upper() or defecto) != nuevo:
                    out.append((col, nuevo, viejo))
            else:
                nuevo = w.value.strip().replace("%", "")
                if nuevo != viejo.replace("%", ""):
                    out.append((col, nuevo, viejo))
        return out

    def _grupo_comparten(self, c):
        """Otros proveedores (fuera de esta certificación) que usan el mismo grupo de precios."""
        if "GRUPO" not in self.conf.columns:
            return []
        g = self.conf["GRUPO"].astype(str).str.casefold() == str(c["grupo"]).casefold()
        return sorted(set(self.conf.index[g]) - set(c.get("miembros", [c["proveedor"]])), key=str.casefold)

    def _cfg_unidad(self, c):
        """Fila del config que manda en la certificación (el primer punto de entrega con CUIT)."""
        ms = [p for p in c.get("miembros", [c["proveedor"]]) if p in self.conf.index]
        con_cuit = [p for p in ms if str(self.conf.at[p, "CUIT"]).strip()]
        return self.conf.loc[(con_cuit or ms)[0]] if ms else {}

    def _miembros_cfg(self, c):
        return list(c.get("miembros") or [c["proveedor"]])

    @staticmethod
    def _txt_num(v, n=2):
        if v is None or pd.isna(v):
            return ""
        t = f"{float(v):.{n}f}".rstrip("0").rstrip(".")
        return t.replace(".", ",") if t else "0"

    @staticmethod
    def _tip(w, texto):
        for attr in ("tooltip", "description_tooltip"):
            if w.has_trait(attr):
                setattr(w, attr, texto)

    @staticmethod
    def _num(txt):
        """Texto de un campo → número (acepta coma y %); '' → None; inválido → 'ERROR'."""
        t = str(txt).strip().replace("%", "").replace(" ", "")
        if t == "":
            return None
        if "," in t:
            t = t.replace(".", "").replace(",", ".")
        try:
            return float(t)
        except ValueError:
            return "ERROR"

    def _manuales_cambiados(self, campo, base_val, base_man, escala=1.0):
        """Descuentos editados por camión: [(fila, nuevo valor | None = volver a automático | 'ERROR', anterior)]."""
        out = []
        for x in self.filas_w:
            w = x["w"]
            if x["origen"] != "cert" or w["del"].value or campo not in w or w[campo].disabled:
                continue
            nuevo = self._num(w[campo].value)
            antes = x["base"].get(base_val)
            antes = 0.0 if antes is None or pd.isna(antes) else float(antes) * escala
            era_manual = bool(x["base"].get(base_man))
            if nuevo == "ERROR":
                out.append((x, "ERROR", antes))
            elif nuevo is None:
                if era_manual:
                    out.append((x, None, antes))
            elif round(nuevo, 2) != round(antes, 2):
                out.append((x, nuevo, antes))
        return out

    @staticmethod
    def _txt_finos(v):
        return "" if v is None or pd.isna(v) else f"{v * 100:.1f}".replace(".", ",")

    def _finos_cambiados(self, campo="Finos"):
        """[(fila, valor nuevo como fracción o None, valor anterior)] de los finos (o humedad) editados en la tabla."""
        out = []
        for x in self.filas_w:
            if x["origen"] != "cert" or x["w"]["del"].value or campo not in x["w"]:
                continue
            txt = x["w"][campo].value.strip().replace("%", "").replace(",", ".")
            antes = x["base"].get(campo)
            antes = None if antes is None or pd.isna(antes) else float(antes)
            if txt == "":
                nuevo = None
            else:
                try:
                    nuevo = float(txt) / 100
                except ValueError:
                    out.append((x, "ERROR", antes))
                    continue
            if (nuevo is None) != (antes is None) or (nuevo is not None and round(nuevo * 100, 1) != round(antes * 100, 1)):
                out.append((x, nuevo, antes))
        return out

    def _cambios_valores(self, t):
        wh, wl, hh = self.w_hum[t]
        man = {}
        for w, k, base in ((wh, "H", hh["H"]), (wl, "L", hh["L"])):
            b = None if base is None else round(base * 100, 2)
            if w.value and (b is None or abs(w.value - b) > 0.001):
                man[k] = w.value / 100
        return man

    def _agregar_fila_w(self, f, origen, obs=()):
        nuevo = origen != "cert"
        w = {"del": W.Checkbox(value=False, indent=False, layout=_L(ANCHOS["del"])),
             "Batea": W.Text(value=str(f["Batea"]), layout=_L(ANCHOS["Batea"])),
             "Fecha": W.DatePicker(value=f["Fecha"], layout=_L(ANCHOS["Fecha"])),
             "Rto. Planta": W.Text(value=str(f["Rto. Planta"]), disabled=origen != "manual",
                                   layout=_L(ANCHOS["Rto. Planta"])),
             "Remito Proveedor": W.Text(value=str(f["Remito Proveedor"]), layout=_L(ANCHOS["Remito Proveedor"])),
             "Producto": W.Dropdown(options=list(dict.fromkeys(cb.TIPOLOGIAS + [f["Producto"]])), value=f["Producto"],
                                    layout=_L(ANCHOS["Producto"])),
             "Kilos": W.FloatText(value=float(f["Kilos"]), layout=_L(ANCHOS["Kilos"]))}
        # Granulometría del camión (editable solo en remitos ya guardados; vacío = sin muestra)
        hora, org = f.get("FinosHora") or "", f.get("FinosOrigen") or ""
        w["Finos"] = W.Text(value=self._txt_finos(f.get("Finos")), placeholder="—", disabled=origen != "cert",
                            layout=_L(ANCHOS["Finos"]))
        for attr in ("tooltip", "description_tooltip"):
            if w["Finos"].has_trait(attr):
                setattr(w["Finos"], attr, (f"muestra {hora} · {org}" if hora else org) or "sin muestra asignada")
        dp, dk = float(f.get("DescPct") or 0), float(f.get("DescKg") or 0)
        extras = [w["Finos"]]
        gyg = getattr(self, "gyg", False)
        if True:                       # humedad del laboratorio de cada camión (todos los proveedores)
            w["Hum"] = W.Text(value=self._txt_finos(f.get("Hum")), placeholder="—", disabled=origen != "cert",
                              layout=_L(ANCHOS["Hum"]))
            extras.append(w["Hum"])
        # Descuento aplicado (editable): se puede poner a mano o en 0 para quitarlo; vacío = automático
        cact = self.certs.get(self.key, {}) if getattr(self, "key", None) else {}
        es_chip = f.get("Producto") == "CHIP" and origen == "cert" and not cact.get("sin_gran")
        w["DescMan"] = W.Text(value=self._txt_num(dp) if es_chip else "", placeholder="auto",
                              disabled=not es_chip, layout=_L(ANCHOS["DescMan"]))
        self._tip(w["DescMan"], "Puesto a mano (borralo para volver al automático)" if f.get("DescManual")
                  else "Automático según el laboratorio · escribí otro % o 0 para quitarlo")
        extras.append(w["DescMan"])
        if True:                       # descuento de humedad por camión (todos los proveedores)
            es_ch = f.get("Producto") in ("CHIP", "ASERRIN") and origen == "cert" and not cact.get("sin_hum")
            pen = f.get("Pen") or 0.0
            w["PenMan"] = W.Text(value=self._txt_num((pen or 0) * 100) if es_ch else "", placeholder="auto",
                                 disabled=not es_ch, layout=_L(ANCHOS["PenMan"]))
            self._tip(w["PenMan"], "Puesto a mano (borralo para volver al automático)" if f.get("PenManual")
                      else "Automático: humedad medida − referencia · escribí otro % o 0 para quitarlo")
            extras.append(w["PenMan"])
            pen = f.get("Pen") or 0
            partes = (["✋"] if f.get("DescManual") or f.get("PenManual") else []) + \
                     ([f"{cb.decimal(dk, 0)} kg aserrín"] if dk > 0 else []) + \
                     ([f"−{cb.decimal(pen * 100, 2)} % hum."] if pen and pen > 0 else [])
            extras.append(W.HTML(f"<span style='background:#FFF2A8;padding:1px 4px'>{' · '.join(partes)}</span>"
                                 if partes else "", layout=_L(ANCHOS["Desc"])))
        else:
            extras.append(W.HTML(f"<span style='background:#FFF2A8;padding:1px 4px'>{'✋ ' if f.get('DescManual') else ''}"
                                 f"{cb.decimal(dk, 1)} kg → aserrín</span>" if dp > 0 else
                                 ("✋ sin descuento" if f.get("DescManual") else ""), layout=_L(ANCHOS["Desc"])))
        if getattr(self, "multi", False):
            extras.append(W.HTML(f"<small>{f.get('Proveedor', '') or ''}</small>", layout=_L(ANCHOS["Punto"])))
        if obs:
            peor = "🔴" if any(o["sev"] == cb.CRIT for o in obs) else "🟠"
            txt = " | ".join(dict.fromkeys(o["msg"] for o in obs)).replace("'", "’")
            icono = W.HTML(f"<span title='{txt}' style='cursor:help'>{peor}</span>", layout=_L(ANCHOS["obs"]))
        else:
            icono = W.HTML("🆕" if nuevo else "", layout=_L(ANCHOS["obs"]))
        for k in ("Kilos", "del"):
            w[k].observe(lambda ch: self._actualizar_tn(), "value")
        caja = W.HBox([w[k] for k in ["del", "Batea", "Fecha", "Rto. Planta", "Remito Proveedor", "Producto", "Kilos"]]
                      + extras + [icono])
        self.filas_w.append({"w": w, "id": f.get("_id"), "origen": origen, "base": f})
        self.w_tabla.children = list(self.w_tabla.children) + [caja]
        if hasattr(self, "w_tn_pantalla"):
            self._actualizar_tn()

    def _actualizar_tn(self):
        tot = sum((x["w"]["Kilos"].value or 0) for x in self.filas_w if not x["w"]["del"].value)
        n = sum(1 for x in self.filas_w if not x["w"]["del"].value)
        self.w_tn_pantalla.value = f"<b>En pantalla: {n} remitos · {cb.decimal(tot / 1000)} TN</b>"

    def _msg(self, html, limpiar=True):
        with self.w_out:
            if limpiar:
                self.w_out.clear_output()
            display(W.HTML(html))

    def _traer(self):
        rem = self.w_nuevo_rem.value.strip()
        c = self.certs[self.key]
        if not rem:
            return self._msg("<b style=color:red>Escribí el N° de remito de planta.</b>")
        if any(x["w"]["Rto. Planta"].value == rem for x in self.filas_w):
            return self._msg(f"<b style=color:red>El remito {rem} ya está en la tabla.</b>")
        m = self.d[self.d[cb.C_REM_PLANTA] == rem]
        if len(m):
            for _, f in m.iterrows():
                base = {k: f[cb.CAMPOS[k]] for k in cb.FILA_COLS}
                base["Fecha"], base["_id"] = f[cb.C_DIA].date(), f["_id"]
                base["Proveedor"] = f[cb.C_PROV]
                self._agregar_fila_w(base, origen="maestro")
                fuera = not (c["desde"] <= base["Fecha"] <= c["hasta"])
                self._msg(f"Traído <b>{rem}</b>: proveedor en el maestro <b>{f[cb.C_PROV]}</b>, fecha "
                          f"<b>{base['Fecha']:%d/%m/%Y}</b>."
                          + ("<br><b style=color:#B26B00>La fecha está fuera de este período: corregila o el remito "
                             "no va a aparecer acá.</b>" if fuera else "")
                          + "<br>Completá el motivo y tocá <b>Guardar cambios</b>.")
            return
        r = self.raw[self.raw[cb.C_REM_PLANTA].apply(cb._txt) == rem]
        if len(r):
            f = r.iloc[0]
            base = {"Batea": cb._txt(f[cb.C_BATEA]), "Fecha": f[cb.C_DIA].date() if pd.notna(f[cb.C_DIA]) else c["hasta"],
                    "Rto. Planta": rem, "Remito Proveedor": cb._txt(f[cb.C_REM_PROV]),
                    "Producto": cb._canon_tipo(f[cb.C_TIPO]), "Kilos": float(pd.to_numeric(f[cb.C_NETO], errors="coerce") or 0),
                    "_id": None, "Proveedor": cb._txt(f[cb.C_PROV])}
            self._agregar_fila_w(base, origen="raw")
            return self._msg(f"Traído <b>{rem}</b> desde el maestro (estaba como movimiento "
                             f"'<b>{cb._txt(f[cb.C_MOV])}</b>' de <b>{base['Proveedor']}</b>). Revisalo, completá el motivo y guardá.")
        self._msg(f"<b style=color:red>No encontré el remito {rem} en el maestro.</b> Si es un remito real que no se "
                  "cargó, usá <i>Agregar fila vacía</i>.")

    # --------------------------------------------------------------- guardar
    def _guardar(self, silencioso=False):
        c = self.certs[self.key]
        prov = c["proveedor"]
        miembros = self._miembros_cfg(c)
        destino = miembros[0]          # punto de entrega al que se asignan los remitos agregados/traídos
        usuario = self.w_usuario.value.strip()
        motivo = self.w_motivo.value.strip()
        errores, cambios = [], []
        if not usuario:
            errores.append("Completá <b>Responsable</b> (arriba de todo).")
        ids_actuales = set(c["filas"]["_id"])
        nuevos_rem = []
        for x in self.filas_w:
            w = x["w"]
            if w["del"].value:
                continue
            rem = w["Rto. Planta"].value.strip()
            if w["Fecha"].value is None:
                errores.append(f"Remito {rem or '(nuevo)'}: falta la fecha.")
            if not w["Kilos"].value or w["Kilos"].value <= 0:
                errores.append(f"Remito {rem or '(nuevo)'}: los kilos deben ser mayores a 0.")
            elif w["Kilos"].value > cb.PESO_MAX * 1.2:
                errores.append(f"Remito {rem}: {cb.miles(w['Kilos'].value)} kg es imposible para un camión.")
            if w["Producto"].value not in cb.TIPOLOGIAS:
                errores.append(f"Remito {rem}: elegí un producto válido.")
            if x["origen"] == "manual":
                if not rem:
                    errores.append("Una fila nueva no tiene N° de remito de planta.")
                elif rem in nuevos_rem or (self.d[cb.C_REM_PLANTA] == rem).any():
                    errores.append(f"El remito {rem} ya existe: usá <i>Traer remito del maestro</i>.")
                nuevos_rem.append(rem)

        vals = lambda w: {"Batea": w["Batea"].value.strip(), "Fecha": w["Fecha"].value,
                          "Rto. Planta": w["Rto. Planta"].value.strip(),
                          "Remito Proveedor": w["Remito Proveedor"].value.strip(),
                          "Producto": w["Producto"].value, "Kilos": float(w["Kilos"].value or 0)}
        for x in self.filas_w:
            w, base = x["w"], x["base"]
            v = vals(w)
            if x["origen"] == "cert":
                if w["del"].value:
                    cambios.append(("eliminar", x["id"], v["Rto. Planta"], "", "", ""))
                    continue
                for k in ("Batea", "Fecha", "Remito Proveedor", "Producto", "Kilos"):
                    antes = base[k]
                    if (float(antes) != v[k]) if k == "Kilos" else (str(antes) != str(v[k])):
                        cambios.append(("editar", x["id"], v["Rto. Planta"], k, antes, v[k]))
            elif w["del"].value:
                continue
            elif x["origen"] == "maestro":
                if base["Proveedor"] not in miembros:
                    cambios.append(("editar", x["id"], v["Rto. Planta"], "Proveedor", base["Proveedor"], destino))
                for k in ("Batea", "Fecha", "Remito Proveedor", "Producto", "Kilos"):
                    if (float(base[k]) != v[k]) if k == "Kilos" else (str(base[k]) != str(v[k])):
                        cambios.append(("editar", x["id"], v["Rto. Planta"], k, base[k], v[k]))
                if not any(cm[1] == x["id"] for cm in cambios):
                    cambios.append(("editar", x["id"], v["Rto. Planta"], "Proveedor", base["Proveedor"],
                                    base["Proveedor"] if base["Proveedor"] in miembros else destino))
            else:   # raw o manual → remito agregado
                cambios.append(("agregar", None, v["Rto. Planta"], "", "", v))
        if cambios and not motivo:
            errores.append("Escribí el <b>motivo</b> de los cambios en remitos.")
        if self.w_ack.value and cb.criticas_revisables(c) and not self.w_ack_com.value.strip():
            errores.append("Para marcar las observaciones como revisadas escribí un <b>comentario</b>.")
        cuit = self.w_cuit.value.strip()
        if cuit and not cb.cuit_valido(cuit):
            errores.append(f"El CUIT <b>{cuit}</b> no es válido (dígito verificador). Revisalo.")
        for t, w in self.w_precios.items():
            if w.value is not None and w.value < 0:
                errores.append(f"Precio de {t} negativo.")
        cambios_cfg = self._cambios_descuentos(prov)
        if cambios_cfg and not motivo:
            errores.append("Escribí el <b>motivo</b> del cambio en los descuentos del proveedor.")
        for campo, nuevo, _ in cambios_cfg:
            if campo == "LIMITE HUMEDAD %" and nuevo and not (isinstance(cb.puntos(nuevo), float)
                                                             and 0.2 <= cb.puntos(nuevo) < 0.9):
                errores.append(f"Límite fijo de humedad '{nuevo}': tiene que ser un % entre 20 y 90.")
            elif "TOLERANCIA" in campo and nuevo and not isinstance(cb.puntos(nuevo), float):
                errores.append(f"{campo}: '{nuevo}' no es un número.")
            elif "TOLERANCIA" in campo and nuevo and not (0 <= cb.puntos(nuevo) < (0.2 if "HUMEDAD" in campo else 1)):
                errores.append(f"{campo} fuera de rango.")
        finos_mod = self._finos_cambiados()
        desc_mod = self._manuales_cambiados("DescMan", "DescPct", "DescManual")
        pen_mod = self._manuales_cambiados("PenMan", "Pen", "PenManual", escala=100)
        for x, nuevo, _ in desc_mod + pen_mod:
            rem = x["w"]["Rto. Planta"].value
            if nuevo == "ERROR":
                errores.append(f"Remito {rem}: el descuento escrito no es un número.")
            elif nuevo is not None and not (0 <= nuevo <= 100):
                errores.append(f"Remito {rem}: el descuento tiene que estar entre 0 y 100 %.")
        sw_mod = [(k, w.value) for k, w in (("sin_gran", self.w_sin_gran), ("sin_hum", self.w_sin_hum))
                  if bool(self.estado["meta"].get(self.key, {}).get(k)) != w.value]
        if (desc_mod or pen_mod or sw_mod) and not motivo:
            errores.append("Escribí el <b>motivo</b> del cambio en los descuentos.")
        for x, nuevo, _ in finos_mod:
            rem = x["w"]["Rto. Planta"].value
            if nuevo == "ERROR":
                errores.append(f"Remito {rem}: los finos '{x['w']['Finos'].value}' no son un número.")
            elif nuevo is not None and not (0 <= nuevo < 1):
                errores.append(f"Remito {rem}: finos fuera de rango (0 a 100 %).")
        if finos_mod and not motivo:
            errores.append("Escribí el <b>motivo</b> de la corrección de finos (granulometría).")
        hum_mod = self._finos_cambiados("Hum")
        for x, nuevo, _ in hum_mod:
            rem = x["w"]["Rto. Planta"].value
            if nuevo == "ERROR":
                errores.append(f"Remito {rem}: la humedad '{x['w']['Hum'].value}' no es un número.")
            elif nuevo is not None and not (0 < nuevo < 1):
                errores.append(f"Remito {rem}: humedad fuera de rango (0 a 100 %).")
        if hum_mod and not motivo:
            errores.append("Escribí el <b>motivo</b> de la corrección de humedad del camión.")
        for t, (wh, wl, hh) in self.w_hum.items():
            if self._cambios_valores(t) and not motivo:
                errores.append(f"Escribí el <b>motivo</b> de la corrección de humedad de {t}.")
            for w, n in ((wh, "humedad"), (wl, "límite")):
                if w.value and not (0 < w.value < 100):
                    errores.append(f"{n.capitalize()} de {t} fuera de rango (0 a 100 %).")
        if errores:
            self._msg("<b style=color:red>No se guardó:</b><ul>" + "".join(f"<li>{e}</li>" for e in errores) + "</ul>")
            return False

        corr, meta = self.estado["corr"], self.estado["meta"].setdefault(self.key, {})
        for accion, _id, rem, campo, antes, despues in cambios:
            if accion == "eliminar":
                corr["eliminados"][_id] = motivo
                self._log("Quitar remito", self.key, rem, "", "", "", motivo)
            elif accion == "editar":
                ed = corr["editados"].setdefault(_id, {})
                ed[campo] = despues.isoformat() if isinstance(despues, date) else despues
                self._log("Corregir remito", self.key, rem, campo,
                          antes.strftime("%d/%m/%Y") if isinstance(antes, date) else antes,
                          despues.strftime("%d/%m/%Y") if isinstance(despues, date) else despues, motivo)
            else:
                v = dict(despues, Proveedor=destino, Fecha=despues["Fecha"].isoformat())
                corr["agregados"][f"{rem}#A{len(corr['agregados']) + 1}"] = v
                self._log("Agregar remito", self.key, rem, "", "", f"{v['Kilos']:.0f} kg {v['Producto']} {v['Fecha']}", motivo)

        # Finos por camión corregidos a mano (quedan registrados)
        for x, nuevo, antes in finos_mod:
            corr.setdefault("finos", {})[x["id"]] = nuevo
            self._log("Finos (granulometría) corregidos a mano", self.key, x["w"]["Rto. Planta"].value, "Finos %",
                      "sin muestra" if antes is None else f"{antes * 100:.1f} %",
                      "sin muestra" if nuevo is None else f"{nuevo * 100:.1f} %", motivo)

        for x, nuevo, antes in hum_mod:
            corr.setdefault("humedad_camion", {})[x["id"]] = nuevo
            self._log("Humedad del camión corregida a mano", self.key, x["w"]["Rto. Planta"].value, "Humedad %",
                      "sin muestra" if antes is None else f"{antes * 100:.1f} %",
                      "sin muestra" if nuevo is None else f"{nuevo * 100:.1f} %", motivo)

        # Encabezado → config de todos los puntos de entrega de la certificación (vale para todos sus períodos)
        actual = self._cfg_unidad(c)
        g0 = lambda col: str(actual.get(col, "") if hasattr(actual, "get") else "").strip()
        for m in miembros:
            if m not in self.conf.index:
                fila = {col: "" for col in self.conf.columns}
                fila.update({"RAZÓN SOCIAL": g0("RAZÓN SOCIAL") or prov, "CUIT": g0("CUIT"), "GRUPO": g0("GRUPO") or prov,
                             "CERTIFICAR": "SI", "DESCUENTA HUMEDAD": g0("DESCUENTA HUMEDAD") or "NO",
                             "DESCUENTA GRANULOMETRIA": g0("DESCUENTA GRANULOMETRIA") or "SI",
                             "CERTIFICAR CON": prov if prov != m else ""})
                self.conf.loc[m] = fila
                self._log("Alta de proveedor en config", self.key, "", "", "", m)
        for campo, w in (("RAZÓN SOCIAL", self.w_razon), ("CUIT", self.w_cuit)):
            nuevo, viejo = w.value.strip(), g0(campo)
            if nuevo != viejo or any(str(self.conf.at[m, campo]).strip() != nuevo for m in miembros):
                for m in miembros:
                    self.conf.at[m, campo] = nuevo
                if nuevo != viejo:
                    self._log("Config proveedor", self.key, "", campo, viejo, nuevo)
        # Precios
        # Descuentos quitados o puestos a mano (por camión o en toda la certificación)
        for x, nuevo, antes in desc_mod:
            if nuevo is None:
                corr.setdefault("desc_gran", {}).pop(x["id"], None)
            else:
                corr.setdefault("desc_gran", {})[x["id"]] = nuevo
            self._log("Descuento de granulometría a mano", self.key, x["w"]["Rto. Planta"].value, "Desc. %",
                      f"{antes:g} %", "automático" if nuevo is None else f"{nuevo:g} %", motivo)
        for x, nuevo, antes in pen_mod:
            if nuevo is None:
                corr.setdefault("pen_hum", {}).pop(x["id"], None)
            else:
                corr.setdefault("pen_hum", {})[x["id"]] = nuevo / 100
            self._log("Descuento de humedad a mano", self.key, x["w"]["Rto. Planta"].value, "Penalización %",
                      f"{antes:g} %", "automático" if nuevo is None else f"{nuevo:g} %", motivo)
        for k, v in sw_mod:
            if v:
                meta[k] = True
            else:
                meta.pop(k, None)
            self._log("Descuento " + ("de granulometría" if k == "sin_gran" else "de humedad")
                      + (" QUITADO en la certificación" if v else " vuelve a aplicarse"), self.key, "", "", "", "", motivo)
        for t, w in self.w_precios.items():
            viejo = self.precios_base.get(t)
            if w.value and (viejo is None or abs(w.value - viejo) > 0.001):
                if self.w_guardar_cfg.value:
                    self.precios = cb.fijar_precio(self.precios, c["grupo"], t, c["tn_grupo"], w.value)
                    meta.get("precios", {}).pop(t, None)
                    self._log("Precio en config", self.key, "", f"{c['grupo']} / {t}", viejo or "", w.value)
                else:
                    meta.setdefault("precios", {})[t] = w.value
                    self._log("Precio solo para esta certificación", self.key, "", t, viejo or "", w.value)
        # Qué se descuenta y tolerancias del proveedor → config
        for campo, nuevo, viejo in cambios_cfg:
            if campo not in self.conf.columns:
                self.conf[campo] = ""
            for m in miembros:
                self.conf.at[m, campo] = nuevo
            self._log("Descuentos del proveedor (config)", self.key, "", campo, viejo, nuevo, motivo)
        # Humedad / finos corregidos a mano (quedan registrados)
        for t in self.w_hum:
            man = self._cambios_valores(t)
            previo = meta.get("humedad", {}).get(t) or {}
            if man and dict(previo, **man) != previo:
                meta.setdefault("humedad", {})[t] = dict(previo, **man)
                for k, v in man.items():
                    self._log("Valor de laboratorio corregido a mano", self.key, "", f"{t} · "
                              + {"H": "humedad", "L": "límite humedad"}[k], "", f"{v * 100:g} %", motivo)
        # TN grupo, número, fecha
        if abs(self.w_tng.value - c["tn_grupo_auto"]) > 0.005:
            if abs(self.w_tng.value - c["tn_grupo"]) > 0.005:
                self._log("TN grupo manual", self.key, "", "TN grupo", c["tn_grupo"], self.w_tng.value)
            meta["tn_grupo"] = self.w_tng.value
        else:
            meta.pop("tn_grupo", None)
        if self.w_num.value.strip() != c["numero"]:
            self._log("Número de certificación", self.key, "", "Nº", c["numero"], self.w_num.value.strip())
            meta["numero"] = self.w_num.value.strip()
        if self.w_fecha.value and self.w_fecha.value != c["fecha"]:
            meta["fecha"] = self.w_fecha.value.isoformat()
        # Revisión de observaciones
        if self.w_ack.value and cb.criticas_revisables(c):
            meta["ack"] = {"msgs": cb.criticas_revisables(c), "comentario": self.w_ack_com.value.strip(),
                           "usuario": usuario, "fecha": datetime.now().isoformat(timespec="seconds")}
            self._log("Observaciones revisadas", self.key, "", "", "", self.w_ack_com.value.strip())

        self.reconstruir()
        self.guardar_estado()
        if not silencioso:
            fuera = [x["w"]["Rto. Planta"].value for x in self.filas_w if not x["w"]["del"].value
                     and x["w"]["Fecha"].value and not (c["desde"] <= x["w"]["Fecha"].value <= c["hasta"])]
            self._msg(f"<b style=color:green>✔ Guardado.</b> {len(cambios)} cambio(s) en remitos."
                      + (f"<br>Los remitos {', '.join(fuera)} quedaron con fecha fuera de este período y "
                         "pasaron a la certificación que corresponde." if fuera else ""))
        self._refrescar_lista(mantener=self.key)
        return True

    # ------------------------------------------------------------- confirmar
    def _confirmar(self):
        key = self.key
        if not self._guardar(silencioso=True):
            return
        ok, msg, rutas = self.confirmar_cert(key, self.w_usuario.value.strip())
        self._msg(msg)
        if ok and self.descargar_pdf:
            self._descargar(rutas)
        self._refrescar_lista(mantener=key)

    def _descargar(self, rutas):
        """Descarga archivos desde el área de mensajes (en Colab las descargas de un botón tienen que salir de ahí)."""
        if not _colab_files:
            return
        with self.w_out:
            for r in rutas:
                try:
                    _colab_files.download(str(r))
                except Exception as e:
                    display(W.HTML(f"<small style=color:#B26B00>No se pudo descargar {Path(r).name} ({e}). "
                                   "Queda en el zip del Paso 7.</small>"))

    def confirmar_cert(self, key, usuario=""):
        """Confirma una certificación: número, PDF y Excel. Devuelve (ok, mensaje_html, [rutas])."""
        if key not in self.certs:
            return False, "<b style=color:red>Esta certificación no existe o quedó sin remitos.</b>", []
        c = self.certs[key]
        meta = self.estado["meta"].setdefault(key, {})
        if c["estado"] == "CONFIRMADA":
            return False, (f"Ya estaba confirmada (Nº {c['numero']}) y no cambió.<br>PDF: {Path(meta.get('pdf', '')).name}"
                           f"<br>Excel: {Path(meta.get('excel', '')).name}"), []
        bloq = [o["msg"] for o in c["obs"] if o["bloq"]]
        if bloq:
            return False, ("<b style=color:red>⛔ No se puede confirmar. Corregí primero:</b><ul>"
                           + "".join(f"<li>{m}</li>" for m in dict.fromkeys(bloq)) + "</ul>"), []
        if c["estado"] == "REVISAR":
            return False, ("<b style=color:#B26B00>⚠️ Hay observaciones críticas sin revisar.</b> Revisalas, marcá "
                           "<i>Revisé las observaciones críticas</i>, escribí el comentario y volvé a confirmar."), []
        if not c["numero"]:
            meta["numero"] = f"{self.estado['prefijo']}-{self.estado['proximo']:08d}"
            self.estado["proximo"] += 1
        meta.setdefault("fecha", c["fecha"].isoformat())
        self.recalcular(key)
        c = self.certs[key]
        nombre = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in c["proveedor"]).strip("_")
        base = f"{c['numero']}_{nombre}_{c['periodo']}_{cb.MESES[c['desde'].month]}_{c['desde'].year}"
        carpeta = self.salida / "PDF" / c["periodo"]
        carpeta.mkdir(parents=True, exist_ok=True)
        ruta = carpeta / f"{base}.pdf"
        cb.pdf_certificacion(ruta, c, self.logo)
        regenerada = bool(meta.get("huella"))
        meta.update({"huella": cb.huella(c), "pdf": str(ruta), "confirmado_por": usuario,
                     "confirmado_el": datetime.now().isoformat(timespec="seconds")})
        self.estado["log"].append([datetime.now().strftime("%d/%m/%Y %H:%M:%S"), usuario,
                                   "PDF regenerado" if regenerada else "Confirmada", key, "", "Nº", "", c["numero"],
                                   f"{cb.decimal(c['tn'])} TN · {cb.pesos(c['importe'])}"])
        self.recalcular(key)
        c = self.certs[key]
        carpeta_x = self.salida / "EXCEL" / c["periodo"]
        carpeta_x.mkdir(parents=True, exist_ok=True)
        ruta_x = carpeta_x / f"{base}.xlsx"
        cb.excel_certificacion(ruta_x, c, self.logo)
        meta["excel"] = str(ruta_x)
        self.guardar_estado()
        tn_p = sum(cb.tn_pagar(c, t) for t in c["precios"])
        desc = f" · {cb.decimal(tn_p)} TN a pagar (descuentos −{cb.decimal(c['tn'] - tn_p)})" if abs(c["tn"] - tn_p) > 0.004 else ""
        msg = (f"<b style=color:green>✅ Confirmada Nº {c['numero']}</b> · {cb.decimal(c['tn'])} TN{desc} · "
               f"<b>{cb.pesos(c['importe'])}</b><br>PDF: {ruta.name}<br>Excel: {ruta_x.name}")
        return True, msg, [ruta, ruta_x]

    def marcar_revisadas(self, usuario, comentario, keys):
        """Sin editor: marca como revisadas las observaciones críticas (no bloqueantes) de esas certificaciones."""
        for k in keys:
            c = self.certs[k]
            crit = cb.criticas_revisables(c)
            if crit:
                self.estado["meta"].setdefault(k, {})["ack"] = {"msgs": crit, "comentario": comentario, "usuario": usuario,
                                                               "fecha": datetime.now().isoformat(timespec="seconds")}
                self.estado["log"].append([datetime.now().strftime("%d/%m/%Y %H:%M:%S"), usuario,
                                           "Observaciones revisadas (sin editor)", k, "", "", "", comentario, ""])
            self.recalcular(k)

    def confirmar_lote(self, usuario, proveedores=None, periodos=None):
        """Sin editor: confirma todas las certificaciones LISTAS (o las elegidas) y genera PDF + Excel.
        Las que no se pueden confirmar se listan con el motivo."""
        hechas, pendientes, rutas = [], [], []
        for k, c in sorted(self.certs.items()):
            if proveedores and c["proveedor"] not in proveedores and not set(c.get("miembros", [])) & set(proveedores):
                continue
            if periodos and c["periodo"] not in periodos:
                continue
            if c["estado"] == "CONFIRMADA":
                continue
            ok, msg, rr = self.confirmar_cert(k, usuario)
            if ok:
                hechas.append(k)
                rutas += rr
            else:
                motivos = [o["msg"] for o in c["obs"] if o["bloq"]] or (
                    ["observaciones críticas sin revisar: revisalas en el editor"] if c["estado"] == "REVISAR" else [])
                pendientes.append((k, list(dict.fromkeys(motivos))))
        return hechas, pendientes, rutas

    def _siguiente(self):
        ks = [k for k, _ in self.w_sel.options]
        pend = [k for k in ks if self.certs[k]["estado"] != "CONFIRMADA"]
        if not pend:
            return self._msg("No quedan certificaciones pendientes en esta vista. 🎉")
        i = ks.index(self.key) if self.key in ks else -1
        sig = next((k for k in ks[i + 1:] if k in pend), pend[0])
        self.w_sel.value = sig

    # --------------------------------------------------------------- exportar
    def archivos_confirmados(self):
        """PDF y Excel de cada certificación confirmada (los que existen en disco)."""
        pdfs, xls = [], []
        for k, c in sorted(self.certs.items(), key=lambda x: (x[1]["numero"], x[0])):
            if c["estado"] != "CONFIRMADA":
                continue
            m = self.estado["meta"].get(k, {})
            if m.get("pdf") and Path(m["pdf"]).exists():
                pdfs.append(Path(m["pdf"]))
            if m.get("excel") and Path(m["excel"]).exists():
                xls.append(Path(m["excel"]))
        return pdfs, xls

    def descargar_archivos(self, rutas):
        """Descarga cada archivo por separado (sin zip)."""
        if not _colab_files:
            for r in rutas:
                print("   ", r)
            return
        import time
        for r in rutas:
            _colab_files.download(str(r))
            time.sleep(0.4)          # el navegador puede pedir 'Permitir descargas múltiples'

    def guardar_cierre(self):
        """Escribe el libro Excel por período (todos los proveedores) y el RESUMEN de control. Sin descargas."""
        anio, mes = self.estado["anio"], self.estado["mes"]
        libros = []
        for per in ("1Q", "2Q", "Mensual"):
            cs = [c for c in self.certs.values() if c["periodo"] == per]
            if cs:
                cs.sort(key=lambda c: c["proveedor"].casefold())
                ruta = self.salida / f"Certificaciones_{per}_{cb.MESES[mes]}_{anio}.xlsx"
                try:
                    cb.escribir_libro(ruta, cs, self.logo)
                except PermissionError:
                    print(f"⚠ {ruta.name} está abierto en Excel: cerralo y volvé a ejecutar para actualizarlo.")
                    continue
                libros.append(ruta)
        ruta_res = self.salida / f"RESUMEN_{cb.MESES[mes]}_{anio}.xlsx"
        try:
            cb.escribir_resumen(ruta_res, self.certs, self.globales, self.obs_cfg, self.cuad, self.estado["log"],
                                self.mapa_humedad)
        except PermissionError:
            print(f"⚠ {ruta_res.name} está abierto en Excel: cerralo y volvé a ejecutar para actualizarlo.")
        self.guardar_estado()
        return libros, ruta_res

    def exportar(self, pdf=True, excel=True, config_y_estado=True, otros=False, descargar=True):
        """Guarda todo en la carpeta de trabajo y descarga, archivo por archivo, solo lo elegido:
        PDF y/o Excel de cada certificación confirmada; config + estado (para seguir la numeración);
        otros = libro Excel por período (todos los proveedores) y RESUMEN de control."""
        libros, ruta_res = self.guardar_cierre()
        pdfs, xls = self.archivos_confirmados()
        a_bajar = (pdfs if pdf else []) + (xls if excel else [])
        if config_y_estado:
            a_bajar += [self.ruta_config, self.ruta_estado]
        if otros:
            a_bajar += libros + [ruta_res]
        print(f"✔ {len(pdfs)} certificaciones confirmadas: {len(pdfs)} PDF y {len(xls)} Excel")
        print(self.cuad[["Período", "TN maestro", "Ajustes manuales", "TN certificadas", "Diferencia", "Resultado"]]
              .to_string(index=False))
        if not a_bajar:
            print("No hay nada para descargar con las opciones elegidas.")
            return []
        print(f"\n⬇ Descargando {len(a_bajar)} archivos (si el navegador pregunta, elegí 'Permitir'):")
        for r in a_bajar:
            print("   ", Path(r).name)
        if descargar:
            self.descargar_archivos(a_bajar)
        return a_bajar


## Paso 3 · Subir archivos
Seleccioná **a la vez** (con Ctrl):
- **el maestro** `.xlsb` (obligatorio);
- **la certificación modelo** `.xlsx` (opcional: de ahí se toma el logo; sin ella las certificaciones salen sin logo);
- **la planilla de calidad** (humedad y granulometría) **actualizada al cierre del período**. Se reconoce por su contenido, así que el nombre puede cambiar (REV 11, REV 12…);
- **`config_proveedores.xlsx`**, si lo tenés: CUIT, precios y numeración;
- **`estado_AAAA-MM.json`**, solo si estás retomando un trabajo a medias.

In [ ]:
import os, glob, json, importlib, pandas as pd
import cert_biomasa as cb, editor_colab as ed
importlib.reload(cb); importlib.reload(ed)
try:
    from google.colab import files
    subidos = list(files.upload().keys())
except ImportError:
    subidos = glob.glob("*.xlsb") + glob.glob("*.xlsx") + glob.glob("*.json")

def elegir(cond):
    c = [f for f in subidos if cond(f.lower())]
    return c[0] if c else None

# Cada archivo se reconoce por su contenido, no por el nombre
tipos = {f: cb.tipo_archivo(f) for f in subidos}
def elegir(tipo):
    c = [f for f, t in tipos.items() if t == tipo]
    if len(c) > 1:
        print(f"⚠ Subiste {len(c)} archivos de tipo '{tipo}': se usa {c[0]}")
    return c[0] if c else None

RUTA_ESTADO  = elegir("estado")
RUTA_CONFIG  = elegir("config")
RUTA_HUMEDAD = elegir("calidad")
RUTA_MODELO  = elegir("modelo")
RUTA_MAESTRO = elegir("maestro")
for f, t in tipos.items():
    if t is None:
        print(f"⚠ No reconocí el archivo '{f}': no se usa")
assert RUTA_MAESTRO, "Falta subir el maestro de ingreso de biomasa (.xlsb)"

print("Leyendo el maestro…")
maestro = cb.leer_maestro(RUTA_MAESTRO)
if RUTA_CONFIG:
    conf, precios, par = cb.leer_config(RUTA_CONFIG)
else:
    conf, precios, par = cb.plantilla_config(cb.preparar(maestro)[0])
humedad = None
if RUTA_HUMEDAD:
    print("Leyendo la planilla de humedad…")
    humedad = cb.leer_humedad(RUTA_HUMEDAD)
print("Maestro:", RUTA_MAESTRO, f"({len(maestro):,} filas)")
print("Calidad:", f"{RUTA_HUMEDAD} ({len(humedad):,} muestras, la última del {humedad['Fecha'].max():%d/%m/%Y})"
      if RUTA_HUMEDAD else "— ⚠ NO se subió la planilla de calidad: las certificaciones salen SIN descuentos de humedad ni granulometría")
print("Modelo :", RUTA_MODELO or "— (sin logo: subí la certificación modelo para tenerlo)")
print("Config :", RUTA_CONFIG or "— (se crea uno nuevo: cargá CUIT y precios en el editor, y los datos de la empresa en la hoja Parametros)")
print("Estado :", RUTA_ESTADO or "— (empieza de cero)")

## Paso 4 · Elegir qué certificar
- **Año y mes.**
- **Último día de la 1ª quincena** (corte): por defecto el del config (`DIA_CORTE_QUINCENA`, 12). Si el mes ya tiene certificaciones confirmadas se mantiene el corte con el que se hicieron.
- **Períodos:** marcá por separado **1ª quincena**, **2ª quincena** y/o **Mensual**.
- **Proveedor:** elegilo de la lista (sale del maestro, con sus remitos y toneladas del mes) o dejá **Todos**. Podés buscarlo escribiendo parte del nombre.

Después ejecutá el **Paso 5**. Si cambiás la elección, volvé a ejecutar el Paso 5.

In [ ]:
sel = ed.Selector(maestro, conf)
sel.mostrar()

## Paso 5 · Armar las certificaciones

In [ ]:
ANIO, MES = int(sel.w_anio.value), int(sel.w_mes.value)
assert sel.periodos, "Marcá al menos un período en el Paso 4"
DESCARGAR_PDF_AL_CONFIRMAR = True   # el PDF y el Excel se descargan al confirmar

estado = None
if RUTA_ESTADO:
    estado = json.load(open(RUTA_ESTADO, encoding="utf-8"))
    if (int(estado["anio"]), int(estado["mes"])) != (ANIO, MES):
        print(f"⚠ El estado subido es de {estado['mes']}/{estado['anio']}: no se usa.")
        estado = None
    else:
        print(f"Retomando: {len(estado['log'])} cambios registrados.")

app = ed.App(maestro, conf, precios, par, ANIO, MES, sel.periodos, salida=f"certificaciones_{ANIO}-{MES:02d}",
             logo=cb.extraer_logo(RUTA_MODELO), proveedores=sel.proveedores, corte=sel.corte,
             estado=estado, origen=os.path.basename(RUTA_MAESTRO), descargar_pdf=DESCARGAR_PDF_AL_CONFIRMAR,
             humedad=humedad)

quien = "todos los proveedores" if sel.proveedores is None else sel.proveedores[0]
print(f"\n{len(app.certs)} certificaciones · {quien} · {', '.join(sel.periodos)} de {cb.MESES[MES]} {ANIO}")
print(app.resumen_estados(), "\n")
print("CUADRE CONTRA EL MAESTRO")
print(app.cuad[["Período", "TN maestro", "TN certificadas", "Diferencia", "Resultado"]].to_string(index=False))
print("\nTIPOLOGÍAS POR CERTIFICACIÓN (TN a pagar)")
for c in sorted(app.certs.values(), key=lambda c: (c["proveedor"].casefold(), c["periodo"])):
    etq = cb.etiqueta_gyg if c.get("formato") == "GYG" else cb.nombre_producto
    print(f"   {c['periodo']:<8} {c['proveedor']:<38} " + " · ".join(
        f"{etq(t)} {cb.decimal(cb.tn_pagar(c, t))}" for t in cb.orden_productos(c["tn_prod"])))
if app.humedad is not None:
    print(f"\nDESCUENTOS DE CALIDAD · 1ª quincena del 1 al {cb.DIA_CORTE}")
    hay = False
    for c in app.certs.values():
        fl = c["filas"]
        kg = fl["DescKg"].sum()
        if kg > 0:
            hay = True
            print(f"   {c['periodo']:<8} {c['proveedor']:<38} granulometría: {int((fl['DescPct'] > 0).sum())} camiones · "
                  f"{cb.decimal(kg / 1000)} TN de CHIP pasan a ASERRÍN")
        if "Pen" in fl and (fl["Pen"] > 0).any():
            hay = True
            kg_h = (fl["Kilos"] * fl["Pen"]).sum()
            print(f"   {c['periodo']:<8} {c['proveedor']:<38} humedad por camión: {int((fl['Pen'] > 0).sum())} camiones "
                  f"sobre la referencia · −{cb.decimal(kg_h / 1000)} TN")
        for t, h in c["humedad"].items():
            if h["tn_desc"] > 0:
                hay = True
                print(f"   {c['periodo']:<8} {c['proveedor']:<38} humedad {t}: {cb.pct(h['H'], 1)} vs {cb._txt_limite(h)} "
                      f"→ −{cb.pct(h['desc_hum'])} = −{cb.decimal(h['tn_desc'])} TN")
    if not hay:
        print("   Ningún proveedor tiene descuento en el período.")
    ult_lab = app.humedad["Fecha"].max().date()
    sin_m = sorted({c["proveedor"] for c in app.certs.values() if c["desc_gran"]
                    and ((c["filas"]["Producto"] == "CHIP") & c["filas"]["Finos"].isna()
                         & (c["filas"]["Fecha"] <= ult_lab)).any()})
    if sin_m:
        print(f"   ℹ {len(sin_m)} proveedores con camiones de CHIP sin muestra de granulometría asignada "
              f"hasta el {ult_lab:%d/%m} (no se les descuenta en esos camiones; se puede cargar el % a mano en el editor): " + ", ".join(sin_m))
    sin_hum = sorted({c["proveedor"] for c in app.certs.values() for h in c["humedad"].values()
                      if not h["aplica_hum"] and h["H"] is not None and h["L"] is not None and h["H"] > h["L"]})
    if sin_hum:
        print("   ℹ Superan su límite de humedad pero NO tienen DESCUENTA HUMEDAD = SI (no se les descuenta): "
              + ", ".join(sin_hum))
    sv = app.mapa_humedad[app.mapa_humedad["Proveedor en maestro"].str.startswith("—")]
    nombres_mes = set(app.humedad.loc[app.humedad["Fecha"].dt.month == MES, "Proveedor"])
    sv = sv[sv["Nombre en planilla de humedad"].isin(nombres_mes)]
    if len(sv):
        print("   ⚠ Proveedores de la planilla de humedad que no se pudieron vincular (cargalos en NOMBRE HUMEDAD del config):")
        for n in sv["Nombre en planilla de humedad"]:
            print("     -", n)
if app.globales:
    print("\n⚠ Avisos que no pertenecen a ninguna certificación:")
    for g in app.globales:
        print(f"   {g['prov']} {g['rem']}: {g['msg']}")

### Paso 5b · (Opcional) Actualizar solo la planilla de calidad
Tildá `ACTUALIZAR_PLANILLA` y ejecutá esta celda. Si el laboratorio cargó más muestras mientras estás certificando, subí acá la planilla nueva. Se recalculan los descuentos **sin perder** tus correcciones ni las certificaciones confirmadas. Si a alguna ya confirmada le cambia el descuento, pasa a 🔁 *Modificada* para que la vuelvas a confirmar (conserva el número).

In [ ]:
ACTUALIZAR_PLANILLA = False  #@param {type:"boolean"}
nuevos = []
if not ACTUALIZAR_PLANILLA:
    print("Paso 5b no activado (tildá ACTUALIZAR_PLANILLA y ejecutá esta celda si necesitás subir una planilla nueva).")
else:
    try:
        nuevos = list(files.upload().keys())
    except NameError:
        nuevos = []
nueva = next((f for f in nuevos if cb.tipo_archivo(f) == "calidad"), None)
if not ACTUALIZAR_PLANILLA:
    pass
elif not nueva:
    print("No subiste una planilla de calidad (hoja GENERAL con humedad y granulometría).")
else:
    humedad = cb.leer_humedad(nueva)
    cambiadas = app.actualizar_humedad(humedad)
    print(f"✔ Planilla de calidad actualizada: {nueva} · {len(humedad):,} muestras, la última del {humedad['Fecha'].max():%d/%m/%Y}")
    print(app.resumen_estados())
    if cambiadas:
        print("🔁 Cambió el descuento de certificaciones ya confirmadas (volvé a confirmarlas):")
        for k in cambiadas:
            print("   ", k.replace("|", " · "))
    print("Si el editor (Paso 6) ya estaba abierto, ejecutá de nuevo su celda para ver los valores nuevos.")

## Paso 6 · Revisar, corregir y confirmar

1. Escribí tu nombre en **Responsable**.
2. Elegí una certificación de la lista. El ícono indica su estado:

| Ícono | Estado | Qué hacer |
|---|---|---|
| ⛔ | **Bloqueada** | Falta un dato obligatorio (CUIT, precio) o hay un error grave (kilos en 0, producto inválido). Corregilo. |
| ⚠️ | **Revisar** | Tiene observaciones críticas: revisalas, marcá *Revisé…* y dejá un comentario. |
| 🟢 | **Lista** | Se puede confirmar. |
| ✅ | **Confirmada** | Ya tiene número y PDF. |
| 🔁 | **Modificada** | Cambió algo después de confirmarla: volvé a confirmar para regenerar el PDF (conserva el número). |

3. **Corregir o agregar datos:**
   - Editá cualquier celda de la tabla.
   - Para quitar un remito marcá 🗑.
   - Para sumar un remito que quedó afuera (por ejemplo con la fecha mal tipeada), usá **Traer remito del maestro** con su N° de planta.
   - Si el remito no está cargado en el maestro, usá **Agregar fila vacía**.
   - Pasá el mouse por 🔴/🟠 para ver qué detectó el control en cada remito.
4. **Descuentos (sección 3):**
   - **Quitar el descuento de GRANULOMETRÍA / de HUMEDAD en esta certificación:** tildalo y no se descuenta nada de eso en esa certificación (la quincena y el mensual se manejan por separado).
   - **Por camión**, en la tabla de remitos: la columna **Desc. gran. %** muestra el % que pasa a aserrín según el laboratorio. Escribí otro % o **0 para quitarlo**; si lo borrás vuelve al automático. En el formato GYG también está **Desc. hum. %** (penalización por humedad). Los camiones cambiados a mano se marcan con ✋.
   - **Finos lab % / Humedad lab %:** corregilos solo si el laboratorio cargó mal la muestra.
   - **Qué se le descuenta a este proveedor:** vale para todos sus períodos y se guarda en el config.
   - Todo cambio en descuentos pide **motivo** y queda en el *Registro de cambios*.
5. **Precios (sección 4):** cada producto con su **$ por TN**, editable. Elegí **Guardar para este proveedor** (queda en el config para las próximas certificaciones) o **Usar solo en esta certificación**. Cargá también el **CUIT** si falta (sección 1).
6. Escribí el **motivo** de los cambios y tocá **Guardar cambios**.
7. Tocá **Confirmar y generar PDF**: se descargan el PDF y el Excel de la certificación. Después tocá **Siguiente pendiente**.

> Las correcciones se hacen **por remito**: si corregís un remito en la quincena, el mensual también queda corregido. Todo queda en el *Registro de cambios*.

> **Excel de cada certificación (corrección a mano en el archivo):** cada camión muestra su **Humedad %** y **Granulom. %** del laboratorio. Las celdas **celestes** se pueden cambiar a mano: **Desc. humedad %**, **Desc. granulom. %** (por camión) y los **precios**. Kilos, aserrín, TN y total a facturar se recalculan solos. Si falta el CUIT o un precio, la certificación igual se genera y esa celda queda marcada para completarla.


In [ ]:
print(f"{len(app.certs)} certificaciones · {app.resumen_estados()}")
print("Para generar PDF y Excel la certificación tiene que estar 🟢 LISTA (o ⚠️ con las observaciones revisadas).")
print("Si está ⛔, al tocar 'Confirmar' te dice qué falta (CUIT, precio, tolerancia de finos…).\n")
app.mostrar()

### Paso 6b · (Respaldo) Generar PDF y Excel sin usar los botones
**Solo si el editor no te funciona:** tildá `USAR_RESPALDO`, completá `RESPONSABLE` y ejecutá esta celda. Esta celda confirma **todas las certificaciones 🟢 LISTAS** (y las ⚠️ si marcás que las revisaste, con comentario) de lo elegido en el Paso 4, genera su **PDF y Excel** y los descarga como archivos sueltos (elegís PDF, Excel o los dos). Las que no se pueden confirmar se listan con el motivo, para que cargues lo que falta en el config y vuelvas a ejecutar desde el Paso 3.

In [ ]:
USAR_RESPALDO = False  #@param {type:"boolean"}
RESPONSABLE = "Tu nombre"  #@param {type:"string"}
# Marcá esto solo si ya revisaste las observaciones críticas (⚠️) en el RESUMEN o en el editor:
MARCAR_REVISADAS = False  #@param {type:"boolean"}
DESCARGAR_PDF = True      #@param {type:"boolean"}
DESCARGAR_EXCEL = True    #@param {type:"boolean"}
COMENTARIO_REVISION = ""  #@param {type:"string"}
if not USAR_RESPALDO:
    print("Paso 6b no activado: usá el editor del Paso 6. Si el editor no funciona, tildá USAR_RESPALDO, completá RESPONSABLE y ejecutá esta celda.")
elif not RESPONSABLE.strip() or RESPONSABLE == "Tu nombre":
    print("⚠ Completá RESPONSABLE y volvé a ejecutar esta celda.")
elif MARCAR_REVISADAS and not COMENTARIO_REVISION.strip():
    print("⚠ Para marcar como revisadas escribí un COMENTARIO_REVISION.")
else:
    if MARCAR_REVISADAS:
        app.marcar_revisadas(RESPONSABLE, COMENTARIO_REVISION, [k for k, c in app.certs.items() if c["estado"] == "REVISAR"])
    hechas, pendientes, rutas = app.confirmar_lote(RESPONSABLE)
    print(f"✅ {len(hechas)} certificaciones confirmadas con PDF y Excel")
    for k in hechas:
        c = app.certs[k]
        print(f"   Nº {c['numero']} · {c['periodo']} · {c['proveedor']} · {cb.pesos(c['importe'])}")
    if pendientes:
        print(f"\n⛔ {len(pendientes)} no se pudieron confirmar:")
        for k, motivos in pendientes:
            print(f"   {k.replace('|', ' · ')}: " + " | ".join(motivos[:3]))
    a_bajar = [r for r in rutas if (DESCARGAR_PDF and str(r).endswith(".pdf")) or (DESCARGAR_EXCEL and str(r).endswith(".xlsx"))]
    if a_bajar:
        print(f"\n⬇ Descargando {len(a_bajar)} archivos (si el navegador pregunta, elegí 'Permitir'):")
        for r in a_bajar:
            print("   ", os.path.basename(r))
        app.descargar_archivos(a_bajar)
    app.guardar_estado()


## Paso 7 · Descargar
Descarga **archivo por archivo** (sin zip) las certificaciones confirmadas:
- **PDF** de cada certificación;
- **Excel** de cada certificación;
- **config + estado**: el `config_proveedores.xlsx` actualizado (CUIT, precios, tolerancias, próximo número) y el `estado_AAAA-MM.json`. Conviene bajarlos siempre: son los que mantienen la numeración correlativa y te dejan retomar.
- **otros** (opcional): el Excel por período con todos los proveedores y el RESUMEN de control.

Si el navegador pregunta por descargas múltiples, elegí **Permitir**.

In [ ]:
DESCARGAR_PDF = True              #@param {type:"boolean"}
DESCARGAR_EXCEL = True            #@param {type:"boolean"}
DESCARGAR_CONFIG_Y_ESTADO = True  #@param {type:"boolean"}
DESCARGAR_OTROS = False           #@param {type:"boolean"}
_ = app.exportar(pdf=DESCARGAR_PDF, excel=DESCARGAR_EXCEL, config_y_estado=DESCARGAR_CONFIG_Y_ESTADO,
             otros=DESCARGAR_OTROS)